<span style="font-size:2em; font-weight:bold;">Objective 2: Evaluation of Ground Motion Spatial Correlation in New Zealand Through Empirical Approaches</span>
***

This code was written to develop stationary SCMs using the traditional approach as well as produce conduct comparative analysis and evaluation of the non-stationary vs stationary fitted SCMs on NZ Data.

# 1.0 Import Modules and Data

## 1.1 Import Modules

In [ ]:
# Repository paths (works when Jupyter starts at the repository root or in notebooks/).
from pathlib import Path
import sys

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (parent for parent in (_current, *_current.parents)
     if (parent / "src" / "cbb21_cossim.py").is_file()
     and (parent / "data").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Repository root not found. Start Jupyter in the repository or notebooks/ directory.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = REPO_ROOT / "data"
RESIDUALS_DIR = DATA_DIR / "residuals"
SCM_DIR = DATA_DIR / "nzwide_scm"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
import matplotlib.gridspec as gridspec
mpl.rcParams.update(mpl.rcParamsDefault)
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import AutoMinorLocator, LogLocator, NullFormatter, FuncFormatter, LogFormatter, ScalarFormatter
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import bootstrap, gaussian_kde
from scipy.special import logsumexp
from collections import defaultdict

# scikitlearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
import seaborn as sns
from itertools import combinations
from sklearn.neighbors import BallTree

# VarioCorreKrigE
from VarioCorreKrigE.variofit import variofit, variofitmulti
from VarioCorreKrigE.correfit import correfit, correfitmulti

# Chen et al., non-stationary cloud computation
import cbb21_gmscm as gmscm
from cbb21_cossim import cossim, smooth

## 1.2 Function definitions

### 1.2.1 Distance metrics

In [ ]:
# Angular distance given angle relative to earthquake
def getAngDist(theta1,theta2):
    cos_angle = np.cos(np.abs(theta1-theta2))
    distA = np.arccos(np.clip(cos_angle,-1,1))
    return distA

# Soil dissimilarity as a function of the absolute difference Vs30
def getSoilDiss(vs30_1,vs30_2):
    # vs1 = np.clip(vs30_1, 0, 1130)
    # vs2 = np.clip(vs30_2, 0, 1130)
    distS = np.abs(vs30_1-vs30_2)
    return distS

# Circular Median for angular distance
def circmedian(angles, period = 2*np.pi):
    """
    Compute the circular median of angles in radians on [0, 2π)
    """
    if len(angles) == 0:
        return np.nan

    # Wrap angles into [0, 2π)
    angles = np.mod(angles, period)
    sorted_angles = np.sort(angles)

    # Total circular deviation for each candidate
    def angular_distance(a, b):
        diff = np.abs(a - b)
        return np.minimum(diff, period - diff)

    total_deviation = np.array([
        np.sum(angular_distance(angle, sorted_angles)) for angle in sorted_angles
    ])

    return sorted_angles[np.argmin(total_deviation)]

# Circular Mean for angular distance
def circmean(angles, period=2*np.pi):
    """
    Compute the circular mean of angles in radians on [0, 2π)
    """
    angles = np.mod(angles, period)
    sin_sum = np.sum(np.sin(angles))
    cos_sum = np.sum(np.cos(angles))
    return np.arctan2(sin_sum, cos_sum) % period

# source: https://docs.openquake.org/oq-engine/2.7/_modules/openquake/hmtk/seismicity/utils.html#haversine
# def haversine_oq(lon1, lat1, lon2, lat2, radians=False, earth_rad=6371.227):
def haversine_oq(lon1, lat1, lon2, lat2, radians=False, earth_rad=6371):
    """
    Allows to calculate geographical distance
    using the haversine formula.

    :param lon1: longitude of the first set of locations
    :type lon1: numpy.ndarray
    :param lat1: latitude of the frist set of locations
    :type lat1: numpy.ndarray
    :param lon2: longitude of the second set of locations
    :type lon2: numpy.float64
    :param lat2: latitude of the second set of locations
    :type lat2: numpy.float64
    :keyword radians: states if locations are given in terms of radians
    :type radians: bool
    :keyword earth_rad: radius of the earth in km
    :type earth_rad: float
    :returns: geographical distance in km
    :rtype: numpy.ndarray
    """
    if not radians:
        cfact = np.pi / 180.
        lon1 = cfact * lon1
        lat1 = cfact * lat1
        lon2 = cfact * lon2
        lat2 = cfact * lat2

    # Number of locations in each set of points
    if not np.shape(lon1):
        nlocs1 = 1
        lon1 = np.array([lon1])
        lat1 = np.array([lat1])
    else:
        nlocs1 = np.max(np.shape(lon1))
    if not np.shape(lon2):
        nlocs2 = 1
        lon2 = np.array([lon2])
        lat2 = np.array([lat2])
    else:
        nlocs2 = np.max(np.shape(lon2))
    # Pre-allocate array
    distance = np.zeros((nlocs1, nlocs2))
    i = 0
    while i < nlocs2:
        # Perform distance calculation
        dlat = lat1 - lat2[i]
        dlon = lon1 - lon2[i]
        aval = (np.sin(dlat / 2.) ** 2.) + (np.cos(lat1) * np.cos(lat2[i]) *
                                            (np.sin(dlon / 2.) ** 2.))
        distance[:, i] = (2. * earth_rad * np.arctan2(np.sqrt(aval),
                                                      np.sqrt(1 - aval))).T
        i += 1
    return distance

### 1.2.2 Non-stationary correlation computation (Chen et al. 2021)

In [ ]:
def pearsoncorr_BM23(im_sim_all, nj_lim, hmax):
    """
    Optimized (same-structure) correlation cloud builder:
    - Keeps uncentered Pearson definition
    - Uses circmedian like your original
    - Returns the same columns (ordering preserved)
    """
    EARTH_R_KM = 6371.0088

    # Group by station (keep only the columns we actually use in the loop)
    needed_cols = ['evid', 'dw_bar', 'epicenter_azimuth']
    station_groups_full = dict(tuple(im_sim_all.groupby("sta")))
    station_groups = {sta: g[needed_cols].copy() for sta, g in station_groups_full.items()}

    station_meta = (
        im_sim_all.drop_duplicates('sta')
                  .set_index('sta')[['sta_lat','sta_lon','vs30','Z1.0','TRI','k0','TWI','geoage']]
    )
    stations = list(station_groups.keys())
    S = len(stations)

    # Build quick-lookup caches
    ev_sets = {sta: set(g['evid'].values) for sta, g in station_groups.items()}

    lat = station_meta['sta_lat'].to_dict()
    lon = station_meta['sta_lon'].to_dict()
    vs30 = station_meta['vs30'].to_dict()
    Z1   = station_meta['Z1.0'].to_dict()
    TRI  = station_meta['TRI'].to_dict()
    K0   = station_meta['k0'].to_dict()
    TWI  = station_meta['TWI'].to_dict()
    GEO  = station_meta['geoage'].to_dict()

    # spatial pre-pruning: generate only pairs with h <= hmax
    try:
        coords_rad = np.deg2rad(np.c_[ [lat[s] for s in stations], [lon[s] for s in stations] ])
        tree = BallTree(coords_rad, metric='haversine')
        # query radius in radians
        rad = (hmax / EARTH_R_KM) if np.isfinite(hmax) else np.pi
        nbrs = tree.query_radius(coords_rad, r=rad, return_distance=False)

        # Build unique i<j pairs from neighbor lists
        candidate_pairs = []
        for i, neigh in enumerate(nbrs):
            for j in neigh:
                if j > i:  # ensure i<j
                    candidate_pairs.append((i, j))
    except Exception:
        # Fallback: all pairs, distance check inside loop
        candidate_pairs = list(combinations(range(S), 2))

    # Accumulators
    pspear = []
    dsep = []
    stat_i_id = []
    stat_j_id = []
    nj = []
    stat_vs30_i = []
    stat_vs30_j = []
    stat_Z1_i = []
    stat_Z1_j = []
    stat_TRI_i = []
    stat_TRI_j = []
    stat_K0_i = []
    stat_K0_j = []
    stat_TWI_i = []
    stat_TWI_j = []
    stat_geoa_i = []
    stat_geoa_j = []
    stat_lat_i = []
    stat_lon_i = []
    stat_lat_j = []
    stat_lon_j = []
    theta_i = []
    theta_j = []

    # Loop over (unique) station pairs
    for i_idx, j_idx in candidate_pairs:
        sta_i = stations[i_idx]
        sta_j = stations[j_idx]

        # Distance early
        if not np.isfinite(hmax) or 'BallTree' in globals():
            pass
        else:
            distance_temp = haversine_oq(
                np.array([lon[sta_j]]), np.array([lat[sta_j]]),
                np.array([lon[sta_i]]), np.array([lat[sta_i]])
            )[0][0]
            if (distance_temp <= 0) or (distance_temp > hmax):
                continue

        # Overlap check via set intersection
        n_overlap = len(ev_sets[sta_i] & ev_sets[sta_j])
        if n_overlap < nj_lim:
            continue

        # Align on evid once (fast inner merge)
        gi = station_groups[sta_i]
        gj = station_groups[sta_j]
        aligned = gi.merge(gj, on='evid', how='inner', suffixes=('_i', '_j'))
        n_j_temp = len(aligned)
        if n_j_temp < nj_lim:
            continue

        # Distance (compute precisely once if using BallTree or fallback precise)
        distance_temp = haversine_oq(
            np.array([lon[sta_j]]), np.array([lat[sta_j]]),
            np.array([lon[sta_i]]), np.array([lat[sta_i]])
        )[0][0]
        if (distance_temp <= 0) or (distance_temp > hmax):
            continue

        # Uncentered Pearson p_hat
        xi = aligned['dw_bar_i'].to_numpy()
        xj = aligned['dw_bar_j'].to_numpy()
        num = np.dot(xi, xj)
        den = np.sqrt(np.dot(xi, xi) * np.dot(xj, xj))
        if den == 0:
            # degenerate; skip
            continue
        p_temp = num / den

        # Median azimuths over the *overlapped* events
        theta1_temp = circmedian(aligned['epicenter_azimuth_j'])
        theta2_temp = circmedian(aligned['epicenter_azimuth_i'])

        # Append values
        pspear.append(p_temp)
        dsep.append(distance_temp)
        stat_i_id.append(sta_i)
        stat_j_id.append(sta_j)
        nj.append(n_j_temp)

        stat_vs30_i.append(vs30[sta_i])
        stat_vs30_j.append(vs30[sta_j])
        stat_Z1_i.append(Z1[sta_i])
        stat_Z1_j.append(Z1[sta_j])
        stat_TRI_i.append(TRI[sta_i])
        stat_TRI_j.append(TRI[sta_j])
        stat_K0_i.append(K0[sta_i])
        stat_K0_j.append(K0[sta_j])
        stat_TWI_i.append(TWI[sta_i])
        stat_TWI_j.append(TWI[sta_j])
        stat_geoa_i.append(GEO[sta_i])
        stat_geoa_j.append(GEO[sta_j])
        theta_i.append(theta1_temp)
        theta_j.append(theta2_temp)
        stat_lat_i.append(lat[sta_i])
        stat_lon_i.append(lon[sta_i])
        stat_lat_j.append(lat[sta_j])
        stat_lon_j.append(lon[sta_j])

    # Assemble DataFrame
    cbsparcorr = pd.DataFrame()
    cbsparcorr['p_hat'] = pspear
    cbsparcorr['h'] = dsep
    cbsparcorr['sta_j'] = stat_j_id
    cbsparcorr['sta_k'] = stat_i_id
    cbsparcorr['n_j'] = nj
    cbsparcorr['vs30_j'] = stat_vs30_j
    cbsparcorr['vs30_k'] = stat_vs30_i
    cbsparcorr['dS'] = getSoilDiss(np.array(stat_vs30_i), np.array(stat_vs30_j))
    cbsparcorr['Z1_j'] = stat_Z1_j
    cbsparcorr['Z1_k'] = stat_Z1_i
    cbsparcorr['dZ1'] = np.abs(np.array(stat_Z1_i) - np.array(stat_Z1_j))
    cbsparcorr['TRI_j'] = stat_TRI_j
    cbsparcorr['TRI_k'] = stat_TRI_i
    cbsparcorr['dTRI'] = np.abs(np.array(stat_TRI_i) - np.array(stat_TRI_j))
    cbsparcorr['K0_j'] = stat_K0_j
    cbsparcorr['K0_k'] = stat_K0_i
    cbsparcorr['dK0'] = np.abs(np.array(stat_K0_i) - np.array(stat_K0_j))
    cbsparcorr['TWI_j'] = stat_TWI_j
    cbsparcorr['TWI_k'] = stat_TWI_i
    cbsparcorr['dTWI'] = np.abs(np.array(stat_TWI_i) - np.array(stat_TWI_j))
    cbsparcorr['geoa_j'] = stat_geoa_j
    cbsparcorr['geoa_k'] = stat_geoa_i
    cbsparcorr['dGEO'] = np.abs(np.array(stat_geoa_i) - np.array(stat_geoa_j))
    cbsparcorr['theta_i'] = theta_i
    cbsparcorr['theta_j'] = theta_j
    cbsparcorr['dA'] = getAngDist(cbsparcorr.theta_i, cbsparcorr.theta_j) * 180 / np.pi
    cbsparcorr['stat_lon_j'] = stat_lon_j
    cbsparcorr['stat_lat_j'] = stat_lat_j
    cbsparcorr['stat_lon_k'] = stat_lon_i
    cbsparcorr['stat_lat_k'] = stat_lat_i

    # Final filtering
    cbsparcorr = cbsparcorr[cbsparcorr.h > 0]
    if np.isfinite(hmax):
        cbsparcorr = cbsparcorr[cbsparcorr.h <= hmax]
    cbsparcorr = cbsparcorr[cbsparcorr.n_j >= nj_lim]
    cbsparcorr = cbsparcorr.sort_values('h').reset_index(drop=True)

    # Fisher transform
    # clip for numerical safety (avoid +/-1 exactly)
    p = np.clip(cbsparcorr['p_hat'].to_numpy(), -1 + 1e-12, 1 - 1e-12)
    cbsparcorr["z_p_hat"] = 0.5 * np.log((1 + p) / (1 - p))

    return cbsparcorr

### 1.2.3 Existing SCMs

#### 1.2.3.1 Jayaram and Baker 2009

In [ ]:
# Jayaram and Baker (2009) GM SCM
def GMCSM_JB09(case, T,dh,n_rec, cbb2021corr, plot = False):
    """Compute correlation coefficients and deviations based on Jayaram and Baker (2009)."""

    if T < 1 & case == 1:
        # compute coefficient
        b = 8.5+17.2*T

    elif T < 1 & case == 2:
        # compute coefficient
        b = 20

    else:
        b = 22+3.7*T

    # Compute correlation coefficient based on seperation distance in data
    p = np.exp(-3*cbb2021corr.h/b)

    # Compute Fisher's Z-transformation
    z_p = 0.5*np.log((1+p)/(1-p))

    # Compute correlation deviation, e
    e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)

    # compute correlation mean and standard deviation
    e_mean = np.nanmean(e)
    e_std = np.nanstd(e)

    # 5-95% confidence
    data = (e,)
    res = bootstrap(data, np.mean)
    res_std = bootstrap(data, np.std)
    e_5 = res.confidence_interval.low
    e_95 = res.confidence_interval.high
    estd_5 = res_std.confidence_interval.low
    estd_95 = res_std.confidence_interval.high

    # create text string
    if T == 0:
        plot_str = 'PGA'
    else:
        plot_str = 'pSA = %0.2fs'%T

    # Plot results
    if plot:
        x = np.linspace(0,dh,100)
        p_dh = np.exp(-3*x/b)
        p_std = np.square(1-p_dh)/np.sqrt(n_rec)
        fig, ax = plt.subplots(figsize = (8,5), dpi = 150, constrained_layout=True)
        im = ax.scatter(cbb2021corr.h,cbb2021corr.p_hat, c = e, vmin = -4, vmax = 4, cmap = 'bwr', edgecolors = 'black', s = 60)
        plt.plot(x,p_dh,'-k')
        plt.plot(x,p_dh - p_std,'--k')
        plt.plot(x,p_dh + p_std,'--k')
        plt.xlim(0, dh)
        plt.ylim(-1,1)
        # plt.title('Jayaram and Baker (2009): %s'%plot_str)
        ax.set_ylabel(r'Correlation, $\hat{p}$ (%s)'%plot_str)
        ax.set_xlabel('Separation Distance, h (km)')
        # Display mean and standard deviation as text on plot
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label = 'Correlation Deviation, e',aspect = 20)
        plt.show(fig)

    return p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95

#### 3.1.2.2 Goda and Hong 2008

In [ ]:
def GMCSM_GH08(T,dh,n_rec, cbb2021corr, plot = False):
    """Compute correlation coefficients and deviations based on Goda and Hong (2008)."""

    if 0.1 <= T <= 3:
        # compute coefficients
        alpha = -0.16*np.log(T)+0.68
        beta = 0.44

        # Compute correlation coefficient based on seperation distance in data
        p = np.exp(-alpha*cbb2021corr.h**beta)

        # Compute correlation coefficient based on seperation distance specified (for plotted purposes)
        p_dh = np.exp(-alpha*dh**beta)

        # Compute standard deviation of the model based on seperation distance specified (for plotted purposes)
        p_std = np.square(1-p_dh)/np.sqrt(n_rec)

        # Compute Fisher's Z-transformation
        z_p = 0.5*np.log((1+p)/(1-p))

        # Compute correlation deviation, e
        e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)

        # compute correlation mean and standard deviation
        e_mean = np.nanmean(e)
        e_std = np.nanstd(e)

        # 5 - 95% confidence
        data = (e,)
        res = bootstrap(data, np.mean)
        res_std = bootstrap(data, np.std)
        e_5 = res.confidence_interval.low
        e_95 = res.confidence_interval.high
        estd_5 = res_std.confidence_interval.low
        estd_95 = res_std.confidence_interval.high

    elif T == 0.01:
        # compute coefficients
        alpha = 0.93
        beta = 0.49

        # Compute correlation coefficient based on seperation distance in data
        p = np.exp(-alpha*cbb2021corr.h**beta)

        # Compute correlation coefficient based on seperation distance specified (for plotted purposes)
        p_dh = np.exp(-alpha*dh**beta)

        # Compute standard deviation of the model based on seperation distance specified (for plotted purposes)
        p_std = np.square(1-p_dh)/np.sqrt(n_rec)

        # Compute Fisher's Z-transformation
        z_p = 0.5*np.log((1+p)/(1-p))

        # Compute correlation deviation, e
        e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)

        # compute correlation mean and standard deviation
        e_mean = np.nanmean(e)
        e_std = np.nanstd(e)

        # 5 - 95% confidence
        data = (e,)
        res = bootstrap(data, np.mean)
        res_std = bootstrap(data, np.std)
        e_5 = res.confidence_interval.low
        e_95 = res.confidence_interval.high
        estd_5 = res_std.confidence_interval.low
        estd_95 = res_std.confidence_interval.high

    else:
        print('Goda and Hong (2008) not available for pSA = %0.2fs'%T)
        p = cbb2021corr.h*np.nan
        p_dh = dh*np.nan
        e = cbb2021corr.z_p_hat*np.nan
        e_mean = np.nan
        e_std = np.nan
        e_5 = np.nan
        e_95 = np.nan
        estd_5 = np.nan
        estd_95 = np.nan
        plot = False

    # create text string
    if T == 0:
        plot_str = 'PGA'
    else:
        plot_str = 'pSA = %0.1fs'%T

    # Plot results
    if plot:
        fig, ax = plt.subplots(figsize = (8,5), dpi = 150, constrained_layout=True)
        im = ax.scatter(cbb2021corr.h,cbb2021corr.p_hat, c = e, vmin = -4, vmax = 4, cmap = 'bwr', edgecolors = 'black', s = 60)
        plt.plot(dh,p_dh,'-k')
        plt.plot(dh,p_dh - p_std,'--k')
        plt.plot(dh,p_dh + p_std,'--k')
        plt.xlim(0, max(dh))
        plt.ylim(-1,1)
        plt.title('Goda and Hong (2008): %s'%plot_str)
        ax.set_ylabel(r'Correlation, $\hat{p}$ (%s)'%plot_str)
        ax.set_xlabel('Separation Distance, h (km)')
        # Display mean and standard deviation as text on plot
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label = 'Correlation Deviation, e',aspect = 20)
        plt.show(fig)

    return p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95

#### 3.1.2.3 Goda and Atkinson 2010

In [ ]:
def GMCSM_GA10(T,dh,n_rec, cbb2021corr, plot = False):
    """Compute correlation coefficients and deviations based on Goda and Atkinson (2010)."""

    # Define PSA Range
    period = [0.01,0.1,0.2,0.3,0.5,1.0,2.0,3.0,5.0]

    if T in period:
        # Coefficients
        alpha = [0.060,0.062,0.073,0.086,0.073,0.051,0.061,0.092,0.071]
        beta = [0.283,0.276,0.248,0.219,0.248,0.329,0.421,0.671,0.741]
        gamma = [5.000,5.000,5.000,5.000,5.000,5.000,3.035,1.189,1.201]

        # Get location of Coefficient
        loc = period.index(T)

        # Compute correlation coefficient based on seperation distance in data
        p = gamma[loc]*np.exp(-alpha[loc]*cbb2021corr.h**beta[loc])-gamma[loc]+1

        # Compute correlation coefficient based on seperation distance specified (for plotted purposes)
        p_dh = gamma[loc]*np.exp(-alpha[loc]*dh**beta[loc])-gamma[loc]+1

        # Compute standard deviation of the model based on seperation distance specified (for plotted purposes)
        p_std = np.square(1-p_dh)/np.sqrt(n_rec)

        # Compute Fisher's Z-transformation
        z_p = 0.5*np.log((1+p)/(1-p))

        # Compute correlation deviation, e
        e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)

        # compute correlation mean and standard deviation
        e_mean = np.nanmean(e)
        e_std = np.nanstd(e)

        # 5- 95% confidence
        data = (e,)
        res = bootstrap(data, np.mean)
        res_std = bootstrap(data, np.std)
        e_5 = res.confidence_interval.low
        e_95 = res.confidence_interval.high
        estd_5 = res_std.confidence_interval.low
        estd_95 = res_std.confidence_interval.high

    else:
        print('Goda and Atkinson (2010) not available for pSA = %0.2fs'%T)
        p = cbb2021corr.h*np.nan
        p_dh = dh*np.nan
        p_std = dh*np.nan
        e = cbb2021corr.z_p_hat*np.nan
        e_mean = np.nan
        e_std = np.nan
        e_5 = np.nan
        e_95 = np.nan
        estd_5 = np.nan
        estd_95 = np.nan
        plot = False

    # create text string
    if T == 0:
        plot_str = 'PGA'
    else:
        plot_str = 'pSA = %0.1fs'%T

    # Plot results
    if plot:
        fig, ax = plt.subplots(figsize = (8,5), dpi = 150, constrained_layout=True)
        im = ax.scatter(cbb2021corr.h,cbb2021corr.p_hat, c = e, vmin = -4, vmax = 4, cmap = 'bwr', edgecolors = 'black', s = 60)
        plt.plot(dh,p_dh,'-k')
        plt.plot(dh,p_dh - p_std,'--k')
        plt.plot(dh,p_dh + p_std,'--k')
        plt.xlim(0, max(dh))
        plt.ylim(-1,1)
        plt.title('Goda and Atkinson (2010): %s'%plot_str)
        ax.set_ylabel(r'Correlation, $\hat{p}$ (%s)'%plot_str)
        ax.set_xlabel('Separation Distance, h (km)')
        # Display mean and standard deviation as text on plot
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label = 'Correlation Deviation, e',aspect = 20)
        plt.show(fig)

    return p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95

#### 3.1.2.4 Eposito and Iervolino 2012

In [ ]:
def GMCSM_EI12(T,dh,n_rec, cbb2021corr, plot = False):
    """Compute correlation coefficients and deviations based on Jayaram and Baker (2009)."""

    # Calculate Range
    b = 8.6 + 11.6*T

    # Compute correlation coefficient based on seperation distance in data
    p = np.exp(-3*cbb2021corr.h/b)

    # Compute correlation coefficient based on seperation distance specified (for plotted purposes)
    p_dh = np.exp(-3*dh/b)

    # Compute standard deviation of the model based on seperation distance specified (for plotted purposes)
    p_std = np.square(1-p_dh)/np.sqrt(n_rec)

    # Compute Fisher's Z-transformation
    z_p = 0.5*np.log((1+p)/(1-p))

    # Compute correlation deviation, e
    e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)

    # compute correlation mean and standard deviation
    e_mean = np.nanmean(e)
    e_std = np.nanstd(e)

    # 95% confidence
    data = (e,)
    res = bootstrap(data, np.mean)
    res_std = bootstrap(data, np.std)
    e_5 = res.confidence_interval.low
    e_95 = res.confidence_interval.high
    estd_5 = res_std.confidence_interval.low
    estd_95 = res_std.confidence_interval.high

    # create text string
    if T == 0:
        plot_str = 'PGA'
    else:
        plot_str = 'pSA = %0.2fs'%T

    # Plot results
    if plot:
        fig, ax = plt.subplots(figsize = (8,5), dpi = 150, constrained_layout=True)
        im = ax.scatter(cbb2021corr.h,cbb2021corr.p_hat, c = e, vmin = -4, vmax = 4, cmap = 'bwr', edgecolors = 'black', s = 60)
        plt.plot(dh,p_dh,'-k')
        plt.plot(dh,p_dh - p_std,'--k')
        plt.plot(dh,p_dh + p_std,'--k')
        plt.xlim(0, max(dh))
        plt.ylim(-1,1)
        plt.title('Eposito and Iervolino (2012): %s'%plot_str)
        ax.set_ylabel(r'Correlation, $\hat{p}$ (%s)'%plot_str)
        ax.set_xlabel('Separation Distance, h (km)')
        # Display mean and standard deviation as text on plot
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label = 'Correlation Deviation, e',aspect = 20)
        plt.show(fig)

    return p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95

#### 3.1.2.5 Heresi and Miranda 2019

In [ ]:
def GMCSM_HM19(T,dh,n_rec, cbb2021corr, plot = False):
    """Compute correlation coefficients and deviations based on Heresi and Miranda (2019)."""
    if T < 1.37:
        # compute coefficient
        beta = 4.231*T**2 - 5.180*T + 13.392

    else:
        # compute coefficient
        beta = 0.140*T**2 - 2.249*T + 17.050

    # Compute correlation coefficient based on seperation distance in data
    p = np.exp(-(cbb2021corr.h/beta)**0.55)

    # Compute correlation coefficient based on seperation distance specified (for plotted purposes)
    p_dh = np.exp(-(dh/beta)**0.55)

    # Compute standard deviation of the model based on seperation distance specified (for plotted purposes)
    p_std = np.square(1-p_dh)/np.sqrt(n_rec)

    # Compute Fisher's Z-transformation
    z_p = 0.5*np.log((1+p)/(1-p))

    # Compute correlation deviation, e
    e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)

    # compute correlation mean and standard deviation
    e_mean = np.nanmean(e)
    e_std = np.nanstd(e)

    # 95% confidence
    data = (e,)
    res = bootstrap(data, np.mean)
    res_std = bootstrap(data, np.std)
    e_5 = res.confidence_interval.low
    e_95 = res.confidence_interval.high
    estd_5 = res_std.confidence_interval.low
    estd_95 = res_std.confidence_interval.high

    # create text string
    if T == 0:
        plot_str = 'PGA'
    else:
        plot_str = 'pSA = %0.2fs'%T

    # Plot results
    if plot:
        fig, ax = plt.subplots(figsize = (8,5), dpi = 150, constrained_layout=True)
        im = ax.scatter(cbb2021corr.h,cbb2021corr.p_hat, c = e, vmin = -4, vmax = 4, cmap = 'bwr', edgecolors = 'black', s = 60)
        plt.plot(dh,p_dh,'-k')
        plt.plot(dh,p_dh - p_std,'--k')
        plt.plot(dh,p_dh + p_std,'--k')
        plt.xlim(0, max(dh))
        plt.ylim(-1,1)
        plt.title('Heresi and Miranda (2019): %s'%plot_str)
        ax.set_ylabel(r'Correlation, $\hat{p}$ (%s)'%plot_str)
        ax.set_xlabel('Separation Distance, h (km)')
        # Display mean and standard deviation as text on plot
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label = 'Correlation Deviation, e',aspect = 20)
        plt.show(fig)

    return p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95

#### 1.2.3.6 Loth and Baker 2013

In [ ]:
# Short range coregionalization matrix, B1
B1 = np.array(
    [
        [0.29, 0.25, 0.23, 0.23, 0.18, 0.1, 0.06, 0.06, 0.06],
        [0.25, 0.30, 0.2, 0.16, 0.1, 0.04, 0.03, 0.04, 0.05],
        [0.23, 0.20, 0.27, 0.18, 0.1, 0.03, 0, 0.01, 0.02],
        [0.23, 0.16, 0.18, 0.31, 0.22, 0.14, 0.08, 0.07, 0.07],
        [0.18, 0.10, 0.1, 0.22, 0.33, 0.24, 0.16, 0.13, 0.12],
        [0.10, 0.04, 0.03, 0.14, 0.24, 0.33, 0.26, 0.21, 0.19],
        [0.06, 0.03, 0, 0.08, 0.16, 0.26, 0.37, 0.3, 0.26],
        [0.06, 0.04, 0.01, 0.07, 0.13, 0.21, 0.3, 0.28, 0.24],
        [0.06, 0.05, 0.02, 0.07, 0.12, 0.19, 0.26, 0.24, 0.23],
    ]
)

# Long range coregionalization matrix, B2
B2 = np.array(
    [
        [0.47, 0.4, 0.43, 0.35, 0.27, 0.15, 0.13, 0.09, 0.12],
        [0.4, 0.42, 0.37, 0.25, 0.15, 0.03, 0.04, 0, 0.03],
        [0.43, 0.37, 0.45, 0.36, 0.26, 0.15, 0.09, 0.05, 0.08],
        [0.35, 0.25, 0.36, 0.42, 0.37, 0.29, 0.2, 0.16, 0.16],
        [0.27, 0.15, 0.26, 0.37, 0.48, 0.41, 0.26, 0.21, 0.21],
        [0.15, 0.03, 0.15, 0.29, 0.41, 0.55, 0.37, 0.33, 0.32],
        [0.13, 0.04, 0.09, 0.2, 0.26, 0.37, 0.51, 0.49, 0.49],
        [0.09, 0.00, 0.05, 0.16, 0.21, 0.33, 0.49, 0.62, 0.6],
        [0.12, 0.03, 0.08, 0.16, 0.21, 0.32, 0.49, 0.6, 0.68],
    ]
)

# Nugget effect coregionalization matrix, B3
B3 = np.array(
    [
        [
            0.240000000000000,
            0.219983028675722,
            0.209991239369580,
            0.0899940658151642,
            -0.0199982490874490,
            0.0100004273375877,
            0.0299729607606612,
            0.0200291990885140,
            0.00995702711846606,
        ],
        [
            0.219983028675722,
            0.280000000000000,
            0.199999710563431,
            0.0400020556476041,
            -0.0500003168664929,
            -5.02841885169300e-07,
            0.0100141900421009,
            0.00994747690890486,
            -0.00996663511790072,
        ],
        [
            0.209991239369580,
            0.199999710563431,
            0.280000000000000,
            0.0500007637487926,
            -0.0600002196848805,
            -1.80663938055364e-07,
            0.0399992445541918,
            0.0299487635912810,
            0.0100035235224244,
        ],
        [
            0.0899940658151642,
            0.0400020556476041,
            0.0500007637487926,
            0.270000000000000,
            0.139999321454879,
            0.0499996979574019,
            0.0499981807238188,
            0.0499227563681531,
            0.0399858842409999,
        ],
        [
            -0.0199982490874490,
            -0.0500003168664929,
            -0.0600002196848805,
            0.139999321454879,
            0.190000000000000,
            0.0700000354290215,
            0.0499897414826383,
            0.0499443288879162,
            0.0499652709189241,
        ],
        [
            0.0100004273375877,
            -5.02841885169300e-07,
            -1.80663938055364e-07,
            0.0499996979574019,
            0.0700000354290215,
            0.120000000000000,
            0.0799859494118349,
            0.0699172702775962,
            0.0599608152312721,
        ],
        [
            0.0299729607606612,
            0.0100141900421009,
            0.0399992445541918,
            0.0499981807238188,
            0.0499897414826383,
            0.0799859494118349,
            0.120000000000000,
            0.0997643834727755,
            0.0800031285024676,
        ],
        [
            0.0200291990885140,
            0.00994747690890486,
            0.0299487635912810,
            0.0499227563681531,
            0.0499443288879162,
            0.0699172702775962,
            0.0997643834727755,
            0.100000000000000,
            0.0896690207890228,
        ],
        [
            0.00995702711846606,
            -0.00996663511790072,
            0.0100035235224244,
            0.0399858842409999,
            0.0499652709189241,
            0.0599608152312721,
            0.0800031285024676,
            0.0896690207890228,
            0.0900000000000000,
        ],
    ]
)

T_list = np.array([0.01, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0001])

B_coeff1_interpolator = RegularGridInterpolator((T_list, T_list), B1, bounds_error=True, method="linear")
B_coeff2_interpolator = RegularGridInterpolator((T_list, T_list), B2, bounds_error=True, method="linear")
B_coeff3_interpolator = RegularGridInterpolator((T_list, T_list), B3, bounds_error=True, method="linear")

def _interpolate_B_coefficients(T1: float, T2: float, index_1: int, index_2: int, B: np.ndarray):
    """Diagonal-preserving interpolation for Loth & Baker matrices."""
    T1_vals = T_list[index_1:index_1 + 2]
    T2_vals = T_list[index_2:index_2 + 2]
    B_vals = B[index_1:index_1 + 2, index_2:index_2 + 2]

    T_avg_targ = np.mean([T1, T2])
    T_avg_vals = np.mean([T1_vals, T2_vals], axis=0)
    B_avg_vals = np.diagonal(B_vals)
    B_diag_val = np.interp(T_avg_targ, T_avg_vals, B_avg_vals)

    T_diff_targ = np.abs(T1 - T2)
    T_diff_vals = [0, np.abs(np.min(T1_vals) - np.max(T2_vals))]
    B_diff_vals = [B_diag_val, B_vals[0, 1]]
    return np.interp(T_diff_targ, T_diff_vals, B_diff_vals)

def GMCSM_LB13(T1, T2, dh, n_rec, corrdata, plot=False):
    """
    Compute correlation coefficients and deviations based on Loth & Baker (2009).

    Parameters
    ----------
    T1, T2 : float
        Periods (s). If T1 == T2, diagonal correlation is computed.
    dh : float
        Maximum separation distance (km).
    n_rec : int
        Number of records.
    corrdata : object
        Must contain attributes h (distances), p_hat (empirical corr),
        z_p_hat (Fisher Z transform), n_j (sample size).
    plot : bool
        If True, plot correlation vs distance.

    Returns
    -------
    p : np.ndarray
        Correlation coefficients.
    e : np.ndarray
        Correlation deviations.
    e_mean, e_std : float
        Mean and std of deviations.
    e_5, e_95 : float
        5–95% confidence interval for mean.
    estd_5, estd_95 : float
        5–95% confidence interval for std.
    """

    if min(T1, T2) < 0.01 or max(T1, T2) > 10:
        raise ValueError("Periods must be between 0.01 and 10 seconds")

    # Interpolate coefficients
    index_1 = np.flatnonzero(T_list <= T1)[-1]
    index_2 = np.flatnonzero(T_list <= T2)[-1]

    if index_1 == index_2:
        B1c = _interpolate_B_coefficients(T1, T2, index_1, index_2, B1)
        B2c = _interpolate_B_coefficients(T1, T2, index_1, index_2, B2)
        B3c = _interpolate_B_coefficients(T1, T2, index_1, index_2, B3)
    else:
        B1c = B_coeff1_interpolator([T1, T2])
        B2c = B_coeff2_interpolator([T1, T2])
        B3c = B_coeff3_interpolator([T1, T2])

    # Correlation model
    p = B1c * np.exp(-3 * corrdata.h / 20.0) \
        + B2c * np.exp(-3 * corrdata.h / 70.0) \
        + np.where(np.isclose(corrdata.h, 0.0), B3c, 0.0)

    # Fisher Z transform
    z_p = 0.5 * np.log((1 + p) / (1 - p))

    # Deviations
    e = (corrdata.z_p_hat - z_p) * np.sqrt(corrdata.n_j - 2)
    e_mean = np.nanmean(e)
    e_std = np.nanstd(e)

    # Bootstrap confidence intervals
    data = (e,)
    res = bootstrap(data, np.nanmean)
    res_std = bootstrap(data, np.nanstd)
    e_5, e_95 = res.confidence_interval.low, res.confidence_interval.high
    estd_5, estd_95 = res_std.confidence_interval.low, res_std.confidence_interval.high

    # Plot
    if plot:
        # create text string
        if np.isclose(T1, T2):
            if T1 == 0:
                plot_str = 'PGA'
            else:
                plot_str = f'pSA = {T1:.2f}s'
        else:
            plot_str = f'pSA = {T1:.2f}s vs {T2:.2f}s'

        x = np.linspace(0, dh, 200)
        p_dh = B1c * np.exp(-3 * x / 20.0) + B2c * np.exp(-3 * x / 70.0)

        # Standard deviation band (same style as HM19/JB09)
        p_std = np.square(1 - p_dh) / np.sqrt(n_rec)

        fig, ax = plt.subplots(figsize=(8, 5), dpi=150, constrained_layout=True)
        im = ax.scatter(corrdata.h, corrdata.p_hat, c=e, vmin=-4, vmax=4,
                        cmap='bwr', edgecolors='black', s=60)
        ax.plot(x, p_dh, '-k', label='LB09 mean corr')
        ax.plot(x, p_dh - p_std, '--k', label='-1σ')
        ax.plot(x, p_dh + p_std, '--k', label='+1σ')
        plt.title(f'Loth and Baker (2013): {plot_str}')
        ax.set_ylabel(rf'Correlation, $\hat{{p}}$ ({plot_str})')
        ax.set_xlabel('Separation Distance, h (km)')
        ax.set_xlim(0, dh)
        ax.set_ylim(-1,1)
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label='Correlation Deviation, e', aspect=20)
        plt.show(fig)

    return p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95

#### 3.1.2.7 Bodenmann et al., 2023

In [ ]:
period_BM23 = [0.01,0.03,0.06,0.1,0.3,0.6,1,3,6]

# model E
modelE_lE_BM23 = [10.5, 10.8, 10.4, 10.7, 10.3, 14.9, 16, 21.4, 37.3]
modelE_lE5_BM23 = [9.8, 10.1, 9.8, 10, 9.5, 13.7, 14.7, 19.3, 33.1]
modelE_lE95_BM23 = [11.2, 11.5, 11.1, 11.4, 11.1, 16.2, 17.4, 23.7, 41.9]
modelE_gE_BM23 = [0.43, 0.43, 0.43, 0.42, 0.39, 0.39, 0.4, 0.44, 0.45]
modelE_gE5_BM23 = [0.42, 0.42, 0.42, 0.41, 0.38, 0.37, 0.38, 0.42, 0.42]
modelE_gE95_BM23 = [0.45, 0.45, 0.45, 0.44, 0.41, 0.40, 0.42, 0.46, 0.47]

# model EA
modelEA_LE_BM23 = [13.1, 13.5, 13.2, 13.8, 12.3, 19.1, 21.3, 29, 54.8]
modelEA_LE5_BM23 = [12, 12.4, 12.1, 12.6, 11.3, 17.3, 19.2, 25.4, 46.4]
modelEA_LE95_BM23 = [14.3, 14.7, 14.4, 15, 13.6, 21.1, 23.5, 33, 64.2]
modelEA_gE_BM23 = [0.35, 0.35, 0.34, 0.33, 0.33, 0.33, 0.35, 0.38, 0.39]
modelEA_gE5_BM23 = [0.33, 0.33, 0.32, 0.31, 0.31, 0.32, 0.33, 0.36, 0.36]
modelEA_gE95_BM23 = [0.36, 0.36, 0.36, 0.34, 0.35, 0.35, 0.36, 0.41, 0.41]
modelEA_LA_BM23 = [27.3, 28.2, 26.8, 24.8, 28.8, 27.6, 23.6, 25.1, 26.2]
modelEA_LA5_BM23 = [23.8, 24.5, 23.5, 21.9, 24.9, 24.2, 20.8, 20.8, 21.1]
modelEA_LA95_BM23 = [31.2, 32.4, 30.4, 28, 33.3, 31.5, 26.7, 30.1, 32.8]

# model EAS
modelEAS_LE_BM23 = [16.4, 16.9, 16.6, 16.3, 15.1, 25.6, 29.8, 42.1, 70.2]
modelEAS_LE5_BM23 = [14.9, 15.4, 15.2, 14.8, 13.6, 23.3, 27, 37.4, 60.1]
modelEAS_LE95_BM23 = [17.9, 18.5, 18.2, 17.9, 16.6, 28.3, 32.8, 47.2, 81.7]
modelEAS_gE_BM23 = [0.36, 0.36, 0.35, 0.34, 0.34, 0.37, 0.41, 0.46, 0.49]
modelEAS_gE5_BM23 = [0.34, 0.35, 0.34, 0.32, 0.32, 0.35, 0.38, 0.43, 0.45]
modelEAS_gE95_BM23 = [0.38, 0.38, 0.37, 0.36, 0.36, 0.39, 0.43, 0.49, 0.54]
modelEAS_LA_BM23 = [24.9, 25.6, 24.4, 23.3, 26.1, 24.2, 20.5, 18.5, 17.3]
modelEAS_LA5_BM23 = [21.6, 22.1, 21.4, 20.5, 22.4, 20.8, 17.8, 14.6, 12.9]
modelEAS_LA95_BM23 = [28.6, 29.3, 27.8, 26.5, 30.5, 28, 23.6, 22.8, 22.5]
modelEAS_LS_BM23 = [171.2, 185.6, 190.2, 189.8, 199.9, 222.8, 169.2, 358, 372.2]
modelEAS_LS5_BM23 = [106.6, 115.3, 123.8, 115.7, 116.5, 129, 106.1, 251.7, 228]
modelEAS_LS95_BM23 = [256.1, 279.3, 276.7, 290, 315.7, 341, 250.8, 482.6, 536.9]
modelEAS_w_BM23 = [0.84, 0.84, 0.84, 0.88, 0.85, 0.73, 0.7, 0.5, 0.54]
modelEAS_w5_BM23 = [0.8, 0.8, 0.8, 0.84, 0.8, 0.65, 0.62, 0.4, 0.42]
modelEAS_w95_BM23 = [0.88, 0.88, 0.88, 0.92, 0.89, 0.8, 0.77, 0.6, 0.65]

BM23_parameters = pd.DataFrame()
BM23_parameters['period'] = period_BM23
BM23_parameters['modelE_LE'] = modelE_lE_BM23
BM23_parameters['modelE_LE_5'] = modelE_lE5_BM23
BM23_parameters['modelE_LE_95'] = modelE_lE95_BM23
BM23_parameters['modelE_gammaE'] = modelE_gE_BM23
BM23_parameters['modelE_gammaE5'] = modelE_gE5_BM23
BM23_parameters['modelE_gammaE95'] = modelE_gE95_BM23
BM23_parameters['modelEA_LE'] = modelEA_LE_BM23
BM23_parameters['modelEA_LE_5'] = modelEA_LE5_BM23
BM23_parameters['modelEA_LE_95'] = modelEA_LE95_BM23
BM23_parameters['modelEA_gammaE'] = modelEA_gE_BM23
BM23_parameters['modelEA_gammaE_5'] = modelEA_gE5_BM23
BM23_parameters['modelEA_gammaE_95'] = modelEA_gE95_BM23
BM23_parameters['modelEA_LA'] = modelEA_LA_BM23
BM23_parameters['modelEA_LA_5'] = modelEA_LA5_BM23
BM23_parameters['modelEA_LA_95'] = modelEA_LA95_BM23
BM23_parameters['modelEAS_LE'] = modelEAS_LE_BM23
BM23_parameters['modelEAS_LE_5'] = modelEAS_LE5_BM23
BM23_parameters['modelEAS_LE_95'] = modelEAS_LE95_BM23
BM23_parameters['modelEAS_gammaE'] = modelEAS_gE_BM23
BM23_parameters['modelEAS_gammaE_5'] = modelEAS_gE5_BM23
BM23_parameters['modelEAS_gammaE_95'] = modelEAS_gE95_BM23
BM23_parameters['modelEAS_LA'] = modelEAS_LA_BM23
BM23_parameters['modelEAS_LA_5'] = modelEAS_LA5_BM23
BM23_parameters['modelEAS_LA_95'] = modelEAS_LA95_BM23
BM23_parameters['modelEAS_LS'] = modelEAS_LS_BM23
BM23_parameters['modelEAS_LS_5'] = modelEAS_LS5_BM23
BM23_parameters['modelEAS_LS_95'] = modelEAS_LS95_BM23
BM23_parameters['modelEAS_w'] = modelEAS_w_BM23
BM23_parameters['modelEAS_w_5'] = modelEAS_w5_BM23
BM23_parameters['modelEAS_w_95'] = modelEAS_w95_BM23

# Euclidean Distance Correlation Model
def getpE(dE,lE,gamma_E):
    pE = np.zeros((len(dE),1))
    pE[:,0] = np.exp(-(dE/lE)**gamma_E)
    return pE

# Angular Distance Correlation Model
def getpA(dA,lA):
    pA = (1+dA/lA)*(1-dA/180)**(180/lA)
    return pA

# Soil Dissimilarity Correlation Model
def getpS(dS,lS):
    pS = np.exp(-dS/lS)
    return pS

# combine and call model: EA
def MpEA(pE,pA):
    pEA = pE*pA
    return pEA

# combine and call model: EAS
def MpEAS(pE,pA,pS,w):
    pEAS = pE*(w*pA+(1-w)*pS)
    return pEAS

# main correlation function
def BM23_CBB21(cbb21_pSAobs_T, period, parameters):

    eE_mean_all = []; eE_std_all = []; eE_5_all = []; eE_95_all = []; eEstd_5_all = []; eEstd_95_all = []
    eEA_mean_all = []; eEA_std_all = []; eEA_5_all = []; eEA_95_all = []; eEAstd_5_all = []; eEAstd_95_all = []
    eEAS_mean_all = []; eEAS_std_all = []; eEAS_5_all = []; eEAS_95_all = []; eEASstd_5_all = []; eEASstd_95_all = []

    for i, T in enumerate(period):

        # Model E
        pEE = getpE(cbb21_pSAobs_T[i].h, parameters.modelE_LE[i], parameters.modelE_gammaE[i])
        pEE= pEE.reshape(-1)

        # Model EA
        pEAA = getpE(cbb21_pSAobs_T[i].h, parameters.modelEA_LE[i], parameters.modelEA_gammaE[i])
        pEAA= pEAA.reshape(-1)
        pAA = getpA(cbb21_pSAobs_T[i].dA, parameters.modelEA_LA[i])
        pEA = MpEA(pEAA, pAA)

        # model EAS
        pE = getpE(cbb21_pSAobs_T[i].h, parameters.modelEAS_LE[i], parameters.modelEAS_gammaE[i])
        pA = getpA(cbb21_pSAobs_T[i].dA, parameters.modelEAS_LA[i])
        pS = getpS(cbb21_pSAobs_T[i].dS, parameters.modelEAS_LS[i])
        pEAS = MpEAS(pE.reshape(-1), pA, pS, parameters.modelEAS_w[i])

        ## compute correlation mean and standard deviation
        # Model E
        z_pE = 0.5*np.log((1+pEE)/(1-pEE))
        eE = (cbb21_pSAobs_T[i].z_p_hat - z_pE)*np.sqrt(cbb21_pSAobs_T[i].n_j-2)
        eE_mean = np.nanmean(eE)
        eE_std = np.nanstd(eE)

        # Model EA
        z_pEA = 0.5*np.log((1+pEA)/(1-pEA))
        eEA = (cbb21_pSAobs_T[i].z_p_hat - z_pEA)*np.sqrt(cbb21_pSAobs_T[i].n_j-2)
        eEA_mean = np.nanmean(eEA)
        eEA_std = np.nanstd(eEA)

        # Model EAS
        z_pEAS = 0.5*np.log((1+pEAS)/(1-pEAS))
        eEAS = (cbb21_pSAobs_T[i].z_p_hat - z_pEAS)*np.sqrt(cbb21_pSAobs_T[i].n_j-2)
        eEAS_mean = np.nanmean(eEAS)
        eEAS_std = np.nanstd(eEAS)

        # 5 - 95 CI: Model E
        data_E = (eE,); res_E = bootstrap(data_E, np.nanmean); resstd_E = bootstrap(data_E, np.nanstd)
        eE_5 = res_E.confidence_interval.low; eE_95 = res_E.confidence_interval.high
        eE_std_5 = resstd_E.confidence_interval.low; eE_std_95 = resstd_E.confidence_interval.high

        # 5 - 95 CI: Model EA
        data_EA = (eEA,); res_EA = bootstrap(data_EA, np.nanmean); resstd_EA = bootstrap(data_EA, np.nanstd)
        eEA_5 = res_EA.confidence_interval.low; eEA_95 = res_EA.confidence_interval.high
        eEA_std_5 = resstd_EA.confidence_interval.low; eEA_std_95 = resstd_EA.confidence_interval.high

        # 5 - 95 CI: Model EAS
        data_EAS = (eEAS,); res_EAS = bootstrap(data_EAS, np.nanmean); resstd_EAS = bootstrap(data_EAS, np.nanstd)
        eEAS_5 = res_EAS.confidence_interval.low; eEAS_95 = res_EAS.confidence_interval.high
        eEAS_std_5 = resstd_EAS.confidence_interval.low; eEAS_std_95 = resstd_EAS.confidence_interval.high

        ## save to list
        # Model E
        eE_mean_all.append(eE_mean); eE_std_all.append(eE_std)
        eE_5_all.append(eE_5); eE_95_all.append(eE_95)
        eEstd_5_all.append(eE_std_5); eEstd_95_all.append(eE_std_95)

        # Model EA
        eEA_mean_all.append(eEA_mean); eEA_std_all.append(eEA_std)
        eEA_5_all.append(eEA_5); eEA_95_all.append(eEA_95)
        eEAstd_5_all.append(eEA_std_5); eEAstd_95_all.append(eEA_std_95)

        # Model EAS
        eEAS_mean_all.append(eEAS_mean); eEAS_std_all.append(eEAS_std)
        eEAS_5_all.append(eEAS_5); eEAS_95_all.append(eEAS_95)
        eEASstd_5_all.append(eEAS_std_5); eEASstd_95_all.append(eEAS_std_95)

    return eE_mean_all, eEA_mean_all, eEAS_mean_all, eE_std_all, eEA_std_all, eEAS_std_all, eE_5_all, eE_95_all, eEstd_5_all, eEstd_95_all, eEA_5_all, eEA_95_all, eEAstd_5_all, eEAstd_95_all, eEAS_5_all, eEAS_95_all, eEASstd_5_all, eEASstd_95_all

## 1.3 Import SCM Development Data

### 1.3.1 Define period and file attributes

In [ ]:
# specify filenames prefix
im_dwbar_file = 'NZGMDB_v4p3_dwbar'
im_dwbar_file_crus = 'NZGMDB_v4p3_dwbar_crus'
im_dwbar_file_sub = 'NZGMDB_v4p3_dwbar_sub'
im_dwbar_file_grp1 = 'NZGMDB_v4p3_dwbar_grp1'
im_dwbar_file_grp2 = 'NZGMDB_v4p3_dwbar_grp2'
im_dwbar_file_grpn = 'NZGMDB_v4p3_dwbar_grpn'
im_dwbar_file_grps = 'NZGMDB_v4p3_dwbar_grps'
im_dwbar_file_n3 = 'NZGMDB_v4p3_dwbar_n3'

# input/output directory
input_directory = RESIDUALS_DIR

# specify periods and prefix
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0', 'pSA_7.0', 'pSA_8.0', 'pSA_9.0', 'pSA_10.0']
pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

### 1.3.2 Read and store data

In [ ]:
im_dwbar_pSA_files = []; im_dwbar_pSA_files_crus = []; im_dwbar_pSA_files_sub = []; im_dwbar_pSA_files_grp1 = [];  im_dwbar_pSA_files_grp2 = []; im_dwbar_pSA_files_grpn = []; im_dwbar_pSA_files_grps = []; im_dwbar_pSA_files_n3 = []
for pID in tqdm(range(len(pSAobs_T))):

    #load file: all data
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files.append(im_dwbar)

    #load file: all data crustal
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_crus}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_crus.append(im_dwbar)

     #load file: all data sub
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_sub}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_sub.append(im_dwbar)

    # load file: group 1 wellington
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grp1}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grp1.append(im_dwbar)

    # load file: group 2 canterbury
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grp2}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grp2.append(im_dwbar)

    # load file: group north island
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grpn}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grpn.append(im_dwbar)

    # load file: group south island
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grps}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grps.append(im_dwbar)

    # load file: minimum obseravations per event = 3
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_n3}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_n3.append(im_dwbar)

### 1.3.3 Import original dataframe and filters

In [ ]:
## Load NZ GMDB Flatfile
gm_im_table_50 = pd.read_csv(RESIDUALS_DIR / "ground_motion_im_table_rotd50_flat.csv", low_memory=False, dtype={"sta": "string"})

## Remove non-IM information to optimize storage
gm_im_table_50.drop(gm_im_table_50.iloc[:,182:571],inplace=True,axis = 1)

## Remove Duplicates BUT with HN channels as priority
gm_im_table_50.sort_values('chan',ascending=False,inplace=True)
gm_im_table_50.drop_duplicates(subset=['evid','sta'],keep='first',inplace=True)
gm_im_table_50.sort_values('datetime',ascending=True,inplace=True)

## Remove possible error values
gm_im_table_50 = gm_im_table_50[gm_im_table_50["pSA_0.01"]>0]

## append additional station parameters
stations = pd.read_csv(RESIDUALS_DIR / "site_table_TRI_multiscale_group_TWI.csv", dtype={"sta": "string"})
stations.fillna(0,inplace=True)
stations_filt = pd.DataFrame()
stations_filt['sta'] = stations['sta']
stations_filt['TRI'] = stations['TRI']
stations_filt['TWI'] = stations['TWI']
stations_filt['geoage'] = stations['geoMA_min']
stations_filt['k0'] = stations['k0_mean']
stations_filt['precip'] = stations['precip']
stations_filt['group'] = stations['cregion_group']
gm_im_table_50 = gm_im_table_50.merge(stations_filt,on='sta',how='left')

# apply maxmimum distance filter
gm_im_table_50 = gm_im_table_50[gm_im_table_50.r_jb<=300].reset_index(drop=True)

## Apply filter on the data: Tectonic Classification
gm_im_table_50_crustal = gm_im_table_50[(gm_im_table_50.tect_class=='Crustal')&(gm_im_table_50.mag>=3.5)]
gm_im_table_50_slab = gm_im_table_50[(gm_im_table_50.tect_class=='Slab')&(gm_im_table_50.mag>=5)]
gm_im_table_50_interface = gm_im_table_50[(gm_im_table_50.tect_class=='Interface')&(gm_im_table_50.mag>=5)]
gm_im_table_50 = pd.concat([gm_im_table_50_crustal, gm_im_table_50_slab, gm_im_table_50_interface],axis=0)
gm_im_table_50 = gm_im_table_50.reset_index(drop = True)

# get event ids for crustal
evid_crus = gm_im_table_50_crustal['evid'].unique().tolist()

# get event ids for Mw > 5
evid_mw5 = gm_im_table_50[gm_im_table_50.mag>=6].reset_index(drop = True)['evid'].unique().tolist()

### 1.3.4 Model comparison, LPPDs and directory

In [ ]:
lppd_directory = SCM_DIR

# filename prefix
filename_NZ = f"NZGMDB_v4p3_dwbar"
filename_crus = f"NZGMDB_v4p3_dwbar_crus"
filename_sub = f"NZGMDB_v4p3_dwbar_sub"

# Pooled LPPDs
LPPDP_NZ_reldiff = pd.read_csv(lppd_directory/f'{filename_NZ}_LPPDP.csv')
LPPDP_crus_reldiff = pd.read_csv(lppd_directory/f'{filename_crus}_LPPDP.csv')
LPPDP_sub_reldiff = pd.read_csv(lppd_directory/f'{filename_sub}_LPPDP.csv')

# event-specific LPPDPs
frames = []
for i, period in enumerate(tqdm(pSAobs_T[:12], desc="Processing LPPDES periods")):
    LPPDES_temp = pd.read_csv(lppd_directory / f"{filename_NZ}_{period}_LPPDPES.csv")
    if i == 0:
        LPPDES = LPPDES_temp[["evid", "n_recs", period]].rename(columns={period: f"{period}"})
    else:
        temp = LPPDES_temp[["evid", period]].rename(columns={period: f"{period}"})
        LPPDES = LPPDES.merge(temp, on="evid", how="left")
order = LPPDES['evid']
gm_unique = gm_im_table_50[['evid','mag','tect_class']].drop_duplicates('evid')
LPPDES = LPPDES.merge(gm_unique, on='evid', how='left')

# pooled events LPPDPs
PO_LPPDE_NZ = pd.DataFrame()
PO_LPPDE_crus = pd.DataFrame()
PO_LPPDE_sub = pd.DataFrame()

# NZ-Wide
frames = []
for period in tqdm(pSAobs_T[:12], desc="Processing NZ"):
    pooled_ME_LPPDE = pd.read_csv(lppd_directory.joinpath(f"{filename_NZ}_LPPDE_E_{period}.csv"))
    pooled_MEA_LPPDE = pd.read_csv(lppd_directory.joinpath(f"{filename_NZ}_LPPDE_EA_{period}.csv"))
    pooled_MEAS_LPPDE = pd.read_csv(lppd_directory.joinpath(f"{filename_NZ}_LPPDE_EAS_{period}.csv"))
    pooled_MEASZ_LPPDE = pd.read_csv(lppd_directory.joinpath(f"{filename_NZ}_LPPDE_EASZ_{period}.csv"))
    pooled_MEAST_LPPDE = pd.read_csv(lppd_directory.joinpath(f"{filename_NZ}_LPPDE_EAST_{period}.csv"))
    pooled_MEASZT_LPPDE = pd.read_csv(lppd_directory.joinpath(f"{filename_NZ}_LPPDE_EASZT_{period}.csv"))

    evids = pooled_ME_LPPDE.columns.str.replace('z_', '')

    temp = pd.DataFrame({
        "evid": evids,
        f"E_{period}": -np.log(len(pooled_ME_LPPDE)) + logsumexp(pooled_ME_LPPDE, axis=0),
        f"EA_{period}": -np.log(len(pooled_MEA_LPPDE)) + logsumexp(pooled_MEA_LPPDE, axis=0),
        f"EAS_{period}": -np.log(len(pooled_MEAS_LPPDE)) + logsumexp(pooled_MEAS_LPPDE, axis=0),
        f"EASZ_{period}": -np.log(len(pooled_MEASZ_LPPDE)) + logsumexp(pooled_MEASZ_LPPDE, axis=0),
        f"EAST_{period}": -np.log(len(pooled_MEAST_LPPDE)) + logsumexp(pooled_MEAST_LPPDE, axis=0),
        f"EASZT_{period}": -np.log(len(pooled_MEASZT_LPPDE)) + logsumexp(pooled_MEASZT_LPPDE, axis=0),
    })
    frames.append(temp)
PO_LPPDE_NZ = pd.concat(frames).groupby("evid", as_index=False).first()
PO_LPPDE_NZ = PO_LPPDE_NZ.set_index('evid').loc[order].reset_index()

# Crustal
frames = []
for period in tqdm(pSAobs_T[:12], desc="Processing CRUS"):
    pooled_ME_LPPDE   = pd.read_csv(lppd_directory / f"{filename_crus}_LPPDE_E_{period}.csv")
    pooled_MEA_LPPDE  = pd.read_csv(lppd_directory / f"{filename_crus}_LPPDE_EA_{period}.csv")
    pooled_MEAS_LPPDE = pd.read_csv(lppd_directory / f"{filename_crus}_LPPDE_EAS_{period}.csv")
    pooled_MEASZ_LPPDE= pd.read_csv(lppd_directory / f"{filename_crus}_LPPDE_EASZ_{period}.csv")
    pooled_MEAST_LPPDE= pd.read_csv(lppd_directory / f"{filename_crus}_LPPDE_EAST_{period}.csv")
    pooled_MEASZT_LPPDE=pd.read_csv(lppd_directory / f"{filename_crus}_LPPDE_EASZT_{period}.csv")

    evids = pooled_ME_LPPDE.columns.str.replace('z_', '')

    temp = pd.DataFrame({
        "evid": evids,
        f"E_{period}":    -np.log(len(pooled_ME_LPPDE))   + logsumexp(pooled_ME_LPPDE, axis=0),
        f"EA_{period}":   -np.log(len(pooled_MEA_LPPDE))  + logsumexp(pooled_MEA_LPPDE, axis=0),
        f"EAS_{period}":  -np.log(len(pooled_MEAS_LPPDE)) + logsumexp(pooled_MEAS_LPPDE, axis=0),
        f"EASZ_{period}": -np.log(len(pooled_MEASZ_LPPDE))+ logsumexp(pooled_MEASZ_LPPDE, axis=0),
        f"EAST_{period}": -np.log(len(pooled_MEAST_LPPDE))+ logsumexp(pooled_MEAST_LPPDE, axis=0),
        f"EASZT_{period}":-np.log(len(pooled_MEASZT_LPPDE))+ logsumexp(pooled_MEASZT_LPPDE, axis=0),
    })
    frames.append(temp)
PO_LPPDE_crus = pd.concat(frames).groupby("evid", as_index=False).first()
PO_LPPDE_crus = PO_LPPDE_crus.set_index("evid").reindex(order).reset_index()

# subduction
frames = []
for period in tqdm(pSAobs_T[:12], desc="Processing SUB"):
    pooled_ME_LPPDE   = pd.read_csv(lppd_directory / f"{filename_sub}_LPPDE_E_{period}.csv")
    pooled_MEA_LPPDE  = pd.read_csv(lppd_directory / f"{filename_sub}_LPPDE_EA_{period}.csv")
    pooled_MEAS_LPPDE = pd.read_csv(lppd_directory / f"{filename_sub}_LPPDE_EAS_{period}.csv")
    pooled_MEASZ_LPPDE= pd.read_csv(lppd_directory / f"{filename_sub}_LPPDE_EASZ_{period}.csv")
    pooled_MEAST_LPPDE= pd.read_csv(lppd_directory / f"{filename_sub}_LPPDE_EAST_{period}.csv")
    pooled_MEASZT_LPPDE=pd.read_csv(lppd_directory / f"{filename_sub}_LPPDE_EASZT_{period}.csv")

    evids = pooled_ME_LPPDE.columns.str.replace('z_', '')

    temp = pd.DataFrame({
        "evid": evids,
        f"E_{period}":    -np.log(len(pooled_ME_LPPDE))   + logsumexp(pooled_ME_LPPDE, axis=0),
        f"EA_{period}":   -np.log(len(pooled_MEA_LPPDE))  + logsumexp(pooled_MEA_LPPDE, axis=0),
        f"EAS_{period}":  -np.log(len(pooled_MEAS_LPPDE)) + logsumexp(pooled_MEAS_LPPDE, axis=0),
        f"EASZ_{period}": -np.log(len(pooled_MEASZ_LPPDE))+ logsumexp(pooled_MEASZ_LPPDE, axis=0),
        f"EAST_{period}": -np.log(len(pooled_MEAST_LPPDE))+ logsumexp(pooled_MEAST_LPPDE, axis=0),
        f"EASZT_{period}":-np.log(len(pooled_MEASZT_LPPDE))+ logsumexp(pooled_MEASZT_LPPDE, axis=0),
    })
    frames.append(temp)
PO_LPPDE_sub = pd.concat(frames).groupby("evid", as_index=False).first()
PO_LPPDE_sub  = PO_LPPDE_sub.set_index("evid").reindex(order).reset_index()

# model comparisons - PSIS-LOO-CV
model_compare_NZ = []
model_compare_crus = []
model_compare_sub = []
for period in pSAobs_T[:12]:
    model_copmare_temp = pd.read_csv(lppd_directory.joinpath('%s_LOOCompare_%s.csv'%(filename_NZ,period)))
    model_copmare_temp= model_copmare_temp.rename(columns={"Unnamed: 0": "model"})
    model_compare_NZ.append(model_copmare_temp)
for period in pSAobs_T[:12]:
    model_copmare_temp = pd.read_csv(lppd_directory.joinpath('%s_LOOCompare_%s.csv'%(filename_crus,period)))
    model_copmare_temp= model_copmare_temp.rename(columns={"Unnamed: 0": "model"})
    model_compare_crus.append(model_copmare_temp)
for period in pSAobs_T[:12]:
    model_copmare_temp = pd.read_csv(lppd_directory.joinpath('%s_LOOCompare_%s.csv'%(filename_sub,period)))
    model_copmare_temp= model_copmare_temp.rename(columns={"Unnamed: 0": "model"})
    model_compare_sub.append(model_copmare_temp)

# 2.0 Traditional Stationary SCM Development

## 2.1 NZ-Wide Data

In [ ]:
# set parameters for fitting
estimator = "Matheron"
model = "exponential"
weight_fn = "exponential weighting"
weight_params = [5]
xmax_factor = 2.5

# list for storing summary
summary_pSA = []
lE_pSA = []; lE5_pSA =[]; lE95_pSA = []; lE5CI_pSA = []; lE95CI_pSA = []; stderr_lE_pSA = []; lEMed_pSA = []

for pID in tqdm(range(len(pSAobs_T)),desc = 'processing period: '):

    im_dwbar_pSA_files_filt = im_dwbar_pSA_files[pID][im_dwbar_pSA_files[pID].evid.isin(evid_crus)].reset_index(drop=True)

    period_col = pSAobs_T[pID]
    summary, df_n_obs, df_gamma, results = variofitmulti(
        df=im_dwbar_pSA_files_filt,
        values_col=period_col,
        index_col="evid",
        coord_cols=("sta_lat","sta_lon"),
        distance_type="geographic",
        max_distance=40,
        bin_size=5,
        estimator_type="Matheron",
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill= False,
        plot_single=False,
        plot_summary=False
    )

    # store parameters summary and standard errors
    summary_pSA.append(summary)
    lE_pSA.append(np.mean(summary.r))
    lE5_pSA.append(np.percentile(summary.r,2.5))
    lE95_pSA.append(np.percentile(summary.r,97.5))
    lEMed_pSA.append(np.median(summary.r))

    data_lE = (summary.r.values,)
    res_lE = bootstrap(data_lE, np.mean)
    stderr_lE_pSA.append(res_lE.standard_error)
    lE95CI_pSA.append(res_lE.confidence_interval.high)
    lE5CI_pSA.append(res_lE.confidence_interval.low)

## 2.2 Group 1: Wellington

In [ ]:
# set parameters for fitting
estimator = "Matheron"
model = "exponential"
weight_fn = "exponential weighting"
weight_params = [5]
xmax_factor = 2.5

# list for storing summary
summary_pSA_grp1 = []
lE_pSA_grp1 = []; lE5_pSA_grp1 =[]; lE95_pSA_grp1 = []; lE5CI_pSA_grp1 = []; lE95CI_pSA_grp1 = []; stderr_lE_pSA_grp1 = []; lEMed_pSA_grp1 = []

for pID in tqdm(range(len(pSAobs_T)),desc = 'processing period: '):

    im_dwbar_pSA_files_filt = im_dwbar_pSA_files_grp1[pID][im_dwbar_pSA_files_grp1[pID].evid.isin(evid_crus)].reset_index(drop=True)

    period_col = pSAobs_T[pID]
    summary, df_n_obs, df_gamma, results = variofitmulti(
        df=im_dwbar_pSA_files_grp1[pID],
        values_col=period_col,
        index_col="evid",
        coord_cols=("sta_lat","sta_lon"),
        distance_type="geographic",
        max_distance=40,
        bin_size=5,
        estimator_type="Matheron",
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill= False,
        plot_single=False,
        plot_summary=False
    )

    # store parameters summary and standard errors
    summary_pSA_grp1.append(summary)
    lE_pSA_grp1.append(np.mean(summary.r))
    lE5_pSA_grp1.append(np.percentile(summary.r,2.5))
    lE95_pSA_grp1.append(np.percentile(summary.r,97.5))
    lEMed_pSA_grp1.append(np.median(summary.r))

    data_lE = (summary.r.values,)
    res_lE = bootstrap(data_lE, np.mean)
    stderr_lE_pSA_grp1.append(res_lE.standard_error)
    lE95CI_pSA_grp1.append(res_lE.confidence_interval.high)
    lE5CI_pSA_grp1.append(res_lE.confidence_interval.low)

## 2.3 Group 2: Canterbury

In [ ]:
# set parameters for fitting
estimator = "Matheron"
model = "exponential"
weight_fn = "exponential weighting"
weight_params = [5]
xmax_factor = 2.5

# list for storing summary
summary_pSA_grp2 = []
lE_pSA_grp2 = []; lE5_pSA_grp2 =[]; lE95_pSA_grp2 = []; lE5CI_pSA_grp2 = []; lE95CI_pSA_grp2 = []; stderr_lE_pSA_grp2 = []; lEMed_pSA_grp2 = []

for pID in tqdm(range(len(pSAobs_T)),desc = 'processing period: '):

    im_dwbar_pSA_files_filt = im_dwbar_pSA_files_grp2[pID][im_dwbar_pSA_files_grp2[pID].evid.isin(evid_crus)].reset_index(drop=True)

    period_col = pSAobs_T[pID]
    summary, df_n_obs, df_gamma, results = variofitmulti(
        df=im_dwbar_pSA_files_filt,
        values_col=period_col,
        index_col="evid",
        coord_cols=("sta_lat","sta_lon"),
        distance_type="geographic",
        max_distance=40,
        bin_size=5,
        estimator_type="Matheron",
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill= False,
        plot_single=False,
        plot_summary=False
    )

    # store parameters summary and standard errors
    summary_pSA_grp2.append(summary)
    lE_pSA_grp2.append(np.mean(summary.r))
    lE5_pSA_grp2.append(np.percentile(summary.r,5))
    lE95_pSA_grp2.append(np.percentile(summary.r,95))
    lEMed_pSA_grp2.append(np.median(summary.r))

    data_lE = (summary.r.values,)
    res_lE = bootstrap(data_lE, np.mean)
    stderr_lE_pSA_grp2.append(res_lE.standard_error)
    lE95CI_pSA_grp2.append(res_lE.confidence_interval.high)
    lE5CI_pSA_grp2.append(res_lE.confidence_interval.low)

## 2.4 North Island Group

In [ ]:
# set parameters for fitting
estimator = "Matheron"
model = "exponential"
weight_fn = "exponential weighting"
weight_params = [5]
xmax_factor = 2.5

# list for storing summary
summary_pSA_grpn = []
lE_pSA_grpn = []; lE5_pSA_grpn =[]; lE95_pSA_grpn = []; lE5CI_pSA_grpn = []; lE95CI_pSA_grpn = []; stderr_lE_pSA_grpn = []; lEMed_pSA_grpn = []

for pID in tqdm(range(len(pSAobs_T)),desc = 'processing period: '):

    im_dwbar_pSA_files_filt = im_dwbar_pSA_files_grpn[pID][im_dwbar_pSA_files_grpn[pID].evid.isin(evid_crus)].reset_index(drop=True)

    period_col = pSAobs_T[pID]
    summary, df_n_obs, df_gamma, results = variofitmulti(
        df=im_dwbar_pSA_files_grpn[pID],
        values_col=period_col,
        index_col="evid",
        coord_cols=("sta_lat","sta_lon"),
        distance_type="geographic",
        max_distance=40,
        bin_size=5,
        estimator_type="Matheron",
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill= False,
        plot_single=False,
        plot_summary=False
    )

    # store parameters summary and standard errors
    summary_pSA_grpn.append(summary)
    lE_pSA_grpn.append(np.mean(summary.r))
    lE5_pSA_grpn.append(np.percentile(summary.r,5))
    lE95_pSA_grpn.append(np.percentile(summary.r,95))
    lEMed_pSA_grpn.append(np.median(summary.r))

    data_lE = (summary.r.values,)
    res_lE = bootstrap(data_lE, np.mean)
    stderr_lE_pSA_grpn.append(res_lE.standard_error)
    lE95CI_pSA_grpn.append(res_lE.confidence_interval.high)
    lE5CI_pSA_grpn.append(res_lE.confidence_interval.low)

## 2.5 South Island Group

In [ ]:
# set parameters for fitting
estimator = "Matheron"
model = "exponential"
weight_fn = "exponential weighting"
weight_params = [5]
xmax_factor = 2.5

# list for storing summary
summary_pSA_grps = []
lE_pSA_grps = []; lE5_pSA_grps =[]; lE95_pSA_grps = []; lE5CI_pSA_grps = []; lE95CI_pSA_grps = []; stderr_lE_pSA_grps = []; lEMed_pSA_grps = []

for pID in tqdm(range(len(pSAobs_T)),desc = 'processing period: '):

    im_dwbar_pSA_files_filt = im_dwbar_pSA_files_grps[pID][im_dwbar_pSA_files_grps[pID].evid.isin(evid_crus)].reset_index(drop=True)

    period_col = pSAobs_T[pID]
    summary, df_n_obs, df_gamma, results = variofitmulti(
        df=im_dwbar_pSA_files_grps[pID],
        values_col=period_col,
        index_col="evid",
        coord_cols=("sta_lat","sta_lon"),
        distance_type="geographic",
        max_distance=40,
        bin_size=5,
        estimator_type="Matheron",
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill= False,
        plot_single=False,
        plot_summary=False
    )

    # store parameters summary and standard errors
    summary_pSA_grps.append(summary)
    lE_pSA_grps.append(np.mean(summary.r))
    lE5_pSA_grps.append(np.percentile(summary.r,5))
    lE95_pSA_grps.append(np.percentile(summary.r,95))
    lEMed_pSA_grps.append(np.median(summary.r))

    data_lE = (summary.r.values,)
    res_lE = bootstrap(data_lE, np.mean)
    stderr_lE_pSA_grps.append(res_lE.standard_error)
    lE95CI_pSA_grps.append(res_lE.confidence_interval.high)
    lE5CI_pSA_grps.append(res_lE.confidence_interval.low)

## 2.6 Traditional Stationary SCM fit and evaluation

### 2.6.1 Model definition

In [ ]:
# Rampersad (2025) GM SCM
def GMCSM_RL25(T,dh,n_rec, cbb2021corr, location, plot = False):
    """Compute correlation coefficients and deviations based on NZ-Specific Model developed by Rampersad et al., 2025."""

    period = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    if location == 'NZ':

        b = lE_pSA

    elif location == 'GRP1':

        b = lE_pSA_grp1

    elif location == 'GRP2':

        b = lE_pSA_grp2

    elif location == 'GRPN':

        b = lE_pSA_grpn

    elif location == 'GRPS':

        b = lE_pSA_grps

    loc = period.index(T)

    # Compute correlation coefficient based on seperation distance in data
    p = np.exp(-cbb2021corr.h / b[loc])

    # Compute Fisher's Z-transformation
    z_p = 0.5*np.log((1+p)/(1-p))

    # Compute correlation deviation, e
    e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)

    # compute correlation mean and standard deviation
    e_mean = np.nanmean(e)
    e_std = np.nanstd(e)

    # 5-95% confidence
    data = (e,)
    res = bootstrap(data, np.mean)
    res_std = bootstrap(data, np.std)
    e_5 = res.confidence_interval.low
    e_95 = res.confidence_interval.high
    estd_5 = res_std.confidence_interval.low
    estd_95 = res_std.confidence_interval.high


    # create text string
    if T == 0.01:
        plot_str = 'PGA'
    else:
        plot_str = 'pSA = %0.2fs'%T

    # Plot results
    if plot:
        x = np.linspace(0,dh,100)
        p_dh = np.exp(-x/b[loc])
        p_std = np.square(1-p_dh)/np.sqrt(n_rec)
        fig, ax = plt.subplots(figsize = (8,5), dpi = 150, constrained_layout=True)
        im = ax.scatter(cbb2021corr.h,cbb2021corr.p_hat, c = e, vmin = -4, vmax = 4, cmap = 'bwr', edgecolors = 'black', s = 60)
        plt.plot(x,p_dh,'-k')
        plt.plot(x,p_dh - p_std,'--k')
        plt.plot(x,p_dh + p_std,'--k')
        plt.xlim(0, dh)
        plt.ylim(-1,1)
        plt.title('Rampersad (2025): %s'%plot_str)
        ax.set_ylabel(r'Correlation, $\hat{p}$ (%s)'%plot_str)
        ax.set_xlabel('Separation Distance, h (km)')
        # Display mean and standard deviation as text on plot
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label = 'Correlation Deviation, e',aspect = 20)
        plt.show(fig)

    return p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95

### 2.6.2 Correlation computation

In [ ]:
n_rec = 6
hmax = 25

cbb21_pSAobs_T = []; cbb21_pSAobs_T_grp1 = []; cbb21_pSAobs_T_grp2 = []; cbb21_pSAobs_T_grpn = []; cbb21_pSAobs_T_grps = []
for i in tqdm(range(len(pSAobs_T)), desc="pSA: "):
    # all data
    im_dwbar_pSA_files[i]['dw_bar'] = im_dwbar_pSA_files[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files[i],n_rec,hmax)
    cbb21_pSAobs_T.append(cbb21)

    # group 1: wellington
    im_dwbar_pSA_files_grp1[i]['dw_bar'] = im_dwbar_pSA_files_grp1[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grp1[i],n_rec,hmax)
    cbb21_pSAobs_T_grp1.append(cbb21)

    # group 2: christchurch
    im_dwbar_pSA_files_grp2[i]['dw_bar'] = im_dwbar_pSA_files_grp2[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grp2[i],n_rec,hmax)
    cbb21_pSAobs_T_grp2.append(cbb21)

    # group n: North Island
    im_dwbar_pSA_files_grpn[i]['dw_bar'] = im_dwbar_pSA_files_grpn[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grpn[i],n_rec,hmax)
    cbb21_pSAobs_T_grpn.append(cbb21)

    # group s: South Island
    im_dwbar_pSA_files_grps[i]['dw_bar'] = im_dwbar_pSA_files_grps[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grps[i],n_rec,hmax)
    cbb21_pSAobs_T_grps.append(cbb21)

### 2.6.3 Empirical correlation predictions

In [ ]:
## Rampersad 2025 Inputs
dh = hmax

## lists for storing information
RL25_e = []; RL25_e5 = []; RL25_e95 = []; RL25_e_std = []; RL25_e5_std = []; RL25_e95_std = []
RL25_e_grp1 = []; RL25_e5_grp1 = []; RL25_e95_grp1 = []; RL25_e_std_grp1 = []; RL25_e5_std_grp1 = []; RL25_e95_std_grp1 = []
RL25_e_grp2 = []; RL25_e5_grp2 = []; RL25_e95_grp2 = []; RL25_e_std_grp2 = []; RL25_e5_std_grp2 = []; RL25_e95_std_grp2 = []
RL25_e_grpn = []; RL25_e5_grpn = []; RL25_e95_grpn = []; RL25_e_std_grpn = []; RL25_e5_std_grpn = []; RL25_e95_std_grpn = []
RL25_e_grps = []; RL25_e5_grps = []; RL25_e95_grps= []; RL25_e_std_grps = []; RL25_e5_std_grps = []; RL25_e95_std_grps = []

for i in tqdm(range(len(pSA_T))):
    p, e, e_mean, e_std, e_5, e_95, estd_5, estd_95 = GMCSM_RL25(pSA_T[i], dh, n_rec, cbb21_pSAobs_T[i], 'NZ', plot = False)
    p_grp1, e_grp1, e_mean_grp1, e_std_grp1, e_5_grp1, e_95_grp1, estd_5_grp1, estd_95_grp1 = GMCSM_RL25(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_grp1[i], 'GRP1', plot = False)
    p_grp2, e_grp2, e_mean_grp2, e_std_grp2, e_5_grp2, e_95_grp2, estd_5_grp2, estd_95_grp2 = GMCSM_RL25(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_grp2[i], 'GRP2', plot = False)
    p_grpn, e_grpn, e_mean_grpn, e_std_grpn, e_5_grpn, e_95_grpn, estd_5_grpn, estd_95_grpn = GMCSM_RL25(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_grpn[i], 'GRPN', plot = False)
    p_grps, e_grps, e_mean_grps, e_std_grps, e_5_grps, e_95_grps, estd_5_grps, estd_95_grps = GMCSM_RL25(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_grps[i], 'GRPS', plot = False)

    # append values
    RL25_e.append(e_mean); RL25_e5.append(e_5); RL25_e95.append(e_95); RL25_e_std.append(e_std); RL25_e5_std.append(estd_5); RL25_e95_std.append(estd_95)

    RL25_e_grp1.append(e_mean_grp1); RL25_e5_grp1.append(e_5_grp1); RL25_e95_grp1.append(e_95_grp1); RL25_e_std_grp1.append(e_std_grp1); RL25_e5_std_grp1.append(estd_5_grp1); RL25_e95_std_grp1.append(estd_95_grp1)

    RL25_e_grp2.append(e_mean_grp2); RL25_e5_grp2.append(e_5_grp2); RL25_e95_grp2.append(e_95_grp2); RL25_e_std_grp2.append(e_std_grp2); RL25_e5_std_grp2.append(estd_5_grp2); RL25_e95_std_grp2.append(estd_95_grp2)

    RL25_e_grpn.append(e_mean_grpn); RL25_e5_grpn.append(e_5_grpn); RL25_e95_grpn.append(e_95_grpn); RL25_e_std_grpn.append(e_std_grpn); RL25_e5_std_grpn.append(estd_5_grpn); RL25_e95_std_grpn.append(estd_95_grpn)

    RL25_e_grps.append(e_mean_grps); RL25_e5_grps.append(e_5_grps); RL25_e95_grps.append(e_95_grps); RL25_e_std_grps.append(e_std_grps); RL25_e5_std_grps.append(estd_5_grps); RL25_e95_std_grps.append(estd_95_grps)

### 2.6.3 Evaluation plots

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(12,7), dpi=150, constrained_layout=True)

# central curves
ax[0].plot(pSA_T, RL25_e, '.-', label='NZ-Wide')
ax[0].plot(pSA_T, RL25_e_grp1, '.-', label='Wellington')
ax[0].plot(pSA_T, RL25_e_grp2, '.-', label='Canterbury')
ax[0].plot(pSA_T, RL25_e_grpn, '.-', label='North Island')
ax[0].plot(pSA_T, RL25_e_grps, '.-', label='South Island')
ax[1].plot(pSA_T, RL25_e_std, '.-', label='NZ-Wide')
ax[1].plot(pSA_T, RL25_e_std_grp1, '.-', label='Wellington')
ax[1].plot(pSA_T, RL25_e_std_grp2, '.-', label='Canterbury')
ax[1].plot(pSA_T, RL25_e_std_grpn, '.-', label='North Island')
ax[1].plot(pSA_T, RL25_e_std_grps, '.-', label='South Island')
# error bands (5–95%)
ax[0].fill_between(pSA_T, RL25_e5, RL25_e95, alpha=0.2, linewidth=0)
ax[0].fill_between(pSA_T, RL25_e5_grp1, RL25_e5_grp1, alpha=0.2, linewidth=0)
ax[0].fill_between(pSA_T, RL25_e5_grp2, RL25_e95_grp2, alpha=0.2, linewidth=0)
ax[0].fill_between(pSA_T, RL25_e5_grpn, RL25_e95_grpn, alpha=0.2, linewidth=0)
ax[0].fill_between(pSA_T, RL25_e5_grps, RL25_e95_grps, alpha=0.2, linewidth=0)
ax[1].fill_between(pSA_T, RL25_e5_std, RL25_e95_std, alpha=0.2, linewidth=0)
ax[1].fill_between(pSA_T, RL25_e5_std_grp1, RL25_e95_std_grp1, alpha=0.2, linewidth=0)
ax[1].fill_between(pSA_T, RL25_e5_std_grp2, RL25_e95_std_grp2, alpha=0.2, linewidth=0)
ax[1].fill_between(pSA_T, RL25_e5_std_grpn, RL25_e95_std_grpn, alpha=0.2, linewidth=0)
ax[1].fill_between(pSA_T, RL25_e5_std_grps, RL25_e95_std_grps, alpha=0.2, linewidth=0)
# baseline and formatting
ax[0].set_ylim(-3, 3)
ax[0].axhline(0, linestyle='--', color='black')
ax[1].set_ylim(1, 7)
for i in range(2):
    ax[i].set_xlim(0.01, 6)
    ax[i].set_xscale('log')
    ax[i].grid(which='both', linestyle='-', linewidth=0.5)
ax[0].set_ylabel(r'Corr. Deviation Mean, $\mu_e$')
ax[0].set_xlabel('Period, T (s)'); ax[1].set_xlabel('Period, T (s)')
ax[1].set_ylabel(r'Corr. Deviation Standard Deviation, $\sigma_{e}$')
ax[0].legend(loc = 'upper center', ncols = 5)
plt.show()


# 3.0 Example Correlation Analysis and Existing SCM Assessment

## 3.1 Define region and get stations

In [ ]:
wel_region = [174.5, 175.2,-41.5, -41]
chc_region = [172,173.5,-44,-43]

# Filter stations within Wellington region
stations_wel = stations[
    (stations['lon'] >= wel_region[0]) & (stations['lon'] <= wel_region[1]) &
    (stations['lat'] >= wel_region[2]) & (stations['lat'] <= wel_region[3])
].reset_index(drop=True)

# Filter stations within Christchurch region
stations_chc = stations[
    (stations['lon'] >= chc_region[0]) & (stations['lon'] <= chc_region[1]) &
    (stations['lat'] >= chc_region[2]) & (stations['lat'] <= chc_region[3])
].reset_index(drop=True)

# Regional groups
group1 = ['Group 1: Wellington']
group2 = ['Group 2: Canterbury']
group3 = ['Group 3: West North Island']
group4 = ['Group 4: East North Island']
group5 = ['Group 5: North South Island']
group6 = ['Group 6: South South Island']

grp1_sta = stations[stations.cregion_group.isin(group1)].sta.tolist()
grp2_sta = stations[stations.cregion_group.isin(group2)].sta.tolist()
grp3_sta = stations[stations.cregion_group.isin(group3)].sta.tolist()
grp4_sta = stations[stations.cregion_group.isin(group4)].sta.tolist()
grp5_sta = stations[stations.cregion_group.isin(group5)].sta.tolist()
grp6_sta = stations[stations.cregion_group.isin(group6)].sta.tolist()

# soil/rock sites
chc_rock_sites = ['D14C', 'GOVS', 'DHSS', 'LPCC', 'STKS', 'LPOC',
              'HALS', 'CMHS', 'CRLZ', 'HUNS', 'HVSC', 'MTPS',
              'MENS', 'GODS', 'SUMS', 'PARS']
chc_soil_sites = ['LINC', 'TPLC', 'RHSC', 'HHSS', 'CBGS', 'MORS', 'CHHC', 'CCCC',
              'REHS', 'CACS', 'PPHS', 'MPSS', 'SHLC', 'DALS', 'PRPC',
              'SMTC', 'OHSS', 'NNBS', 'NBLC']

wel_rock_sites = ['PIPS', 'POTS', 'WEMS', 'TFSS', 'VUWS', 'WEL','LHBS',
              'FKPS', 'TEPS', 'TRTS','NEWS', 'RQGS', 'SOMS', 'WNHS', 'WANS']
wel_soil_sites = ['WNKS', 'MISS', 'WNAS', 'SEAS', 'EBPS', 'LHRS', 'BMTS',
                  'PVCS', 'PGMS', 'LHES', 'LHUS','NBSS', 'FAIS', 'INSS', 'ARKS', 'WDAS']

## 3.2 Compute Non-stationary correlations

In [ ]:
n_rec = 6
hmax = 25

cbb21_pSAobs_T_chc = []; cbb21_pSAobs_T_wel = []

for i in tqdm(range(len(pSAobs_T)), desc="pSA: "):

    # group 1: wellington
    im_dwbar_temp = im_dwbar_pSA_files[i][im_dwbar_pSA_files[i].sta.isin(stations_wel.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_wel.append(cbb21)

    # group 2: christchurch
    im_dwbar_temp = im_dwbar_pSA_files[i][im_dwbar_pSA_files[i].sta.isin(stations_chc.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_chc.append(cbb21)

## 3.3 Correlation deviation mean ($\mu_e$) and standard deviation ($\sigma_e$)

### 3.3.1 All Data

In [ ]:
# Jayaram and Baker 2009 Inputs
case = 2

# lists for storing information
GH08_e, GH08_e5, GH08_e95, GH08_e_std, GH08_e5_std, GH08_e95_std = [], [], [], [], [], []
JB09_e, JB09_e5, JB09_e95, JB09_e_std, JB09_e5_std, JB09_e95_std = [], [], [], [], [], []
GA10_e, GA10_e5, GA10_e95, GA10_e_std, GA10_e5_std, GA10_e95_std = [], [], [], [], [], []
EI12_e, EI12_e5, EI12_e95, EI12_e_std, EI12_e5_std, EI12_e95_std = [], [], [], [], [], []
LB13_e, LB13_e5, LB13_e95, LB13_e_std, LB13_e5_std, LB13_e95_std = [], [], [], [], [], []
HM19_e, HM19_e5, HM19_e95, HM19_e_std, HM19_e5_std, HM19_e95_std = [], [], [], [], [], []

for i in tqdm(range(len(pSA_T[:12]))):
    # Goda and Hong 2008
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_GH08(pSA_T[i], dh, n_rec, cbb21_pSAobs_T[i], plot=False)
    GH08_e.append(e_mean); GH08_e5.append(e5); GH08_e95.append(e95)
    GH08_e_std.append(e_std); GH08_e5_std.append(estd_5); GH08_e95_std.append(estd_95)

    # Jayaram and Baker 2009
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_JB09(case, pSA_T[i], dh, n_rec, cbb21_pSAobs_T[i], plot=False)
    JB09_e.append(e_mean); JB09_e5.append(e5); JB09_e95.append(e95)
    JB09_e_std.append(e_std); JB09_e5_std.append(estd_5); JB09_e95_std.append(estd_95)

    # Goda and Atkinson 2010
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_GA10(pSA_T[i], dh, n_rec, cbb21_pSAobs_T[i], plot=False)
    GA10_e.append(e_mean); GA10_e5.append(e5); GA10_e95.append(e95)
    GA10_e_std.append(e_std); GA10_e5_std.append(estd_5); GA10_e95_std.append(estd_95)

    # Esposito and Iervolino 2012
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_EI12(pSA_T[i], dh, n_rec, cbb21_pSAobs_T[i], plot=False)
    EI12_e.append(e_mean); EI12_e5.append(e5); EI12_e95.append(e95)
    EI12_e_std.append(e_std); EI12_e5_std.append(estd_5); EI12_e95_std.append(estd_95)

    # Loth and Baker 2013
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_LB13(pSA_T[i], pSA_T[i], dh, n_rec, cbb21_pSAobs_T[i], plot=False)
    LB13_e.append(e_mean); LB13_e5.append(e5); LB13_e95.append(e95)
    LB13_e_std.append(e_std); LB13_e5_std.append(estd_5); LB13_e95_std.append(estd_95)

    # Heresi and Miranda 2019
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_HM19(pSA_T[i], dh, n_rec, cbb21_pSAobs_T[i], plot=False)
    HM19_e.append(e_mean); HM19_e5.append(e5); HM19_e95.append(e95)
    HM19_e_std.append(e_std); HM19_e5_std.append(estd_5); HM19_e95_std.append(estd_95)

# Bodenmann et al., 2023
eE_mean_BM23, eEA_mean_BM23, eEAS_mean_BM23, eE_std_BM23, eEA_std_BM23, eEAS_std_BM23, eE_5_BM23, eE_95_BM23, eEstd_5_BM23, eEstd_95_BM23, eEA_5_BM23, eEA_95_BM23, eEAstd_5_BM23, eEAstd_95_BM23, eEAS_5_BM23, eEAS_95_BM23, eEASstd_5_BM23, eEASstd_95_BM23 = BM23_CBB21(cbb21_pSAobs_T, period_BM23, BM23_parameters)

### 3.3.2 Christchurch

In [ ]:
# lists for storing information
GH08_e_chc, GH08_e5_chc, GH08_e95_chc, GH08_e_std_chc, GH08_e5_std_chc, GH08_e95_std_chc = [], [], [], [], [], []
JB09_e_chc, JB09_e5_chc, JB09_e95_chc, JB09_e_std_chc, JB09_e5_std_chc, JB09_e95_std_chc = [], [], [], [], [], []
GA10_e_chc, GA10_e5_chc, GA10_e95_chc, GA10_e_std_chc, GA10_e5_std_chc, GA10_e95_std_chc = [], [], [], [], [], []
EI12_e_chc, EI12_e5_chc, EI12_e95_chc, EI12_e_std_chc, EI12_e5_std_chc, EI12_e95_std_chc = [], [], [], [], [], []
LB13_e_chc, LB13_e5_chc, LB13_e95_chc, LB13_e_std_chc, LB13_e5_std_chc, LB13_e95_std_chc = [], [], [], [], [], []
HM19_e_chc, HM19_e5_chc, HM19_e95_chc, HM19_e_std_chc, HM19_e5_std_chc, HM19_e95_std_chc = [], [], [], [], [], []

for i in tqdm(range(len(pSA_T[:12]))):
    # Goda and Hong 2008
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_GH08(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_chc[i], plot=False)
    GH08_e_chc.append(e_mean); GH08_e5_chc.append(e5); GH08_e95_chc.append(e95)
    GH08_e_std_chc.append(e_std); GH08_e5_std_chc.append(estd_5); GH08_e95_std_chc.append(estd_95)

    # Jayaram and Baker 2009
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_JB09(case, pSA_T[i], dh, n_rec, cbb21_pSAobs_T_chc[i], plot=False)
    JB09_e_chc.append(e_mean); JB09_e5_chc.append(e5); JB09_e95_chc.append(e95)
    JB09_e_std_chc.append(e_std); JB09_e5_std_chc.append(estd_5); JB09_e95_std_chc.append(estd_95)

    # Goda and Atkinson 2010
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_GA10(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_chc[i], plot=False)
    GA10_e_chc.append(e_mean); GA10_e5_chc.append(e5); GA10_e95_chc.append(e95)
    GA10_e_std_chc.append(e_std); GA10_e5_std_chc.append(estd_5); GA10_e95_std_chc.append(estd_95)

    # Esposito and Iervolino 2012
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_EI12(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_chc[i], plot=False)
    EI12_e_chc.append(e_mean); EI12_e5_chc.append(e5); EI12_e95_chc.append(e95)
    EI12_e_std_chc.append(e_std); EI12_e5_std_chc.append(estd_5); EI12_e95_std_chc.append(estd_95)

    # Loth and Baker 2013
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_LB13(pSA_T[i], pSA_T[i], dh, n_rec, cbb21_pSAobs_T_chc[i], plot=False)
    LB13_e_chc.append(e_mean); LB13_e5_chc.append(e5); LB13_e95_chc.append(e95)
    LB13_e_std_chc.append(e_std); LB13_e5_std_chc.append(estd_5); LB13_e95_std_chc.append(estd_95)

    # Heresi and Miranda 2019
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_HM19(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_chc[i], plot=False)
    HM19_e_chc.append(e_mean); HM19_e5_chc.append(e5); HM19_e95_chc.append(e95)
    HM19_e_std_chc.append(e_std); HM19_e5_std_chc.append(estd_5); HM19_e95_std_chc.append(estd_95)

# Bodenmann et al., 2023
eE_mean_BM23_chc, eEA_mean_BM23_chc, eEAS_mean_BM23_chc, eE_std_BM23_chc, eEA_std_BM23_chc, eEAS_std_BM23_chc, eE_5_BM23_chc, eE_95_BM23_chc, eEstd_5_BM23_chc, eEstd_95_BM23_chc, eEA_5_BM23_chc, eEA_95_BM23_chc, eEAstd_5_BM23_chc, eEAstd_95_BM23_chc, eEAS_5_BM23_chc, eEAS_95_BM23_chc, eEASstd_5_BM23_chc, eEASstd_95_BM23_chc = BM23_CBB21(cbb21_pSAobs_T_chc, period_BM23, BM23_parameters)

### 3.3.3 Wellington

In [ ]:
# lists for storing information
GH08_e_wel, GH08_e5_wel, GH08_e95_wel, GH08_e_std_wel, GH08_e5_std_wel, GH08_e95_std_wel = [], [], [], [], [], []
JB09_e_wel, JB09_e5_wel, JB09_e95_wel, JB09_e_std_wel, JB09_e5_std_wel, JB09_e95_std_wel = [], [], [], [], [], []
GA10_e_wel, GA10_e5_wel, GA10_e95_wel, GA10_e_std_wel, GA10_e5_std_wel, GA10_e95_std_wel = [], [], [], [], [], []
EI12_e_wel, EI12_e5_wel, EI12_e95_wel, EI12_e_std_wel, EI12_e5_std_wel, EI12_e95_std_wel = [], [], [], [], [], []
LB13_e_wel, LB13_e5_wel, LB13_e95_wel, LB13_e_std_wel, LB13_e5_std_wel, LB13_e95_std_wel = [], [], [], [], [], []
HM19_e_wel, HM19_e5_wel, HM19_e95_wel, HM19_e_std_wel, HM19_e5_std_wel, HM19_e95_std_wel = [], [], [], [], [], []

for i in tqdm(range(len(pSA_T[:12]))):
    # Goda and Hong 2008
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_GH08(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_wel[i], plot=False)
    GH08_e_wel.append(e_mean); GH08_e5_wel.append(e5); GH08_e95_wel.append(e95)
    GH08_e_std_wel.append(e_std); GH08_e5_std_wel.append(estd_5); GH08_e95_std_wel.append(estd_95)

    # Jayaram and Baker 2009
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_JB09(1, pSA_T[i], dh, n_rec, cbb21_pSAobs_T_wel[i], plot=False)
    JB09_e_wel.append(e_mean); JB09_e5_wel.append(e5); JB09_e95_wel.append(e95)
    JB09_e_std_wel.append(e_std); JB09_e5_std_wel.append(estd_5); JB09_e95_std_wel.append(estd_95)

    # Goda and Atkinson 2010
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_GA10(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_wel[i], plot=False)
    GA10_e_wel.append(e_mean); GA10_e5_wel.append(e5); GA10_e95_wel.append(e95)
    GA10_e_std_wel.append(e_std); GA10_e5_std_wel.append(estd_5); GA10_e95_std_wel.append(estd_95)

    # Esposito and Iervolino 2012
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_EI12(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_wel[i], plot=False)
    EI12_e_wel.append(e_mean); EI12_e5_wel.append(e5); EI12_e95_wel.append(e95)
    EI12_e_std_wel.append(e_std); EI12_e5_std_wel.append(estd_5); EI12_e95_std_wel.append(estd_95)

    # Loth and Baker 2013
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_LB13(pSA_T[i], pSA_T[i], dh, n_rec, cbb21_pSAobs_T_wel[i], plot=False)
    LB13_e_wel.append(e_mean); LB13_e5_wel.append(e5); LB13_e95_wel.append(e95)
    LB13_e_std_wel.append(e_std); LB13_e5_std_wel.append(estd_5); LB13_e95_std_wel.append(estd_95)

    # Heresi and Miranda 2019
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = GMCSM_HM19(pSA_T[i], dh, n_rec, cbb21_pSAobs_T_wel[i], plot=False)
    HM19_e_wel.append(e_mean); HM19_e5_wel.append(e5); HM19_e95_wel.append(e95)
    HM19_e_std_wel.append(e_std); HM19_e5_std_wel.append(estd_5); HM19_e95_std_wel.append(estd_95)

# Bodenmann et al., 2023
    eE_mean_BM23_wel, eEA_mean_BM23_wel, eEAS_mean_BM23_wel, eE_std_BM23_wel, eEA_std_BM23_wel, eEAS_std_BM23_wel, eE_5_BM23_wel, eE_95_BM23_wel, eEstd_5_BM23_wel, eEstd_95_BM23_wel, eEA_5_BM23_wel, eEA_95_BM23_wel, eEAstd_5_BM23_wel, eEAstd_95_BM23_wel, eEAS_5_BM23_wel, eEAS_95_BM23_wel, eEASstd_5_BM23_wel, eEASstd_95_BM23_wel = BM23_CBB21(cbb21_pSAobs_T_wel, period_BM23, BM23_parameters)

## 3.4 Correlation Deviation Mean and Standard Deviation Summary plots

### 3.4.1 Wout Confidence Intervals

In [ ]:
pSA_T_array = np.array(pSA_T[:12])

# all data
maskGH08 = ~np.isnan(GH08_e)
GH08_e = np.array(GH08_e)
GH08_e_std = np.array(GH08_e_std)
GH08_e5 =  np.array(GH08_e5)
GH08_e95 = np.array(GH08_e95)
GH08_e5_std = np.array(GH08_e5_std)
GH08_e95_std = np.array(GH08_e95_std)

maskGA10 = ~np.isnan(GA10_e)
GA10_e = np.array(GA10_e)
GA10_e_std = np.array(GA10_e_std)
GA10_e5 =  np.array(GA10_e5)
GA10_e95 = np.array(GA10_e95)
GA10_e5_std = np.array(GA10_e5_std)
GA10_e95_std = np.array(GA10_e95_std)

maskEI12 = ~np.isnan(EI12_e)
EI12_e = np.array(EI12_e)
EI12_e_std = np.array(EI12_e_std)
EI12_e5 =  np.array(EI12_e5)
EI12_e95 = np.array(EI12_e95)
EI12_e5_std = np.array(EI12_e5_std)
EI12_e95_std = np.array(EI12_e95_std)

maskHM19 = ~np.isnan(HM19_e)
HM19_e = np.array(HM19_e)
HM19_e_std = np.array(HM19_e_std)
HM19_e5 =  np.array(HM19_e5)
HM19_e95 = np.array(HM19_e95)
HM19_e5_std = np.array(HM19_e5_std)
HM19_e95_std = np.array(HM19_e95_std)

# christchurch
maskGH08_chc = ~np.isnan(GH08_e_chc)
GH08_e_chc = np.array(GH08_e_chc)
GH08_e_std_chc = np.array(GH08_e_std_chc)
GH08_e5_chc =  np.array(GH08_e5_chc)
GH08_e95_chc = np.array(GH08_e95_chc)
GH08_e5_std_chc = np.array(GH08_e5_std_chc)
GH08_e95_std_chc = np.array(GH08_e95_std_chc)

maskGA10_chc = ~np.isnan(GA10_e_chc)
GA10_e_chc = np.array(GA10_e_chc)
GA10_e_std_chc = np.array(GA10_e_std_chc)
GA10_e5_chc =  np.array(GA10_e5_chc)
GA10_e95_chc = np.array(GA10_e95_chc)
GA10_e5_std_chc = np.array(GA10_e5_std_chc)
GA10_e95_std_chc = np.array(GA10_e95_std_chc)

maskEI12_chc = ~np.isnan(EI12_e_chc)
EI12_e_chc = np.array(EI12_e_chc)
EI12_e_std_chc = np.array(EI12_e_std_chc)
EI12_e5_chc =  np.array(EI12_e5_chc)
EI12_e95_chc = np.array(EI12_e95_chc)
EI12_e5_std_chc = np.array(EI12_e5_std_chc)
EI12_e95_std_chc = np.array(EI12_e95_std_chc)

maskHM19_chc = ~np.isnan(HM19_e_chc)
HM19_e_chc = np.array(HM19_e_chc)
HM19_e_std_chc = np.array(HM19_e_std_chc)
HM19_e5_chc =  np.array(HM19_e5_chc)
HM19_e95_chc = np.array(HM19_e95_chc)
HM19_e5_std_chc = np.array(HM19_e5_std_chc)
HM19_e95_std_chc = np.array(HM19_e95_std_chc)

# wellington
maskGH08_wel = ~np.isnan(GH08_e_wel)
GH08_e_wel = np.array(GH08_e_wel)
GH08_e_std_wel = np.array(GH08_e_std_wel)
GH08_e5_wel =  np.array(GH08_e5_wel)
GH08_e95_wel = np.array(GH08_e95_wel)
GH08_e5_std_wel = np.array(GH08_e5_std_wel)
GH08_e95_std_wel = np.array(GH08_e95_std_wel)

maskGA10_wel = ~np.isnan(GA10_e_wel)
GA10_e_wel = np.array(GA10_e_wel)
GA10_e_std_wel = np.array(GA10_e_std_wel)
GA10_e5_wel =  np.array(GA10_e5_wel)
GA10_e95_wel = np.array(GA10_e95_wel)
GA10_e5_std_wel = np.array(GA10_e5_std_wel)
GA10_e95_std_wel = np.array(GA10_e95_std_wel)

maskEI12_wel = ~np.isnan(EI12_e_wel)
EI12_e_wel = np.array(EI12_e_wel)
EI12_e_std_wel = np.array(EI12_e_std_wel)
EI12_e5_wel =  np.array(EI12_e5_wel)
EI12_e95_wel = np.array(EI12_e95_wel)
EI12_e5_std_wel = np.array(EI12_e5_std_wel)
EI12_e95_std_wel = np.array(EI12_e95_std_wel)

maskHM19_wel = ~np.isnan(HM19_e_wel)
HM19_e_wel = np.array(HM19_e_wel)
HM19_e_std_wel = np.array(HM19_e_std_wel)
HM19_e5_wel =  np.array(HM19_e5_wel)
HM19_e95_wel = np.array(HM19_e95_wel)
HM19_e5_std_wel = np.array(HM19_e5_std_wel)
HM19_e95_std_wel = np.array(HM19_e95_std_wel)

fig, ax = plt.subplots(2, 3, figsize=(10, 5.5), dpi=200, constrained_layout=True)
# all data
ax[0,0].plot(pSA_T_array[maskGH08], GH08_e[maskGH08], '.-', label = 'GH08', lw = 1)
ax[0,0].plot(pSA_T_array, JB09_e, '.-', label = 'JB09', lw = 1)
ax[0,0].plot(pSA_T_array[maskGA10], GA10_e[maskGA10], '.-', label = 'GA10', lw = 1)
ax[0,0].plot(pSA_T_array[maskEI12], EI12_e[maskEI12], '.-', label = 'EI12', lw = 1)
ax[0,0].plot(pSA_T_array, LB13_e, '.-', label = 'LB13', lw = 1)
ax[0,0].plot(pSA_T_array[maskHM19], HM19_e[maskHM19], '.-', label = 'EI12', lw = 1)
ax[0,0].plot(period_BM23, eE_mean_BM23, '.-', label = 'BM23 (E)', lw = 1)
ax[0,0].plot(period_BM23, eEA_mean_BM23, '.-', label = 'BM23 (EA)', lw = 1)
ax[0,0].plot(period_BM23, eEAS_mean_BM23, '.-', label = 'BM23 (EAS)', lw = 1)

ax[1,0].plot(pSA_T_array[maskGH08], GH08_e_std[maskGH08], '.-', label = 'GH08', lw = 1)
ax[1,0].plot(pSA_T_array, JB09_e_std, '.-', label = 'JB09', lw = 1)
ax[1,0].plot(pSA_T_array[maskGA10], GA10_e_std[maskGA10], '.-', label = 'GA10', lw = 1)
ax[1,0].plot(pSA_T_array[maskEI12], EI12_e_std[maskEI12], '.-', label = 'EI12', lw = 1)
ax[1,0].plot(pSA_T_array, LB13_e_std, '.-', label = 'LB13', lw = 1)
ax[1,0].plot(pSA_T_array[maskHM19], HM19_e_std[maskHM19], '.-', label = 'EI12', lw = 1)
ax[1,0].plot(period_BM23, eE_std_BM23, '.-', label = 'BM23 (E)', lw = 1)
ax[1,0].plot(period_BM23, eEA_std_BM23, '.-', label = 'BM23 (EA)', lw = 1)
ax[1,0].plot(period_BM23, eEAS_std_BM23, '.-', label = 'BM23 (EAS)', lw = 1)

# christchurch
ax[0,1].plot(pSA_T_array[maskGH08_chc], GH08_e_chc[maskGH08_chc], '.-', label = 'GH08', lw = 1)
ax[0,1].plot(pSA_T_array, JB09_e_chc, '.-', label = 'JB09', lw = 1)
ax[0,1].plot(pSA_T_array[maskGA10_chc], GA10_e_chc[maskGA10_chc], '.-', label = 'GA10', lw = 1)
ax[0,1].plot(pSA_T_array[maskEI12_chc], EI12_e_chc[maskEI12_chc], '.-', label = 'EI12', lw = 1)
ax[0,1].plot(pSA_T_array, LB13_e_chc, '.-', label = 'LB13', lw = 1)
ax[0,1].plot(pSA_T_array[maskHM19_chc], HM19_e_chc[maskHM19_chc], '.-', label = 'HM19', lw = 1)
ax[0,1].plot(period_BM23, eE_mean_BM23_chc, '.-', label = 'BM23 (E)', lw = 1)
ax[0,1].plot(period_BM23, eEA_mean_BM23_chc, '.-', label = 'BM23 (EA)', lw = 1)
ax[0,1].plot(period_BM23, eEAS_mean_BM23_chc, '.-', label = 'BM23 (EAS)', lw = 1)

ax[1,1].plot(pSA_T_array[maskGH08_chc], GH08_e_std_chc[maskGH08_chc], '.-', label = 'GH08', lw = 1)
ax[1,1].plot(pSA_T_array, JB09_e_std_chc, '.-', label = 'JB09', lw = 1)
ax[1,1].plot(pSA_T_array[maskGA10_chc], GA10_e_std_chc[maskGA10_chc], '.-', label = 'GA10', lw = 1)
ax[1,1].plot(pSA_T_array[maskEI12_chc], EI12_e_std_chc[maskEI12_chc], '.-', label = 'EI12', lw = 1)
ax[1,1].plot(pSA_T_array, LB13_e_std_chc, '.-', label = 'LB13', lw = 1)
ax[1,1].plot(pSA_T_array[maskHM19_chc], HM19_e_std_chc[maskHM19_chc], '.-', label = 'HM19', lw = 1)
ax[1,1].plot(period_BM23, eE_std_BM23_chc, '.-', label = 'BM23 (E)', lw = 1)
ax[1,1].plot(period_BM23, eEA_std_BM23_chc, '.-', label = 'BM23 (EA)', lw = 1)
ax[1,1].plot(period_BM23, eEAS_std_BM23_chc, '.-', label = 'BM23 (EAS)', lw = 1)

# wellington
ax[0,2].plot(pSA_T_array[maskGH08_wel], GH08_e_wel[maskGH08_wel], '.-', label='GH08', lw = 1)
ax[0,2].plot(pSA_T_array, JB09_e_wel, '.-', label='JB09', lw = 1)
ax[0,2].plot(pSA_T_array[maskGA10_wel], GA10_e_wel[maskGA10_wel], '.-', label='GA10', lw = 1)
ax[0,2].plot(pSA_T_array[maskEI12_wel], EI12_e_wel[maskEI12_wel], '.-', label='EI12', lw = 1)
ax[0,2].plot(pSA_T_array, LB13_e_wel, '.-', label = 'LB13', lw = 1)
ax[0,2].plot(pSA_T_array[maskHM19_wel], HM19_e_wel[maskHM19_wel], '.-', label='HM19', lw = 1)
ax[0,2].plot(period_BM23, eE_mean_BM23_wel, '.-', label='BM23 (E)', lw = 1)
ax[0,2].plot(period_BM23, eEA_mean_BM23_wel, '.-', label='BM23 (EA)', lw = 1)
ax[0,2].plot(period_BM23, eEAS_mean_BM23_wel, '.-', label='BM23 (EAS)', lw = 1)

ax[1,2].plot(pSA_T_array[maskGH08_wel], GH08_e_std_wel[maskGH08_wel], '.-', label='GH08', lw = 1)
ax[1,2].plot(pSA_T_array, JB09_e_std_wel, '.-', label='JB09', lw = 1)
ax[1,2].plot(pSA_T_array[maskGA10_wel], GA10_e_std_wel[maskGA10_wel], '.-', label='GA10', lw = 1)
ax[1,2].plot(pSA_T_array[maskEI12_wel], EI12_e_std_wel[maskEI12_wel], '.-', label='EI12', lw = 1)
ax[1,2].plot(pSA_T_array, LB13_e_std_wel, '.-', label = 'LB13', lw = 1)
ax[1,2].plot(pSA_T_array[maskHM19_wel], HM19_e_std_wel[maskHM19_wel], '.-', label='HM19', lw = 1)
ax[1,2].plot(period_BM23, eE_std_BM23_wel, '.-', label='B23 (E)', lw = 1)
ax[1,2].plot(period_BM23, eEA_std_BM23_wel, '.-', label='B23 (EA)', lw = 1)
ax[1,2].plot(period_BM23, eEAS_std_BM23_wel, '.-', label='B23 (EAS)', lw = 1)

for i in range(2):
    for j in range(3):
        ax[i,j].set_xlim([0.01, 6])
        ax[i,j].set_xscale('log')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[0,j].set_ylim([-3, 3])
        ax[1,j].set_ylim([1.5, 6.5])
        ax[i,j].plot([0.01, 6], [0, 0], '--k')
        ax[1,j].set_xlabel('Period, T (s)', fontsize=16)
        ax[i,j].tick_params(axis='both', labelsize=14)

ax[0,0].set_title('NZ GMDB', fontsize=14)
ax[0,1].set_title('Christchurch', fontsize=14)
ax[0,2].set_title('Wellington', fontsize=14)
ax[0,0].set_ylabel(r'$\mu_e$', fontsize=16)
ax[1,0].set_ylabel(r'$\sigma_{e}$', fontsize=16)
ax[1,2].legend(loc = 'lower left', ncol=2, fontsize=11.5)
# handles, labels = ax[0,0].get_legend_handles_labels()
# fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.10), fontsize=8, ncol=10)
plt.show()

### 3.4.2 With Confidence Intervals

In [ ]:

fig, ax = plt.subplots(2, 3, figsize=(12, 7), dpi=150, constrained_layout=True)
# all data
ax[0,0].plot(pSA_T_array[maskGH08], GH08_e[maskGH08], '.-', label = 'GH08', lw = 1)
ax[0,0].plot(pSA_T_array, JB09_e, '.-', label = 'JB09', lw = 1)
ax[0,0].plot(pSA_T_array[maskGA10], GA10_e[maskGA10], '.-', label = 'GA10', lw = 1)
ax[0,0].plot(pSA_T_array[maskEI12], EI12_e[maskEI12], '.-', label = 'EI12', lw = 1)
ax[0,0].plot(pSA_T_array, LB13_e, '.-', label = 'LB13', lw = 1)
ax[0,0].plot(pSA_T_array[maskHM19], HM19_e[maskHM19], '.-', label = 'EI12', lw = 1)
ax[0,0].plot(period_BM23, eE_mean_BM23, '.-', label = 'BM23 (E)', lw = 1)
ax[0,0].plot(period_BM23, eEA_mean_BM23, '.-', label = 'BM23 (EA)', lw = 1)
ax[0,0].plot(period_BM23, eEAS_mean_BM23, '.-', label = 'BM23 (EAS)', lw = 1)
ax[0,0].fill_between(pSA_T_array[maskGH08], GH08_e5[maskGH08],  GH08_e95[maskGH08], alpha=0.2, linewidth=0)
ax[0,0].fill_between(pSA_T_array, JB09_e5,  JB09_e95, alpha=0.2, linewidth=0)
ax[0,0].fill_between(pSA_T_array[maskGA10], GA10_e5[maskGA10],  GA10_e95[maskGA10], alpha=0.2, linewidth=0)
ax[0,0].fill_between(pSA_T_array[maskEI12], EI12_e5[maskEI12],  EI12_e95[maskEI12], alpha=0.2, linewidth=0)
ax[0,0].fill_between(pSA_T_array, LB13_e5,  LB13_e95, alpha=0.2, linewidth=0)
ax[0,0].fill_between(pSA_T_array[maskHM19], HM19_e5[maskHM19],  HM19_e95[maskHM19], alpha=0.2, linewidth=0)
ax[0,0].fill_between(period_BM23, eEA_5_BM23,  eEA_95_BM23, alpha=0.2, linewidth=0)
ax[0,0].fill_between(period_BM23, eEAS_5_BM23,  eEAS_95_BM23, alpha=0.2, linewidth=0)

ax[1,0].plot(pSA_T_array[maskGH08], GH08_e_std[maskGH08], '.-', label = 'GH08', lw = 1)
ax[1,0].plot(pSA_T_array, JB09_e_std, '.-', label = 'JB09', lw = 1)
ax[1,0].plot(pSA_T_array[maskGA10], GA10_e_std[maskGA10], '.-', label = 'GA10', lw = 1)
ax[1,0].plot(pSA_T_array[maskEI12], EI12_e_std[maskEI12], '.-', label = 'EI12', lw = 1)
ax[1,0].plot(pSA_T_array, LB13_e_std, '.-', label = 'LB13', lw = 1)
ax[1,0].plot(pSA_T_array[maskHM19], HM19_e_std[maskHM19], '.-', label = 'EI12', lw = 1)
ax[1,0].plot(period_BM23, eE_std_BM23, '.-', label = 'BM23 (E)', lw = 1)
ax[1,0].plot(period_BM23, eEA_std_BM23, '.-', label = 'BM23 (EA)', lw = 1)
ax[1,0].plot(period_BM23, eEAS_std_BM23, '.-', label = 'BM23 (EAS)', lw = 1)
ax[1,0].fill_between(pSA_T_array[maskGH08], GH08_e5_std[maskGH08],  GH08_e95_std[maskGH08], alpha=0.2, linewidth=0)
ax[1,0].fill_between(pSA_T_array, JB09_e5_std,  JB09_e95_std, alpha=0.2, linewidth=0)
ax[1,0].fill_between(pSA_T_array[maskGA10], GA10_e5_std[maskGA10],  GA10_e95_std[maskGA10], alpha=0.2, linewidth=0)
ax[1,0].fill_between(pSA_T_array[maskEI12], EI12_e5_std[maskEI12],  EI12_e95_std[maskEI12], alpha=0.2, linewidth=0)
ax[1,0].fill_between(pSA_T_array, LB13_e5_std,  LB13_e95_std, alpha=0.2, linewidth=0)
ax[1,0].fill_between(pSA_T_array[maskHM19], HM19_e5_std[maskHM19],  HM19_e95_std[maskHM19], alpha=0.2, linewidth=0)
ax[1,0].fill_between(period_BM23, eEAstd_5_BM23,  eEAstd_95_BM23, alpha=0.2, linewidth=0)
ax[1,0].fill_between(period_BM23, eEASstd_5_BM23,  eEASstd_95_BM23, alpha=0.2, linewidth=0)

# christchurch
ax[0,1].plot(pSA_T_array[maskGH08_chc], GH08_e_chc[maskGH08_chc], '.-', label = 'GH08', lw = 1)
ax[0,1].plot(pSA_T_array, JB09_e_chc, '.-', label = 'JB09', lw = 1)
ax[0,1].plot(pSA_T_array[maskGA10_chc], GA10_e_chc[maskGA10_chc], '.-', label = 'GA10', lw = 1)
ax[0,1].plot(pSA_T_array[maskEI12_chc], EI12_e_chc[maskEI12_chc], '.-', label = 'EI12', lw = 1)
ax[0,1].plot(pSA_T_array, LB13_e_chc, '.-', label = 'LB13', lw = 1)
ax[0,1].plot(pSA_T_array[maskHM19_chc], HM19_e_chc[maskHM19_chc], '.-', label = 'HM19', lw = 1)
ax[0,1].plot(period_BM23, eE_mean_BM23_chc, '.-', label = 'BM23 (E)', lw = 1)
ax[0,1].plot(period_BM23, eEA_mean_BM23_chc, '.-', label = 'BM23 (EA)', lw = 1)
ax[0,1].plot(period_BM23, eEAS_mean_BM23_chc, '.-', label = 'BM23 (EAS)', lw = 1)
ax[0,1].fill_between(pSA_T_array[maskGH08_chc], GH08_e5_chc[maskGH08_chc],  GH08_e95_chc[maskGH08_chc], alpha=0.2, linewidth=0)
ax[0,1].fill_between(pSA_T_array, JB09_e5_chc,  JB09_e95_chc, alpha=0.2, linewidth=0)
ax[0,1].fill_between(pSA_T_array[maskGA10_chc], GA10_e5_chc[maskGA10_chc],  GA10_e95_chc[maskGA10_chc], alpha=0.2, linewidth=0)
ax[0,1].fill_between(pSA_T_array[maskEI12_chc], EI12_e5_chc[maskEI12_chc],  EI12_e95_chc[maskEI12_chc], alpha=0.2, linewidth=0)
ax[0,1].fill_between(pSA_T_array, LB13_e5_chc,  LB13_e95_chc, alpha=0.2, linewidth=0)
ax[0,1].fill_between(pSA_T_array[maskHM19_chc], HM19_e5_chc[maskHM19_chc],  HM19_e95_chc[maskHM19_chc], alpha=0.2, linewidth=0)
ax[0,1].fill_between(period_BM23, eEA_5_BM23_chc,  eEA_95_BM23_chc, alpha=0.2, linewidth=0)
ax[0,1].fill_between(period_BM23, eEAS_5_BM23_chc,  eEAS_95_BM23_chc, alpha=0.2, linewidth=0)

ax[1,1].plot(pSA_T_array[maskGH08_chc], GH08_e_std_chc[maskGH08_chc], '.-', label = 'GH08', lw = 1)
ax[1,1].plot(pSA_T_array, JB09_e_std_chc, '.-', label = 'JB09', lw = 1)
ax[1,1].plot(pSA_T_array[maskGA10_chc], GA10_e_std_chc[maskGA10_chc], '.-', label = 'GA10', lw = 1)
ax[1,1].plot(pSA_T_array[maskEI12_chc], EI12_e_std_chc[maskEI12_chc], '.-', label = 'EI12', lw = 1)
ax[1,1].plot(pSA_T_array, LB13_e_std_chc, '.-', label = 'LB13', lw = 1)
ax[1,1].plot(pSA_T_array[maskHM19_chc], HM19_e_std_chc[maskHM19_chc], '.-', label = 'HM19', lw = 1)
ax[1,1].plot(period_BM23, eE_std_BM23_chc, '.-', label = 'BM23 (E)', lw = 1)
ax[1,1].plot(period_BM23, eEA_std_BM23_chc, '.-', label = 'BM23 (EA)', lw = 1)
ax[1,1].plot(period_BM23, eEAS_std_BM23_chc, '.-', label = 'BM23 (EAS)', lw = 1)
ax[1,1].fill_between(pSA_T_array[maskGH08_chc], GH08_e5_std_chc[maskGH08_chc],  GH08_e95_std_chc[maskGH08_chc], alpha=0.2, linewidth=0)
ax[1,1].fill_between(pSA_T_array, JB09_e5_std_chc,  JB09_e95_std_chc, alpha=0.2, linewidth=0)
ax[1,1].fill_between(pSA_T_array[maskGA10_chc], GA10_e5_std_chc[maskGA10_chc],  GA10_e95_std_chc[maskGA10_chc], alpha=0.2, linewidth=0)
ax[1,1].fill_between(pSA_T_array[maskEI12_chc], EI12_e5_std_chc[maskEI12_chc],  EI12_e95_std_chc[maskEI12_chc], alpha=0.2, linewidth=0)
ax[1,1].fill_between(pSA_T_array, LB13_e5_std_chc,  LB13_e95_std_chc, alpha=0.2, linewidth=0)
ax[1,1].fill_between(pSA_T_array[maskHM19_chc], HM19_e5_std_chc[maskHM19_chc],  HM19_e95_std_chc[maskHM19_chc], alpha=0.2, linewidth=0)
ax[1,1].fill_between(period_BM23, eEAstd_5_BM23_chc,  eEAstd_95_BM23_chc, alpha=0.2, linewidth=0)
ax[1,1].fill_between(period_BM23, eEASstd_5_BM23_chc,  eEASstd_95_BM23_chc, alpha=0.2, linewidth=0)

# wellington
ax[0,2].plot(pSA_T_array[maskGH08_wel], GH08_e_wel[maskGH08_wel], '.-', label='GH08', lw = 1)
ax[0,2].plot(pSA_T_array, JB09_e_wel, '.-', label='JB09', lw = 1)
ax[0,2].plot(pSA_T_array[maskGA10_wel], GA10_e_wel[maskGA10_wel], '.-', label='GA10', lw = 1)
ax[0,2].plot(pSA_T_array[maskEI12_wel], EI12_e_wel[maskEI12_wel], '.-', label='EI12', lw = 1)
ax[0,2].plot(pSA_T_array, LB13_e_wel, '.-', label = 'LB13', lw = 1)
ax[0,2].plot(pSA_T_array[maskHM19_wel], HM19_e_wel[maskHM19_wel], '.-', label='HM19', lw = 1)
ax[0,2].plot(period_BM23, eE_mean_BM23_wel, '.-', label='BM23 (E)', lw = 1)
ax[0,2].plot(period_BM23, eEA_mean_BM23_wel, '.-', label='BM23 (EA)', lw = 1)
ax[0,2].plot(period_BM23, eEAS_mean_BM23_wel, '.-', label='BM23 (EAS)', lw = 1)
ax[0,2].fill_between(pSA_T_array[maskGH08_wel], GH08_e5_wel[maskGH08_wel],  GH08_e95_wel[maskGH08_wel], alpha=0.2, linewidth=0)
ax[0,2].fill_between(pSA_T_array, JB09_e5_wel,  JB09_e95_wel, alpha=0.2, linewidth=0)
ax[0,2].fill_between(pSA_T_array[maskGA10_wel], GA10_e5_wel[maskGA10_wel],  GA10_e95_wel[maskGA10_wel], alpha=0.2, linewidth=0)
ax[0,2].fill_between(pSA_T_array[maskEI12_wel], EI12_e5_wel[maskEI12_wel],  EI12_e95_wel[maskEI12_wel], alpha=0.2, linewidth=0)
ax[0,2].fill_between(pSA_T_array, LB13_e5_wel,  LB13_e95_wel, alpha=0.2, linewidth=0)
ax[0,2].fill_between(pSA_T_array[maskHM19_wel], HM19_e5_wel[maskHM19_wel],  HM19_e95_wel[maskHM19_wel], alpha=0.2, linewidth=0)
ax[0,2].fill_between(period_BM23, eEA_5_BM23_wel,  eEA_95_BM23_wel, alpha=0.2, linewidth=0)
ax[0,2].fill_between(period_BM23, eEAS_5_BM23_wel,  eEAS_95_BM23_wel, alpha=0.2, linewidth=0)

ax[1,2].plot(pSA_T_array[maskGH08_wel], GH08_e_std_wel[maskGH08_wel], '.-', label='GH08', lw = 1)
ax[1,2].plot(pSA_T_array, JB09_e_std_wel, '.-', label='JB09', lw = 1)
ax[1,2].plot(pSA_T_array[maskGA10_wel], GA10_e_std_wel[maskGA10_wel], '.-', label='GA10', lw = 1)
ax[1,2].plot(pSA_T_array[maskEI12_wel], EI12_e_std_wel[maskEI12_wel], '.-', label='EI12', lw = 1)
ax[1,2].plot(pSA_T_array, LB13_e_std_wel, '.-', label = 'LB13', lw = 1)
ax[1,2].plot(pSA_T_array[maskHM19_wel], HM19_e_std_wel[maskHM19_wel], '.-', label='HM19', lw = 1)
ax[1,2].plot(period_BM23, eE_std_BM23_wel, '.-', label='BM23 (E)', lw = 1)
ax[1,2].plot(period_BM23, eEA_std_BM23_wel, '.-', label='BM23 (EA)', lw = 1)
ax[1,2].plot(period_BM23, eEAS_std_BM23_wel, '.-', label='BM23 (EAS)', lw = 1)
ax[1,2].fill_between(pSA_T_array[maskGH08_wel], GH08_e5_std_wel[maskGH08_wel],  GH08_e95_std_wel[maskGH08_wel], alpha=0.2, linewidth=0)
ax[1,2].fill_between(pSA_T_array, JB09_e5_std_wel,  JB09_e95_std_wel, alpha=0.2, linewidth=0)
ax[1,2].fill_between(pSA_T_array[maskGA10_wel], GA10_e5_std_wel[maskGA10_wel],  GA10_e95_std_wel[maskGA10_wel], alpha=0.2, linewidth=0)
ax[1,2].fill_between(pSA_T_array[maskEI12_wel], EI12_e5_std_wel[maskEI12_wel],  EI12_e95_std_wel[maskEI12_wel], alpha=0.2, linewidth=0)
ax[1,2].fill_between(pSA_T_array, LB13_e5_std_wel,  LB13_e95_std_wel, alpha=0.2, linewidth=0)
ax[1,2].fill_between(pSA_T_array[maskHM19_wel], HM19_e5_std_wel[maskHM19_wel],  HM19_e95_std_wel[maskHM19_wel], alpha=0.2, linewidth=0)
ax[1,2].fill_between(period_BM23, eEAstd_5_BM23_wel,  eEAstd_95_BM23_wel, alpha=0.2, linewidth=0)
ax[1,2].fill_between(period_BM23, eEASstd_5_BM23_wel,  eEASstd_95_BM23_wel, alpha=0.2, linewidth=0)

for i in range(2):
    for j in range(3):
        ax[i,j].set_xlim([0.01, 6])
        ax[i,j].set_xscale('log')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[0,j].set_ylim([-3, 3])
        ax[1,j].set_ylim([1.5, 6.5])
        ax[i,j].plot([0.01, 6], [0, 0], '--k')
        ax[i,j].set_xlabel('Period, T (s)')

ax[0,0].set_title('NZ GMDB')
ax[0,1].set_title('Christchurch CBD')
ax[0,2].set_title('Wellington CBD')
ax[0,0].set_ylabel(r'Corr. Deviation Mean, $\mu_e$')
ax[1,0].set_ylabel(r'Corr. Deviation Standard Deviation, $\sigma_{e}$')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize=10)
# handles, labels = ax[0,0].get_legend_handles_labels()
# fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.10), fontsize=8, ncol=10)
plt.show()

# 4.0 Evaluation of existing SCMs

There are several existing spatial correlation models developed across the world which are summarised in the table below. In this section we test a subset of these Spatial Correlation Models (SCMs) which represent two key development features:

1) Different regional datasets e.g., Kik-net (Japan), ESM (Europe) and NGA-West2 (Global)
2) Different magnitude ranges i.e., large magnitudes events (Jayaram and Baker 2009) and full datasets across all magnitude ranges (Eposito and Iervolino 2012)

| Reference                           | Abbreviation | Intensity Measures (IM) | pSA Range, T (s)                          | Inputs                                     |
|-------------------------------------|--------------|-------------------------|-------------------------------------------|--------------------------------------------|
| Boore et al., (2003)                | BA03         | PGA, pSA                | 0                                         | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Goda and Hong (2008)                | GH08         | PGA, pSA                | 0 - 3                                     | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Goda and Atkinson (2009)            | GA09         | PGA, pSA                | 0, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0 | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Jayaram and Baker (2009)            | JB09         | PGA, pSA                | 0 - 10                                    | case, T, $\Delta h$, $n_{rec}$, $\rho$     |
| Goda and Atkinson (2010)            | GA10         | PGA, pSA                | 0, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0 | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Eposito and Iervolino (2012)        | EI12         | PGA, pSA                | 0 - 10                                    | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Du and Wang (2013)                  | DU13         | PGA, pSA                | 0, 0.2, 0.5, 1.0, 2.0, 5.0                | $b_{vs}$, T, $\Delta h$, $n_{rec}$, $\rho$ |
| Heresi and Miranda (2019)           | HM19         | PGA, pSA                | 0 - 10                                    | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Huang and Galasso (2019)            | HG19         | PGA, pSA                | 0, 0.1, 0.2, 0.5, 1.0, 2.0, 3.0, 4.0      | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Schiappapietra and Douglas (2020)   | SD20         | PGA, pSA                | 0, 0.1, 0.2, 0.5, 1.0, 2.0, 3.0, 4.0      | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Bodenmann et al., (2023): Model E   | BM23         | PGA, pSA                | 0 - 6                                     | T, $\Delta h$, $n_{rec}$, $\rho$           |
| Bodenmann et al., (2023): Model EA  | BM23EA       | PGA, pSA                | 0 - 6                                     | dA, T, $\Delta h$, $n_{rec}$, $\rho$       |
| Bodenmann et al., (2023): Model EAS | BM23EAS      | PGA, pSA                | 0 - 6                                     | dA, dS, T, $\Delta h$, $n_{rec}$, $\rho$   |

It is important to note that the authors do imply that interpolation between the parameters can be made however this should be done with caution given the linkages between pSA periods and the fitted coefficients. For this exercise interpolation will be avoided.

In [ ]:
n_rec = 6
hmax = 25

## 4.1 Correlation computation

In [ ]:
# group 1: Wellington
cbb21_pSAobs_T_grp1 = []
for i in tqdm(range(len(pSAobs_T)), desc="Grp1 pSA: "):
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(grp1_sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_grp1.append(cbb21)

# group 2: Canterbury
cbb21_pSAobs_T_grp2 = []
for i in tqdm(range(len(pSAobs_T)), desc="Grp2 pSA: "):
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(grp2_sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_grp2.append(cbb21)

# group 3: North Auckland|South Auckland|Taranaki
cbb21_pSAobs_T_grp3 = []
for i in tqdm(range(len(pSAobs_T)), desc="Grp3 pSA: "):
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(grp3_sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_grp3.append(cbb21)

# Group 4: Gisborne|Hawkes Bay
cbb21_pSAobs_T_grp4 = []
for i in tqdm(range(len(pSAobs_T)), desc="Grp4 pSA: "):
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(grp4_sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_grp4.append(cbb21)

# Group 5: Marlborough|Nelson
cbb21_pSAobs_T_grp5 = []
for i in tqdm(range(len(pSAobs_T)), desc="Grp5 pSA: "):
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(grp5_sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_grp5.append(cbb21)

# Group 6: Westland|Otago|Southland
cbb21_pSAobs_T_grp6 = []
for i in tqdm(range(len(pSAobs_T)), desc="Grp6 pSA: "):
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(grp6_sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_grp6.append(cbb21)

regions = ['Wellington', 'Christchurch', 'Auckland|Taranaki', 'Gisborne|Hawkes Bay', 'Marlborough|Nelson', 'Westland|Otago|Southland']
print('Group1, Group2, Group3, Group4, Group5, Group6',len(cbb21_pSAobs_T_grp1[0]),len(cbb21_pSAobs_T_grp2[0]),len(cbb21_pSAobs_T_grp3[0]),len(cbb21_pSAobs_T_grp4[0]),len(cbb21_pSAobs_T_grp5[0]),len(cbb21_pSAobs_T_grp6[0]))

## 4.2 Run models for correlation deviation mean and standard deviation

In [ ]:
methods = {
    "GH08": GMCSM_GH08,
    "JB09": GMCSM_JB09,
    "GA10": GMCSM_GA10,
    "EI12": GMCSM_EI12,
    "LB13": GMCSM_LB13,
    "HM19": GMCSM_HM19,
}

groups = {
    "grp1": cbb21_pSAobs_T_grp1,
    "grp2": cbb21_pSAobs_T_grp2,
    "grp3": cbb21_pSAobs_T_grp3,
    "grp4": cbb21_pSAobs_T_grp4,
    "grp5": cbb21_pSAobs_T_grp5,
    "grp6": cbb21_pSAobs_T_grp6,
}

# Initialize results container
results = {
    g: {m: {"mean": [], "std": [], "e5": [], "e95": [], "std5": [], "std95": []}
        for m in methods}
    for g in groups
}

# Fill GMCSM results
for gname, obs in groups.items():
    for i in range(len(pSA_T[:12])):
        for mname, func in methods.items():
            if mname == "JB09":
                p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = func(
                    case, pSA_T[i], dh, n_rec, obs[i], plot=False
                )
            elif mname == "LB13":
                p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = func(
                    pSA_T[i], pSA_T[i], dh, n_rec, obs[i], plot=False
                )
            else:
                p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = func(
                    pSA_T[i], dh, n_rec, obs[i], plot=False
                )

            results[gname][mname]["mean"].append(e_mean)
            results[gname][mname]["std"].append(e_std)
            results[gname][mname]["e5"].append(e5)
            results[gname][mname]["e95"].append(e95)
            results[gname][mname]["std5"].append(estd_5)
            results[gname][mname]["std95"].append(estd_95)

# Convert lists to arrays and add masks
for gname in results:
    for mname in results[gname]:
        for key in results[gname][mname]:
            results[gname][mname][key] = np.array(results[gname][mname][key])
        results[gname][mname]["mask"] = ~np.isnan(results[gname][mname]["mean"])

# BM23 integration
for gname, obs in groups.items():
    bm23 = BM23_CBB21(obs, period_BM23, BM23_parameters)
    results[gname]["BM23_E"] = {
        "mean": np.array(bm23[0]),
        "std":  np.array(bm23[3]),
        "e5":   np.array(bm23[6]),
        "e95":  np.array(bm23[7]),
        "std5": np.array(bm23[8]),
        "std95":np.array(bm23[9]),
    }
    results[gname]["BM23_E"]["mask"] = ~np.isnan(results[gname]["BM23_E"]["mean"])

    results[gname]["BM23_EA"] = {
        "mean": np.array(bm23[1]),
        "std":  np.array(bm23[4]),
        "e5":   np.array(bm23[10]),
        "e95":  np.array(bm23[11]),
        "std5": np.array(bm23[12]),
        "std95":np.array(bm23[13]),
    }
    results[gname]["BM23_EA"]["mask"] = ~np.isnan(results[gname]["BM23_EA"]["mean"])

    results[gname]["BM23_EAS"] = {
        "mean": np.array(bm23[2]),
        "std":  np.array(bm23[5]),
        "e5":   np.array(bm23[14]),
        "e95":  np.array(bm23[15]),
        "std5": np.array(bm23[16]),
        "std95":np.array(bm23[17]),
    }
    results[gname]["BM23_EAS"]["mask"] = ~np.isnan(results[gname]["BM23_EAS"]["mean"])

## 4.3 Plots

### 4.3.1 Correlation deviation mean, $\mu_e$

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 7), dpi=200, constrained_layout=True)
# Group 1
ax[0,0].plot(pSA_T_array[results["grp1"]["GH08"]["mask"]],results["grp1"]["GH08"]["mean"][results["grp1"]["GH08"]["mask"]],'.-', label='GH08', lw = 1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["GH08"]["mask"]],results["grp1"]["GH08"]["e5"][results["grp1"]["GH08"]["mask"]],results["grp1"]["GH08"]["e95"][results["grp1"]["GH08"]["mask"]],alpha=0.2)
ax[0,0].plot(pSA_T_array, results["grp1"]["JB09"]["mean"], '.-', label='JB09')
ax[0,0].fill_between(pSA_T_array,results["grp1"]["JB09"]["e5"],results["grp1"]["JB09"]["e95"],alpha=0.2)
ax[0,0].plot(pSA_T_array[results["grp1"]["GA10"]["mask"]], results["grp1"]["GA10"]["mean"][results["grp1"]["GA10"]["mask"]], '.-', label='GA10', lw = 1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["GA10"]["mask"]],results["grp1"]["GA10"]["e5"][results["grp1"]["GA10"]["mask"]], results["grp1"]["GA10"]["e95"][results["grp1"]["GA10"]["mask"]], alpha=0.2)
ax[0,0].plot(pSA_T_array[results["grp1"]["EI12"]["mask"]], results["grp1"]["EI12"]["mean"][results["grp1"]["EI12"]["mask"]],'.-', label='EI12', lw = 1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["EI12"]["mask"]],results["grp1"]["EI12"]["e5"][results["grp1"]["EI12"]["mask"]],results["grp1"]["EI12"]["e95"][results["grp1"]["EI12"]["mask"]], alpha=0.2)
ax[0,0].plot(pSA_T_array, results["grp1"]["LB13"]["mean"], '.-', label='LB13')
ax[0,0].fill_between(pSA_T_array,results["grp1"]["LB13"]["e5"],results["grp1"]["LB13"]["e95"],alpha=0.2)
ax[0,0].plot(pSA_T_array[results["grp1"]["HM19"]["mask"]],results["grp1"]["HM19"]["mean"][results["grp1"]["HM19"]["mask"]],'.-', label='HM19', lw = 1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["HM19"]["mask"]], results["grp1"]["HM19"]["e5"][results["grp1"]["HM19"]["mask"]],results["grp1"]["HM19"]["e95"][results["grp1"]["HM19"]["mask"]], alpha=0.2)
ax[0,0].plot(period_BM23, results["grp1"]["BM23_E"]["mean"], '.-', label='B23 (E)', lw = 1)
ax[0,0].fill_between(period_BM23, results["grp1"]["BM23_E"]["e5"], results["grp1"]["BM23_E"]["e95"], alpha=0.2)
ax[0,0].plot(period_BM23, results["grp1"]["BM23_EA"]["mean"], '.-', label='B23 (EA)', lw = 1)
ax[0,0].fill_between(period_BM23, results["grp1"]["BM23_EA"]["e5"], results["grp1"]["BM23_EA"]["e95"], alpha=0.2)
ax[0,0].plot(period_BM23, results["grp1"]["BM23_EAS"]["mean"], '.-', label='B23 (EAS)', lw = 1)
ax[0,0].fill_between(period_BM23,results["grp1"]["BM23_EAS"]["e5"], results["grp1"]["BM23_EAS"]["e95"],alpha=0.2)

# Group 2
ax[0,1].plot(pSA_T_array[results["grp2"]["GH08"]["mask"]],results["grp2"]["GH08"]["mean"][results["grp2"]["GH08"]["mask"]], '.-', label='GH08', lw = 1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["GH08"]["mask"]],results["grp2"]["GH08"]["e5"][results["grp2"]["GH08"]["mask"]],results["grp2"]["GH08"]["e95"][results["grp2"]["GH08"]["mask"]],alpha=0.2)
ax[0,1].plot(pSA_T_array, results["grp2"]["JB09"]["mean"], '.-', label='JB09', lw = 1)
ax[0,1].fill_between(pSA_T_array,results["grp2"]["JB09"]["e5"],results["grp2"]["JB09"]["e95"],alpha=0.2)
ax[0,1].plot(pSA_T_array[results["grp2"]["GA10"]["mask"]],results["grp2"]["GA10"]["mean"][results["grp2"]["GA10"]["mask"]], '.-', label='GA10', lw = 1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["GA10"]["mask"]],results["grp2"]["GA10"]["e5"][results["grp2"]["GA10"]["mask"]], results["grp2"]["GA10"]["e95"][results["grp2"]["GA10"]["mask"]], alpha=0.2)
ax[0,1].plot(pSA_T_array[results["grp2"]["EI12"]["mask"]], results["grp2"]["EI12"]["mean"][results["grp2"]["EI12"]["mask"]],'.-', label='EI12', lw = 1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["EI12"]["mask"]],results["grp2"]["EI12"]["e5"][results["grp2"]["EI12"]["mask"]],results["grp2"]["EI12"]["e95"][results["grp2"]["EI12"]["mask"]], alpha=0.2)
ax[0,1].plot(pSA_T_array, results["grp2"]["LB13"]["mean"], '.-', label='LB13')
ax[0,1].fill_between(pSA_T_array,results["grp2"]["LB13"]["e5"],results["grp2"]["LB13"]["e95"],alpha=0.2)
ax[0,1].plot(pSA_T_array[results["grp2"]["HM19"]["mask"]], results["grp2"]["HM19"]["mean"][results["grp2"]["HM19"]["mask"]],'.-', label='HM19', lw = 1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["HM19"]["mask"]], results["grp2"]["HM19"]["e5"][results["grp2"]["HM19"]["mask"]],results["grp2"]["HM19"]["e95"][results["grp2"]["HM19"]["mask"]], alpha=0.2)
ax[0,1].plot(period_BM23, results["grp2"]["BM23_E"]["mean"], '.-', label='B23 (E)', lw = 1)
ax[0,1].fill_between(period_BM23, results["grp2"]["BM23_E"]["e5"], results["grp2"]["BM23_E"]["e95"], alpha=0.2)
ax[0,1].plot(period_BM23, results["grp2"]["BM23_EA"]["mean"], '.-', label='B23 (EA)', lw = 1)
ax[0,1].fill_between(period_BM23, results["grp2"]["BM23_EA"]["e5"], results["grp2"]["BM23_EA"]["e95"], alpha=0.2)
ax[0,1].plot(period_BM23, results["grp2"]["BM23_EAS"]["mean"], '.-', label='B23 (EAS)', lw = 1)
ax[0,1].fill_between(period_BM23,results["grp2"]["BM23_EAS"]["e5"], results["grp2"]["BM23_EAS"]["e95"],alpha=0.2)

# Group 3
ax[0,2].plot(pSA_T_array[results["grp3"]["GH08"]["mask"]],results["grp3"]["GH08"]["mean"][results["grp3"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["GH08"]["mask"]],results["grp3"]["GH08"]["e5"][results["grp3"]["GH08"]["mask"]],results["grp3"]["GH08"]["e95"][results["grp3"]["GH08"]["mask"]],alpha=0.2)
ax[0,2].plot(pSA_T_array,results["grp3"]["JB09"]["mean"],'.-',label='JB09',lw=1)
ax[0,2].fill_between(pSA_T_array,results["grp3"]["JB09"]["e5"],results["grp3"]["JB09"]["e95"],alpha=0.2)
ax[0,2].plot(pSA_T_array[results["grp3"]["GA10"]["mask"]],results["grp3"]["GA10"]["mean"][results["grp3"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["GA10"]["mask"]],results["grp3"]["GA10"]["e5"][results["grp3"]["GA10"]["mask"]],results["grp3"]["GA10"]["e95"][results["grp3"]["GA10"]["mask"]],alpha=0.2)
ax[0,2].plot(pSA_T_array[results["grp3"]["EI12"]["mask"]],results["grp3"]["EI12"]["mean"][results["grp3"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["EI12"]["mask"]],results["grp3"]["EI12"]["e5"][results["grp3"]["EI12"]["mask"]],results["grp3"]["EI12"]["e95"][results["grp3"]["EI12"]["mask"]],alpha=0.2)
ax[0,2].plot(pSA_T_array,results["grp3"]["LB13"]["mean"],'.-',label='LB13',lw=1)
ax[0,2].fill_between(pSA_T_array,results["grp3"]["LB13"]["e5"],results["grp3"]["LB13"]["e95"],alpha=0.2)
ax[0,2].plot(pSA_T_array[results["grp3"]["HM19"]["mask"]],results["grp3"]["HM19"]["mean"][results["grp3"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["HM19"]["mask"]],results["grp3"]["HM19"]["e5"][results["grp3"]["HM19"]["mask"]],results["grp3"]["HM19"]["e95"][results["grp3"]["HM19"]["mask"]],alpha=0.2)
ax[0,2].plot(period_BM23,results["grp3"]["BM23_E"]["mean"],'.-',label='B23 (E)',lw=1)
ax[0,2].fill_between(period_BM23,results["grp3"]["BM23_E"]["e5"],results["grp3"]["BM23_E"]["e95"],alpha=0.2)
ax[0,2].plot(period_BM23,results["grp3"]["BM23_EA"]["mean"],'.-',label='B23 (EA)',lw=1)
ax[0,2].fill_between(period_BM23,results["grp3"]["BM23_EA"]["e5"],results["grp3"]["BM23_EA"]["e95"],alpha=0.2)
ax[0,2].plot(period_BM23,results["grp3"]["BM23_EAS"]["mean"],'.-',label='B23 (EAS)',lw=1)
ax[0,2].fill_between(period_BM23,results["grp3"]["BM23_EAS"]["e5"],results["grp3"]["BM23_EAS"]["e95"],alpha=0.2)

# Group 4
ax[1,0].plot(pSA_T_array[results["grp4"]["GH08"]["mask"]],results["grp4"]["GH08"]["mean"][results["grp4"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["GH08"]["mask"]],results["grp4"]["GH08"]["e5"][results["grp4"]["GH08"]["mask"]],results["grp4"]["GH08"]["e95"][results["grp4"]["GH08"]["mask"]],alpha=0.2)
ax[1,0].plot(pSA_T_array,results["grp4"]["JB09"]["mean"],'.-',label='JB09',lw=1)
ax[1,0].fill_between(pSA_T_array,results["grp4"]["JB09"]["e5"],results["grp4"]["JB09"]["e95"],alpha=0.2)
ax[1,0].plot(pSA_T_array[results["grp4"]["GA10"]["mask"]],results["grp4"]["GA10"]["mean"][results["grp4"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["GA10"]["mask"]],results["grp4"]["GA10"]["e5"][results["grp4"]["GA10"]["mask"]],results["grp4"]["GA10"]["e95"][results["grp4"]["GA10"]["mask"]],alpha=0.2)
ax[1,0].plot(pSA_T_array[results["grp4"]["EI12"]["mask"]],results["grp4"]["EI12"]["mean"][results["grp4"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["EI12"]["mask"]],results["grp4"]["EI12"]["e5"][results["grp4"]["EI12"]["mask"]],results["grp4"]["EI12"]["e95"][results["grp4"]["EI12"]["mask"]],alpha=0.2)
ax[1,0].plot(pSA_T_array,results["grp4"]["LB13"]["mean"],'.-',label='LB13',lw=1)
ax[1,0].fill_between(pSA_T_array,results["grp4"]["LB13"]["e5"],results["grp4"]["LB13"]["e95"],alpha=0.2)
ax[1,0].plot(pSA_T_array[results["grp4"]["HM19"]["mask"]],results["grp4"]["HM19"]["mean"][results["grp4"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["HM19"]["mask"]],results["grp4"]["HM19"]["e5"][results["grp4"]["HM19"]["mask"]],results["grp4"]["HM19"]["e95"][results["grp4"]["HM19"]["mask"]],alpha=0.2)
ax[1,0].plot(period_BM23,results["grp4"]["BM23_E"]["mean"],'.-',label='B23 (E)',lw=1)
ax[1,0].fill_between(period_BM23,results["grp4"]["BM23_E"]["e5"],results["grp4"]["BM23_E"]["e95"],alpha=0.2)
ax[1,0].plot(period_BM23,results["grp4"]["BM23_EA"]["mean"],'.-',label='B23 (EA)',lw=1)
ax[1,0].fill_between(period_BM23,results["grp4"]["BM23_EA"]["e5"],results["grp4"]["BM23_EA"]["e95"],alpha=0.2)
ax[1,0].plot(period_BM23,results["grp4"]["BM23_EAS"]["mean"],'.-',label='B23 (EAS)',lw=1)
ax[1,0].fill_between(period_BM23,results["grp4"]["BM23_EAS"]["e5"],results["grp4"]["BM23_EAS"]["e95"],alpha=0.2)

# Group 5
ax[1,1].plot(pSA_T_array[results["grp5"]["GH08"]["mask"]],results["grp5"]["GH08"]["mean"][results["grp5"]["GH08"]["mask"]],'.-', label='GH08', lw = 1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["GH08"]["mask"]],results["grp5"]["GH08"]["e5"][results["grp5"]["GH08"]["mask"]],results["grp5"]["GH08"]["e95"][results["grp5"]["GH08"]["mask"]],alpha=0.2)
ax[1,1].plot(pSA_T_array, results["grp5"]["JB09"]["mean"], '.-', label='JB09', lw = 1)
ax[1,1].fill_between(pSA_T_array,results["grp5"]["JB09"]["e5"],results["grp5"]["JB09"]["e95"],alpha=0.2)
ax[1,1].plot(pSA_T_array[results["grp5"]["GA10"]["mask"]],results["grp5"]["GA10"]["mean"][results["grp5"]["GA10"]["mask"]],'.-', label='GA10', lw = 1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["GA10"]["mask"]],results["grp5"]["GA10"]["e5"][results["grp5"]["GA10"]["mask"]], results["grp5"]["GA10"]["e95"][results["grp5"]["GA10"]["mask"]], alpha=0.2)
ax[1,1].plot(pSA_T_array[results["grp5"]["EI12"]["mask"]],results["grp5"]["EI12"]["mean"][results["grp5"]["EI12"]["mask"]],'.-', label='EI12', lw = 1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["EI12"]["mask"]],results["grp5"]["EI12"]["e5"][results["grp5"]["EI12"]["mask"]],results["grp5"]["EI12"]["e95"][results["grp5"]["EI12"]["mask"]], alpha=0.2)
ax[1,1].plot(pSA_T_array, results["grp5"]["LB13"]["mean"], '.-', label='JB09', lw = 1)
ax[1,1].fill_between(pSA_T_array,results["grp5"]["LB13"]["e5"],results["grp5"]["LB13"]["e95"],alpha=0.2)
ax[1,1].plot(pSA_T_array[results["grp5"]["HM19"]["mask"]],results["grp5"]["HM19"]["mean"][results["grp5"]["HM19"]["mask"]],'.-', label='HM19', lw = 1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["HM19"]["mask"]], results["grp5"]["HM19"]["e5"][results["grp5"]["HM19"]["mask"]],results["grp5"]["HM19"]["e95"][results["grp5"]["HM19"]["mask"]], alpha=0.2)
ax[1,1].plot(period_BM23, results["grp5"]["BM23_E"]["mean"], '.-', label='B23 (E)', lw = 1)
ax[1,1].fill_between(period_BM23, results["grp5"]["BM23_E"]["e5"], results["grp5"]["BM23_E"]["e95"], alpha=0.2)
ax[1,1].plot(period_BM23, results["grp5"]["BM23_EA"]["mean"], '.-', label='B23 (EA)', lw = 1)
ax[1,1].fill_between(period_BM23, results["grp5"]["BM23_EA"]["e5"], results["grp5"]["BM23_EA"]["e95"], alpha=0.2)
ax[1,1].plot(period_BM23, results["grp5"]["BM23_EAS"]["mean"], '.-', label='B23 (EAS)', lw = 1)
ax[1,1].fill_between(period_BM23,results["grp5"]["BM23_EAS"]["e5"], results["grp5"]["BM23_EAS"]["e95"],alpha=0.2)

# Group 6
ax[1,2].plot(pSA_T_array[results["grp6"]["GH08"]["mask"]],results["grp6"]["GH08"]["mean"][results["grp6"]["GH08"]["mask"]],'.-', label='GH08', lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["GH08"]["mask"]],results["grp6"]["GH08"]["e5"][results["grp6"]["GH08"]["mask"]],results["grp6"]["GH08"]["e95"][results["grp6"]["GH08"]["mask"]],alpha=0.2)
ax[1,2].plot(pSA_T_array,results["grp6"]["JB09"]["mean"],'.-',label='JB09',lw=1)
ax[1,2].fill_between(pSA_T_array,results["grp6"]["JB09"]["e5"],results["grp6"]["JB09"]["e95"],alpha=0.2)
ax[1,2].plot(pSA_T_array[results["grp6"]["GA10"]["mask"]],results["grp6"]["GA10"]["mean"][results["grp6"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["GA10"]["mask"]],results["grp6"]["GA10"]["e5"][results["grp6"]["GA10"]["mask"]],results["grp6"]["GA10"]["e95"][results["grp6"]["GA10"]["mask"]],alpha=0.2)
ax[1,2].plot(pSA_T_array[results["grp6"]["EI12"]["mask"]],results["grp6"]["EI12"]["mean"][results["grp6"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["EI12"]["mask"]],results["grp6"]["EI12"]["e5"][results["grp6"]["EI12"]["mask"]],results["grp6"]["EI12"]["e95"][results["grp6"]["EI12"]["mask"]],alpha=0.2)
ax[1,2].plot(pSA_T_array,results["grp6"]["LB13"]["mean"],'.-',label='LB13',lw=1)
ax[1,2].fill_between(pSA_T_array,results["grp6"]["LB13"]["e5"],results["grp6"]["LB13"]["e95"],alpha=0.2)
ax[1,2].plot(pSA_T_array[results["grp6"]["HM19"]["mask"]],results["grp6"]["HM19"]["mean"][results["grp6"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["HM19"]["mask"]],results["grp6"]["HM19"]["e5"][results["grp6"]["HM19"]["mask"]],results["grp6"]["HM19"]["e95"][results["grp6"]["HM19"]["mask"]],alpha=0.2)
ax[1,2].plot(period_BM23,results["grp6"]["BM23_E"]["mean"],'.-',label='B23 (E)',lw=1)
ax[1,2].fill_between(period_BM23,results["grp6"]["BM23_E"]["e5"],results["grp6"]["BM23_E"]["e95"],alpha=0.2)
ax[1,2].plot(period_BM23,results["grp6"]["BM23_EA"]["mean"],'.-',label='B23 (EA)',lw=1)
ax[1,2].fill_between(period_BM23,results["grp6"]["BM23_EA"]["e5"],results["grp6"]["BM23_EA"]["e95"],alpha=0.2)
ax[1,2].plot(period_BM23,results["grp6"]["BM23_EAS"]["mean"],'.-',label='B23 (EAS)',lw=1)
ax[1,2].fill_between(period_BM23,results["grp6"]["BM23_EAS"]["e5"],results["grp6"]["BM23_EAS"]["e95"],alpha=0.2)

for i in range(2):
    ax[i,0].set_ylabel(r'Correlation Deviation Mean, $\mu_e$')
    for j in range(3):
        ax[i,j].set_xlim([0.01, 6])
        ax[i,j].set_xscale('log')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_ylim([-6, 6])
        ax[i,j].plot([0.01, 6], [0, 0], '--k')
        ax[i,j].set_xlabel('Period, T (s)')

ax[0,0].set_title('Group 1: Wellington')
ax[0,1].set_title('Group 2: Canterbury')
ax[0,2].set_title('Group 3: Auckland|Taranaki')
ax[1,0].set_title('Group 4: Gisborne|Hawkes Bay')
ax[1,1].set_title('Group 5: Marlborough|Nelson')
ax[1,2].set_title('Group 6: Westland|Otago|Southland')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize=10)
plt.show()

### 4.3.2 Correlation deviation mean, $\sigma_e$

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 7), dpi=200, constrained_layout=True)
# Group 1
ax[0,0].plot(pSA_T_array[results["grp1"]["GH08"]["mask"]],results["grp1"]["GH08"]["std"][results["grp1"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["GH08"]["mask"]],results["grp1"]["GH08"]["std5"][results["grp1"]["GH08"]["mask"]],results["grp1"]["GH08"]["std95"][results["grp1"]["GH08"]["mask"]],alpha=0.2)
ax[0,0].plot(pSA_T_array,results["grp1"]["JB09"]["std"],'.-',label='JB09',lw=1)
ax[0,0].fill_between(pSA_T_array,results["grp1"]["JB09"]["std5"],results["grp1"]["JB09"]["std95"],alpha=0.2)
ax[0,0].plot(pSA_T_array[results["grp1"]["GA10"]["mask"]],results["grp1"]["GA10"]["std"][results["grp1"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["GA10"]["mask"]],results["grp1"]["GA10"]["std5"][results["grp1"]["GA10"]["mask"]],results["grp1"]["GA10"]["std95"][results["grp1"]["GA10"]["mask"]],alpha=0.2)
ax[0,0].plot(pSA_T_array[results["grp1"]["EI12"]["mask"]],results["grp1"]["EI12"]["std"][results["grp1"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["EI12"]["mask"]],results["grp1"]["EI12"]["std5"][results["grp1"]["EI12"]["mask"]],results["grp1"]["EI12"]["std95"][results["grp1"]["EI12"]["mask"]],alpha=0.2)
ax[0,0].plot(pSA_T_array,results["grp1"]["LB13"]["std"],'.-',label='LB13',lw=1)
ax[0,0].fill_between(pSA_T_array,results["grp1"]["LB13"]["std5"],results["grp1"]["LB13"]["std95"],alpha=0.2)
ax[0,0].plot(pSA_T_array[results["grp1"]["HM19"]["mask"]],results["grp1"]["HM19"]["std"][results["grp1"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[0,0].fill_between(pSA_T_array[results["grp1"]["HM19"]["mask"]],results["grp1"]["HM19"]["std5"][results["grp1"]["HM19"]["mask"]],results["grp1"]["HM19"]["std95"][results["grp1"]["HM19"]["mask"]],alpha=0.2)
ax[0,0].plot(period_BM23,results["grp1"]["BM23_E"]["std"],'.-',label='B23 (E)',lw=1)
ax[0,0].fill_between(period_BM23,results["grp1"]["BM23_E"]["std5"],results["grp1"]["BM23_E"]["std95"],alpha=0.2)
ax[0,0].plot(period_BM23,results["grp1"]["BM23_EA"]["std"],'.-',label='B23 (EA)',lw=1)
ax[0,0].fill_between(period_BM23,results["grp1"]["BM23_EA"]["std5"],results["grp1"]["BM23_EA"]["std95"],alpha=0.2)
ax[0,0].plot(period_BM23,results["grp1"]["BM23_EAS"]["std"],'.-',label='B23 (EAS)',lw=1)
ax[0,0].fill_between(period_BM23,results["grp1"]["BM23_EAS"]["std5"],results["grp1"]["BM23_EAS"]["std95"],alpha=0.2)

# Group 2
ax[0,1].plot(pSA_T_array[results["grp2"]["GH08"]["mask"]],results["grp2"]["GH08"]["std"][results["grp2"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["GH08"]["mask"]],results["grp2"]["GH08"]["std5"][results["grp2"]["GH08"]["mask"]],results["grp2"]["GH08"]["std95"][results["grp2"]["GH08"]["mask"]],alpha=0.2)
ax[0,1].plot(pSA_T_array,results["grp2"]["JB09"]["std"],'.-',label='JB09',lw=1)
ax[0,1].fill_between(pSA_T_array,results["grp2"]["JB09"]["std5"],results["grp2"]["JB09"]["std95"],alpha=0.2)
ax[0,1].plot(pSA_T_array[results["grp2"]["GA10"]["mask"]],results["grp2"]["GA10"]["std"][results["grp2"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["GA10"]["mask"]],results["grp2"]["GA10"]["std5"][results["grp2"]["GA10"]["mask"]],results["grp2"]["GA10"]["std95"][results["grp2"]["GA10"]["mask"]],alpha=0.2)
ax[0,1].plot(pSA_T_array[results["grp2"]["EI12"]["mask"]],results["grp2"]["EI12"]["std"][results["grp2"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["EI12"]["mask"]],results["grp2"]["EI12"]["std5"][results["grp2"]["EI12"]["mask"]],results["grp2"]["EI12"]["std95"][results["grp2"]["EI12"]["mask"]],alpha=0.2)
ax[0,1].plot(pSA_T_array,results["grp2"]["LB13"]["std"],'.-',label='LB13',lw=1)
ax[0,1].fill_between(pSA_T_array,results["grp2"]["LB13"]["std5"],results["grp2"]["LB13"]["std95"],alpha=0.2)
ax[0,1].plot(pSA_T_array[results["grp2"]["HM19"]["mask"]],results["grp2"]["HM19"]["std"][results["grp2"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[0,1].fill_between(pSA_T_array[results["grp2"]["HM19"]["mask"]],results["grp2"]["HM19"]["std5"][results["grp2"]["HM19"]["mask"]],results["grp2"]["HM19"]["std95"][results["grp2"]["HM19"]["mask"]],alpha=0.2)
ax[0,1].plot(period_BM23,results["grp2"]["BM23_E"]["std"],'.-',label='B23 (E)',lw=1)
ax[0,1].fill_between(period_BM23,results["grp2"]["BM23_E"]["std5"],results["grp2"]["BM23_E"]["std95"],alpha=0.2)
ax[0,1].plot(period_BM23,results["grp2"]["BM23_EA"]["std"],'.-',label='B23 (EA)',lw=1)
ax[0,1].fill_between(period_BM23,results["grp2"]["BM23_EA"]["std5"],results["grp2"]["BM23_EA"]["std95"],alpha=0.2)
ax[0,1].plot(period_BM23,results["grp2"]["BM23_EAS"]["std"],'.-',label='B23 (EAS)',lw=1)
ax[0,1].fill_between(period_BM23,results["grp2"]["BM23_EAS"]["std5"],results["grp2"]["BM23_EAS"]["std95"],alpha=0.2)

# Group 3
ax[0,2].plot(pSA_T_array[results["grp3"]["GH08"]["mask"]],results["grp3"]["GH08"]["std"][results["grp3"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["GH08"]["mask"]],results["grp3"]["GH08"]["std5"][results["grp3"]["GH08"]["mask"]],results["grp3"]["GH08"]["std95"][results["grp3"]["GH08"]["mask"]],alpha=0.2)
ax[0,2].plot(pSA_T_array,results["grp3"]["JB09"]["std"],'.-',label='JB09',lw=1)
ax[0,2].fill_between(pSA_T_array,results["grp3"]["JB09"]["std5"],results["grp3"]["JB09"]["std95"],alpha=0.2)
ax[0,2].plot(pSA_T_array[results["grp3"]["GA10"]["mask"]],results["grp3"]["GA10"]["std"][results["grp3"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["GA10"]["mask"]],results["grp3"]["GA10"]["std5"][results["grp3"]["GA10"]["mask"]],results["grp3"]["GA10"]["std95"][results["grp3"]["GA10"]["mask"]],alpha=0.2)
ax[0,2].plot(pSA_T_array[results["grp3"]["EI12"]["mask"]],results["grp3"]["EI12"]["std"][results["grp3"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["EI12"]["mask"]],results["grp3"]["EI12"]["std5"][results["grp3"]["EI12"]["mask"]],results["grp3"]["EI12"]["std95"][results["grp3"]["EI12"]["mask"]],alpha=0.2)
ax[0,2].plot(pSA_T_array,results["grp3"]["LB13"]["std"],'.-',label='LB13',lw=1)
ax[0,2].fill_between(pSA_T_array,results["grp3"]["LB13"]["std5"],results["grp3"]["LB13"]["std95"],alpha=0.2)
ax[0,2].plot(pSA_T_array[results["grp3"]["HM19"]["mask"]],results["grp3"]["HM19"]["std"][results["grp3"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[0,2].fill_between(pSA_T_array[results["grp3"]["HM19"]["mask"]],results["grp3"]["HM19"]["std5"][results["grp3"]["HM19"]["mask"]],results["grp3"]["HM19"]["std95"][results["grp3"]["HM19"]["mask"]],alpha=0.2)
ax[0,2].plot(period_BM23,results["grp3"]["BM23_E"]["std"],'.-',label='B23 (E)',lw=1)
ax[0,2].fill_between(period_BM23,results["grp3"]["BM23_E"]["std5"],results["grp3"]["BM23_E"]["std95"],alpha=0.2)
ax[0,2].plot(period_BM23,results["grp3"]["BM23_EA"]["std"],'.-',label='B23 (EA)',lw=1)
ax[0,2].fill_between(period_BM23,results["grp3"]["BM23_EA"]["std5"],results["grp3"]["BM23_EA"]["std95"],alpha=0.2)
ax[0,2].plot(period_BM23,results["grp3"]["BM23_EAS"]["std"],'.-',label='B23 (EAS)',lw=1)
ax[0,2].fill_between(period_BM23,results["grp3"]["BM23_EAS"]["std5"],results["grp3"]["BM23_EAS"]["std95"],alpha=0.2)

# Group 4
ax[1,0].plot(pSA_T_array[results["grp4"]["GH08"]["mask"]],results["grp4"]["GH08"]["std"][results["grp4"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["GH08"]["mask"]],results["grp4"]["GH08"]["std5"][results["grp4"]["GH08"]["mask"]],results["grp4"]["GH08"]["std95"][results["grp4"]["GH08"]["mask"]],alpha=0.2)
ax[1,0].plot(pSA_T_array,results["grp4"]["JB09"]["std"],'.-',label='JB09',lw=1)
ax[1,0].fill_between(pSA_T_array,results["grp4"]["JB09"]["std5"],results["grp4"]["JB09"]["std95"],alpha=0.2)
ax[1,0].plot(pSA_T_array[results["grp4"]["GA10"]["mask"]],results["grp4"]["GA10"]["std"][results["grp4"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["GA10"]["mask"]],results["grp4"]["GA10"]["std5"][results["grp4"]["GA10"]["mask"]],results["grp4"]["GA10"]["std95"][results["grp4"]["GA10"]["mask"]],alpha=0.2)
ax[1,0].plot(pSA_T_array[results["grp4"]["EI12"]["mask"]],results["grp4"]["EI12"]["std"][results["grp4"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["EI12"]["mask"]],results["grp4"]["EI12"]["std5"][results["grp4"]["EI12"]["mask"]],results["grp4"]["EI12"]["std95"][results["grp4"]["EI12"]["mask"]],alpha=0.2)
ax[1,0].plot(pSA_T_array,results["grp4"]["LB13"]["std"],'.-',label='LB13',lw=1)
ax[1,0].fill_between(pSA_T_array,results["grp4"]["LB13"]["std5"],results["grp4"]["LB13"]["std95"],alpha=0.2)
ax[1,0].plot(pSA_T_array[results["grp4"]["HM19"]["mask"]],results["grp4"]["HM19"]["std"][results["grp4"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[1,0].fill_between(pSA_T_array[results["grp4"]["HM19"]["mask"]],results["grp4"]["HM19"]["std5"][results["grp4"]["HM19"]["mask"]],results["grp4"]["HM19"]["std95"][results["grp4"]["HM19"]["mask"]],alpha=0.2)
ax[1,0].plot(period_BM23,results["grp4"]["BM23_E"]["std"],'.-',label='B23 (E)',lw=1)
ax[1,0].fill_between(period_BM23,results["grp4"]["BM23_E"]["std5"],results["grp4"]["BM23_E"]["std95"],alpha=0.2)
ax[1,0].plot(period_BM23,results["grp4"]["BM23_EA"]["std"],'.-',label='B23 (EA)',lw=1)
ax[1,0].fill_between(period_BM23,results["grp4"]["BM23_EA"]["std5"],results["grp4"]["BM23_EA"]["std95"],alpha=0.2)
ax[1,0].plot(period_BM23,results["grp4"]["BM23_EAS"]["std"],'.-',label='B23 (EAS)',lw=1)
ax[1,0].fill_between(period_BM23,results["grp4"]["BM23_EAS"]["std5"],results["grp4"]["BM23_EAS"]["std95"],alpha=0.2)

# Group 5
ax[1,1].plot(pSA_T_array[results["grp5"]["GH08"]["mask"]],results["grp5"]["GH08"]["std"][results["grp5"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["GH08"]["mask"]],results["grp5"]["GH08"]["std5"][results["grp5"]["GH08"]["mask"]],results["grp5"]["GH08"]["std95"][results["grp5"]["GH08"]["mask"]],alpha=0.2)
ax[1,1].plot(pSA_T_array,results["grp5"]["JB09"]["std"],'.-',label='JB09',lw=1)
ax[1,1].fill_between(pSA_T_array,results["grp5"]["JB09"]["std5"],results["grp5"]["JB09"]["std95"],alpha=0.2)
ax[1,1].plot(pSA_T_array[results["grp5"]["GA10"]["mask"]],results["grp5"]["GA10"]["std"][results["grp5"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["GA10"]["mask"]],results["grp5"]["GA10"]["std5"][results["grp5"]["GA10"]["mask"]],results["grp5"]["GA10"]["std95"][results["grp5"]["GA10"]["mask"]],alpha=0.2)
ax[1,1].plot(pSA_T_array[results["grp5"]["EI12"]["mask"]],results["grp5"]["EI12"]["std"][results["grp5"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["EI12"]["mask"]],results["grp5"]["EI12"]["std5"][results["grp5"]["EI12"]["mask"]],results["grp5"]["EI12"]["std95"][results["grp5"]["EI12"]["mask"]],alpha=0.2)
ax[1,1].plot(pSA_T_array,results["grp5"]["LB13"]["std"],'.-',label='LB13',lw=1)
ax[1,1].fill_between(pSA_T_array,results["grp5"]["LB13"]["std5"],results["grp5"]["LB13"]["std95"],alpha=0.2)
ax[1,1].plot(pSA_T_array[results["grp5"]["HM19"]["mask"]],results["grp5"]["HM19"]["std"][results["grp5"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[1,1].fill_between(pSA_T_array[results["grp5"]["HM19"]["mask"]],results["grp5"]["HM19"]["std5"][results["grp5"]["HM19"]["mask"]],results["grp5"]["HM19"]["std95"][results["grp5"]["HM19"]["mask"]],alpha=0.2)
ax[1,1].plot(period_BM23,results["grp5"]["BM23_E"]["std"],'.-',label='B23 (E)',lw=1)
ax[1,1].fill_between(period_BM23,results["grp5"]["BM23_E"]["std5"],results["grp5"]["BM23_E"]["std95"],alpha=0.2)
ax[1,1].plot(period_BM23,results["grp5"]["BM23_EA"]["std"],'.-',label='B23 (EA)',lw=1)
ax[1,1].fill_between(period_BM23,results["grp5"]["BM23_EA"]["std5"],results["grp5"]["BM23_EA"]["std95"],alpha=0.2)
ax[1,1].plot(period_BM23,results["grp5"]["BM23_EAS"]["std"],'.-',label='B23 (EAS)',lw=1)
ax[1,1].fill_between(period_BM23,results["grp5"]["BM23_EAS"]["std5"],results["grp5"]["BM23_EAS"]["std95"],alpha=0.2)

# Group 6
ax[1,2].plot(pSA_T_array[results["grp6"]["GH08"]["mask"]],results["grp6"]["GH08"]["std"][results["grp6"]["GH08"]["mask"]],'.-',label='GH08',lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["GH08"]["mask"]],results["grp6"]["GH08"]["std5"][results["grp6"]["GH08"]["mask"]],results["grp6"]["GH08"]["std95"][results["grp6"]["GH08"]["mask"]],alpha=0.2)
ax[1,2].plot(pSA_T_array,results["grp6"]["JB09"]["std"],'.-',label='JB09',lw=1)
ax[1,2].fill_between(pSA_T_array,results["grp6"]["JB09"]["std5"],results["grp6"]["JB09"]["std95"],alpha=0.2)
ax[1,2].plot(pSA_T_array[results["grp6"]["GA10"]["mask"]],results["grp6"]["GA10"]["std"][results["grp6"]["GA10"]["mask"]],'.-',label='GA10',lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["GA10"]["mask"]],results["grp6"]["GA10"]["std5"][results["grp6"]["GA10"]["mask"]],results["grp6"]["GA10"]["std95"][results["grp6"]["GA10"]["mask"]],alpha=0.2)
ax[1,2].plot(pSA_T_array[results["grp6"]["EI12"]["mask"]],results["grp6"]["EI12"]["std"][results["grp6"]["EI12"]["mask"]],'.-',label='EI12',lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["EI12"]["mask"]],results["grp6"]["EI12"]["std5"][results["grp6"]["EI12"]["mask"]],results["grp6"]["EI12"]["std95"][results["grp6"]["EI12"]["mask"]],alpha=0.2)
ax[1,2].plot(pSA_T_array,results["grp6"]["LB13"]["std"],'.-',label='LB13',lw=1)
ax[1,2].fill_between(pSA_T_array,results["grp6"]["LB13"]["std5"],results["grp6"]["LB13"]["std95"],alpha=0.2)
ax[1,2].plot(pSA_T_array[results["grp6"]["HM19"]["mask"]],results["grp6"]["HM19"]["std"][results["grp6"]["HM19"]["mask"]],'.-',label='HM19',lw=1)
ax[1,2].fill_between(pSA_T_array[results["grp6"]["HM19"]["mask"]],results["grp6"]["HM19"]["std5"][results["grp6"]["HM19"]["mask"]],results["grp6"]["HM19"]["std95"][results["grp6"]["HM19"]["mask"]],alpha=0.2)
ax[1,2].plot(period_BM23,results["grp6"]["BM23_E"]["std"],'.-',label='B23 (E)',lw=1)
ax[1,2].fill_between(period_BM23,results["grp6"]["BM23_E"]["std5"],results["grp6"]["BM23_E"]["std95"],alpha=0.2)
ax[1,2].plot(period_BM23,results["grp6"]["BM23_EA"]["std"],'.-',label='B23 (EA)',lw=1)
ax[1,2].fill_between(period_BM23,results["grp6"]["BM23_EA"]["std5"],results["grp6"]["BM23_EA"]["std95"],alpha=0.2)
ax[1,2].plot(period_BM23,results["grp6"]["BM23_EAS"]["std"],'.-',label='B23 (EAS)',lw=1)
ax[1,2].fill_between(period_BM23,results["grp6"]["BM23_EAS"]["std5"],results["grp6"]["BM23_EAS"]["std95"],alpha=0.2)

for i in range(2):
    ax[i,0].set_ylabel(r'Correlation Deviation Standard Deviation, $\sigma_e$')
    for j in range(3):
        ax[i,j].set_xlim([0.01, 6])
        ax[i,j].set_xscale('log')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_ylim([1, 7])
        ax[i,j].set_xlabel('Period, T (s)')

ax[0,0].set_title('Group 1: Wellington')
ax[0,1].set_title('Group 2: Canterbury')
ax[0,2].set_title('Group 3: Auckland|Taranaki')
ax[1,0].set_title('Group 4: Gisborne|Hawkes Bay')
ax[1,1].set_title('Group 5: Marlborough|Nelson')
ax[1,2].set_title('Group 6: Westland|Otago|Southland')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize=10)
plt.show()

# 5.0 WLS vs. Bayesian Inference SCM: Model E

## 5.1 Load Bayesian Inference SCM: Model E

In [ ]:
modelE_params = []

for pSA in pSAobs_T[:12]:

    modelE_file_name = f"NZGMDB_v4p3_dwbar_E_{pSA}.csv"
    modelE_psa = pd.read_csv(
        SCM_DIR
        / modelE_file_name
    )
    modelE_params.append(modelE_psa)

## 5.2 Model E Comparison Plots

In [ ]:
# parameter density plot
fig, ax = plt.subplots(2,3, figsize=(12,6), dpi=200, constrained_layout=True)

# correlation model plot
h = np.linspace(0,40,1000)
for j, pID in enumerate([0, 6, 11]):
    for param in range(len(summary_pSA[pID])):
        ax[0,j].plot(h, np.exp(-h/summary_pSA[pID].r.values[param]), color='tab:blue', lw=0.1, alpha=0.5)

    for posteriors in range(len(modelE_params[pID])):
        ax[1,j].plot(h, np.exp(-(h/modelE_params[pID].LE.values[posteriors])**modelE_params[pID].gammaE.values[posteriors]), color='tab:blue', lw=0.1, alpha=0.5)

    # dummy for labelling
    ax[0,j].plot(h, np.exp(-h/summary_pSA[pID].r.values[param]), color='tab:blue', lw=0.25, alpha=1, label = 'Individual Earthquakes')
    ax[1,j].plot(h, np.exp(-(h/modelE_params[pID].LE.values[posteriors])**modelE_params[pID].gammaE.values[posteriors]), color='tab:blue', lw=0.25, alpha=1, label = 'Posterior Distribution')

    # mean
    ax[0,j].plot(h, np.exp(-h/lE_pSA[pID]), '-k', lw=1, label = 'mean')
    ax[1,j].plot(h, np.exp(-(h/np.mean(modelE_params[pID].LE.values))**np.mean(modelE_params[pID].gammaE.values)), '-k', lw=1, label = 'mean')

    # 5 - 95 CI
    y_low  = np.exp(-h / lE5CI_pSA[pID])
    y_high = np.exp(-h / lE95CI_pSA[pID])
    ax[0,j].fill_between(h, y_low, y_high,
                         color='tab:green', alpha=0.2, linewidth=0,
                         label='5–95% CI')

    y_low_mE = np.exp(-(h/np.percentile(modelE_params[pID].LE.values,2.5))**np.percentile(modelE_params[pID].gammaE.values,2.5))
    y_high_mE = np.exp(-(h/np.percentile(modelE_params[pID].LE.values,97.5))**np.percentile(modelE_params[pID].gammaE.values,97.5))
    ax[1,j].fill_between(h, y_low_mE, y_high_mE,
                         color='tab:green', alpha=0.2, linewidth=0,
                         label='5–95% CI')

    # top row label
    textstr_top = '\n'.join((
        rf'$\ell_E={lE_pSA[pID]:.2f}$',
        rf'$\gamma_E={1:.2f}$'
    ))
    ax[0, j].text(
        0.03, 0.05, textstr_top,
        transform=ax[0, j].transAxes,
        fontsize=10, va='bottom', ha='left',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.70)
    )

    # bottom row label
    textstr_bottom = '\n'.join((
        rf'$\ell_E={np.mean(modelE_params[pID].LE.values):.2f}$',
        rf'$\gamma_E={np.mean(modelE_params[pID].gammaE.values):.2f}$'
    ))
    ax[1, j].text(
        0.03, 0.05, textstr_bottom,
        transform=ax[1, j].transAxes,
        fontsize=10, va='bottom', ha='left',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.70)
    )

for i in range(2):
    ax[i,0].set_ylabel(r'Correlation, $\rho$')
    for j in range(3):
        ax[i,j].set_xlim([0,40])
        ax[i,j].set_ylim([0,1])
        ax[1,j].set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')

ax[0,0].text(-0.11, 1.07, '(a)', transform=ax[0,0].transAxes,
             fontsize=12, fontweight='bold', va='top', ha='right')
ax[1,0].text(-0.11, 1.07, '(b)', transform=ax[1,0].transAxes,
             fontsize=12, fontweight='bold', va='top', ha='right')

ax[0,0].set_title('PGA')
ax[0,1].set_title('SA(1.0s)')
ax[0,2].set_title('SA(6.0s)')
ax[0,2].legend(loc = 'upper right')
ax[1,2].legend(loc = 'upper right')
plt.show(fig)

# 6.0 NZ-Specific SCM evaluation

## 6.1 Model function definition

### 6.1.1 Traditional stationary SCM

In [ ]:
# Rampersad (2025) GM SCM
def GMCSM_RL25(T,dh,n_rec, cbb2021corr, location, plot = False):
    """Compute correlation coefficients and deviations based on NZ-Specific Model developed by Rampersad et al., 2025."""

    period = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    if location == 'NZ':

        b = lE_pSA
        b5 = lE5_pSA
        b95 = lE95_pSA

    elif location == 'GRP1':

        b = lE_pSA_grp1
        b5 = lE_pSA_grp1
        b95 = lE_pSA_grp1

    elif location == 'GRP2':

        b = lE_pSA_grp2
        b5 = lE5_pSA_grp2
        b95 = lE95_pSA_grp2

    elif location == 'GRPN':

        b = lE_pSA_grpn
        b5 = lE5_pSA_grpn
        b95 = lE95_pSA_grpn

    elif location == 'GRPS':

        b = lE_pSA_grps
        b5 = lE5_pSA_grps
        b95 = lE95_pSA_grps

    loc = period.index(T)

    # Compute correlation coefficient based on seperation distance in data
    p = np.exp(-cbb2021corr.h/b[loc])
    p5 = np.exp(-cbb2021corr.h/b5[loc])
    p95 = np.exp(-cbb2021corr.h/b95[loc])

    # Compute Fisher's Z-transformation
    z_p = 0.5*np.log((1+p)/(1-p))
    z_p5 = 0.5*np.log((1+p5)/(1-p5))
    z_p95 = 0.5*np.log((1+p95)/(1-p95))

    # Compute correlation deviation, e
    e = (cbb2021corr.z_p_hat - z_p)*np.sqrt(cbb2021corr.n_j-2)
    e5 = (cbb2021corr.z_p_hat - z_p5)*np.sqrt(cbb2021corr.n_j-2)
    e95 = (cbb2021corr.z_p_hat - z_p95)*np.sqrt(cbb2021corr.n_j-2)

    # compute correlation mean and standard deviation
    e_mean = np.nanmean(e)
    e5_mean = np.nanmean(e5)
    e95_mean = np.nanmean(e95)

    e_std = np.nanstd(e)
    e5_std = np.nanstd(e5)
    e95_std = np.nanstd(e95)

    # create text string
    if T == 0.01:
        plot_str = 'PGA'
    else:
        plot_str = 'pSA = %0.2fs'%T

    # Plot results
    if plot:
        x = np.linspace(0,dh,100)
        p_dh = np.exp(-x/b[loc])
        p_std = np.square(1-p_dh)/np.sqrt(n_rec)
        fig, ax = plt.subplots(figsize = (8,5), dpi = 150, constrained_layout=True)
        im = ax.scatter(cbb2021corr.h,cbb2021corr.p_hat, c = e, vmin = -4, vmax = 4, cmap = 'bwr', edgecolors = 'black', s = 60)
        plt.plot(x,p_dh,'-k')
        plt.plot(x,p_dh - p_std,'--k')
        plt.plot(x,p_dh + p_std,'--k')
        plt.xlim(0, dh)
        plt.ylim(-1,1)
        plt.title('Rampersad (2025): %s'%plot_str)
        ax.set_ylabel(r'Correlation, $\hat{p}$ (%s)'%plot_str)
        ax.set_xlabel('Separation Distance, h (km)')
        # Display mean and standard deviation as text on plot
        textstr = '\n'.join((rf'$\mu_e={e_mean:.2f}$', rf'$\sigma_e={e_std:.2f}$'))
        ax.text(0.99, -0.95, textstr, fontsize=12, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
        fig.colorbar(im, ax=ax, label = 'Correlation Deviation, e',aspect = 20)
        plt.show(fig)

    return p, e, e_mean, e_std, e5_mean, e95_mean, e5_std, e95_std

### 6.1.2 Bayesian MCMC non-stationary SCM

In [ ]:
# Euclidean Distance Correlation Model
def getpE(dE,lE,gamma_E):
    pE = np.zeros((len(dE),1))
    pE[:,0] = np.exp(-(dE/lE)**gamma_E)
    return pE

# Angular Distance Correlation Model
def getpA(dA,lA):
    pA = (1+dA/lA)*(1-dA/180)**(180/lA)
    return pA

# Soil Dissimilarity Correlation Model
def getpS(dS,lS):
    pS = np.exp(-dS/lS)
    return pS

# Basin Dissimilarity Correlation Model
def getpZ(dZ,lZ):
    pZ = np.exp(-dZ/lZ)
    return pZ

# Topographic Dissimilarity Correlation Model (Exponential)
def getpT(dT,lT):
    pT = np.exp(-dT/lT)
    return pT

# combine and call model: EA
def MpEA(pE,pA):
    pEA = pE*pA
    return pEA

# combine and call model: EAS
def MpEAS(pE,pA,pS,w):
    pEAS = pE*(w*pA+(1-w)*pS)
    return pEAS

# combine and call model: EASZ
def MpEASZ(pE,pA,pS,pZ,w,wS):
    pEASZ = pE*(w*pA+(1-w)*(pS*wS + pZ*(1-wS)))
    return pEASZ

# combine and call model: EAST
def MpEAST(pE,pA,pS,pT,w,wS):
    pEAST = pE*(w*pA+(1-w)*(pS*wS + pT*(1-wS)))
    return pEAST

# combine and call model: EASZT
def MpEASZT(pE, pA, pS, pZ, pT, w, wS, wZ, wT):
    pS_w = pS*wS
    pZ_w = pZ*wZ
    pT_w = pT*wT
    pEASZT = pE*(w*pA+(1-w)*(pS_w+pZ_w+pT_w))
    return pEASZT

def BM23EXT_CBB21(cbb21_pSAobs_T, period, parameters):

    # Fisher z residuals with clipping
    def fisher_residual(p, obs, n):
        p = np.clip(p, -0.9999, 0.9999)  # avoid ±1
        z_p = 0.5 * np.log((1 + p) / (1 - p))
        return (obs - z_p) * np.sqrt(n - 2)

    models = ["E","EA","EAS","EASZ","EAST","EASZT"]

    # Storage: each model has mean, mean5, mean95, std, std5, std95
    results = {m: {"mean": [], "mean5": [], "mean95": [],
                   "std": [], "std5": [], "std95": []}
               for m in models}

    for i in range(len(period)):
        obs = cbb21_pSAobs_T[i].z_p_hat
        n   = cbb21_pSAobs_T[i].n_j

        # --- Model E ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pEE = getpE(cbb21_pSAobs_T[i].h,
                        getattr(parameters, f"modelE_LE{suffix}")[i],
                        getattr(parameters, f"modelE_gammaE{suffix}")[i]).reshape(-1)
            eE  = fisher_residual(pEE, obs, n)
            results["E"][key].append(np.nanmean(eE))
            results["E"][key.replace("mean","std")].append(np.nanstd(eE))

        # --- Model EA ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pEAA = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEA_LE{suffix}")[i],
                         getattr(parameters, f"modelEA_gammaE{suffix}")[i]).reshape(-1)
            pAA  = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEA_LA{suffix}")[i])
            pEA  = MpEA(pEAA, pAA)
            eEA  = fisher_residual(pEA, obs, n)
            results["EA"][key].append(np.nanmean(eEA))
            results["EA"][key.replace("mean","std")].append(np.nanstd(eEA))

        # --- Model EAS ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEAS_LE{suffix}")[i],
                         getattr(parameters, f"modelEAS_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEAS_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEAS_LS{suffix}")[i])
            pEAS = MpEAS(pE, pA, pS,
                         getattr(parameters, f"modelEAS_w{suffix}")[i])
            eEAS = fisher_residual(pEAS, obs, n)
            results["EAS"][key].append(np.nanmean(eEAS))
            results["EAS"][key.replace("mean","std")].append(np.nanstd(eEAS))

        # --- Model EASZ ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEASZ_LE{suffix}")[i],
                         getattr(parameters, f"modelEASZ_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEASZ_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEASZ_LS{suffix}")[i])
            pZ   = getpZ(cbb21_pSAobs_T[i].dZ1,
                         getattr(parameters, f"modelEASZ_LZ{suffix}")[i])
            pEASZ = MpEASZ(pE, pA, pS, pZ,
                           getattr(parameters, f"modelEASZ_w{suffix}")[i],
                           getattr(parameters, f"modelEASZ_wS{suffix}")[i])
            eEASZ = fisher_residual(pEASZ, obs, n)
            results["EASZ"][key].append(np.nanmean(eEASZ))
            results["EASZ"][key.replace("mean","std")].append(np.nanstd(eEASZ))

        # --- Model EAST ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEAST_LE{suffix}")[i],
                         getattr(parameters, f"modelEAST_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEAST_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEAST_LS{suffix}")[i])
            pT   = getpT(cbb21_pSAobs_T[i].dTRI,
                         getattr(parameters, f"modelEAST_LT{suffix}")[i])
            pEAST = MpEAST(pE, pA, pS, pT,
                           getattr(parameters, f"modelEAST_w{suffix}")[i],
                           getattr(parameters, f"modelEAST_wS{suffix}")[i])
            eEAST = fisher_residual(pEAST, obs, n)
            results["EAST"][key].append(np.nanmean(eEAST))
            results["EAST"][key.replace("mean","std")].append(np.nanstd(eEAST))

        # --- Model EASZT ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEASZT_LE{suffix}")[i],
                         getattr(parameters, f"modelEASZT_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEASZT_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEASZT_LS{suffix}")[i])
            pZ   = getpZ(cbb21_pSAobs_T[i].dZ1,
                         getattr(parameters, f"modelEASZT_LZ{suffix}")[i])
            pT   = getpT(cbb21_pSAobs_T[i].dTRI,
                         getattr(parameters, f"modelEASZT_LT{suffix}")[i])
            pEASZT = MpEASZT(pE, pA, pS, pZ, pT,
                             getattr(parameters, f"modelEASZT_w{suffix}")[i],
                             getattr(parameters, f"modelEASZT_wS{suffix}")[i],
                             getattr(parameters, f"modelEASZT_wZ{suffix}")[i],
                             getattr(parameters, f"modelEASZT_wT{suffix}")[i])
            eEASZT = fisher_residual(pEASZT, obs, n)
            results["EASZT"][key].append(np.nanmean(eEASZT))
            results["EASZT"][key.replace("mean","std")].append(np.nanstd(eEASZT))

    return results

## 6.2 Parameter definitions

In [ ]:
param_directory = SCM_DIR

# define base filenames
dwbar_files = 'NZGMDB_v4p3_dwbar'

nz_params = pd.read_csv(param_directory / f"{dwbar_files}_param_summary.csv")
nz_params_crus = pd.read_csv(param_directory / f"{dwbar_files}_crus_param_summary.csv")
nz_params_sub = pd.read_csv(param_directory / f"{dwbar_files}_sub_param_summary.csv")
grp1_params = pd.read_csv(param_directory / f"group1/{dwbar_files}_grp1_param_summary.csv")
grp2_params = pd.read_csv(param_directory / f"group2/{dwbar_files}_grp2_param_summary.csv")
grpn_params = pd.read_csv(param_directory / f"groupn/{dwbar_files}_grpn_param_summary.csv")
grps_params = pd.read_csv(param_directory / f"groups/{dwbar_files}_grps_param_summary.csv")

## 6.3 Evaluation dataset

In [ ]:
im_dwbar_pSA_files_crus_n3 = []; im_dwbar_pSA_files_sub_n3 = []; im_dwbar_pSA_files_grp1_n3 = [];  im_dwbar_pSA_files_grp2_n3 = []; im_dwbar_pSA_files_grpn_n3 = []; im_dwbar_pSA_files_grps_n3 = []
for pID in tqdm(range(len(pSAobs_T))):

    #load file: all data crustal
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_crus}_n3_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_crus_n3.append(im_dwbar)

     #load file: all data sub
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_sub}_n3_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_sub_n3.append(im_dwbar)

    # load file: group 1 wellington
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grp1}_n3_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grp1_n3.append(im_dwbar)

    # load file: group 2 canterbury
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grp2}_n3_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grp2_n3.append(im_dwbar)

    # load file: group north island
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grpn}_n3_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grpn_n3.append(im_dwbar)

    # load file: group south island
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file_grps}_n3_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    im_dwbar_pSA_files_grps_n3.append(im_dwbar)

## 6.4 Correlation cloud computation

In [ ]:
n_rec = 6
hmax = 25

cbb21_NZ = []
cbb21_crus = []
cbb21_sub = []
cbb21_grp1 = []
cbb21_grp2 = []
cbb21_grpn = []
cbb21_grps = []

# nz-wide
for i in tqdm(range(len(im_dwbar_pSA_files_n3))):
    im_dwbar_pSA_files_n3[i]['dw_bar'] = im_dwbar_pSA_files_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_n3[i],n_rec,hmax)
    cbb21_NZ.append(cbb21)

# nz-wide: crustal
for i in tqdm(range(len(im_dwbar_pSA_files_crus_n3))):
    im_dwbar_pSA_files_crus_n3[i]['dw_bar'] = im_dwbar_pSA_files_crus_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_crus_n3[i],n_rec,hmax)
    cbb21_crus.append(cbb21)

# nz-wide: subduction
for i in tqdm(range(len(im_dwbar_pSA_files_sub_n3))):
    im_dwbar_pSA_files_sub_n3[i]['dw_bar'] = im_dwbar_pSA_files_sub_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_sub_n3[i],n_rec,hmax)
    cbb21_sub.append(cbb21)

# group1
for i in tqdm(range(len(im_dwbar_pSA_files_grp1_n3))):
    im_dwbar_pSA_files_grp1_n3[i]['dw_bar'] = im_dwbar_pSA_files_grp1_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grp1_n3[i],n_rec,hmax)
    cbb21_grp1.append(cbb21)

# group2
for i in tqdm(range(len(im_dwbar_pSA_files_grp2_n3))):
    im_dwbar_pSA_files_grp2_n3[i]['dw_bar'] = im_dwbar_pSA_files_grp2_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grp2_n3[i],n_rec,hmax)
    cbb21_grp2.append(cbb21)

# groupn
for i in tqdm(range(len(im_dwbar_pSA_files_grpn_n3))):
    im_dwbar_pSA_files_grpn_n3[i]['dw_bar'] = im_dwbar_pSA_files_grpn_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grpn_n3[i],n_rec,hmax)
    cbb21_grpn.append(cbb21)

# groups
for i in tqdm(range(len(im_dwbar_pSA_files_grps_n3))):
    im_dwbar_pSA_files_grps_n3[i]['dw_bar'] = im_dwbar_pSA_files_grps_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grps_n3[i],n_rec,hmax)
    cbb21_grps.append(cbb21)

## 6.5 Model Parameter plots

### 6.5.1 Model E

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, nz_params['modelE_LE'],'.-',label = 'NZ-Wide')
ax[0].fill_between(pSA_T, nz_params['modelE_LE_5'],nz_params['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grp1_params['modelE_LE'],'.-',label = 'Wellington')
ax[0].fill_between(pSA_T, grp1_params['modelE_LE_5'],grp1_params['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grp2_params['modelE_LE'],'.-', label='Canterbury')
ax[0].fill_between(pSA_T, grp2_params['modelE_LE_5'],grp2_params['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grpn_params['modelE_LE'],'.-', label='North Island')
ax[0].fill_between(pSA_T, grpn_params['modelE_LE_5'],grpn_params['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grps_params['modelE_LE'],'.-', label='South Island')
ax[0].fill_between(pSA_T, grps_params['modelE_LE_5'],grps_params['modelE_LE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params['modelE_gammaE_5'],nz_params['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grp1_params['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, grp1_params['modelE_gammaE_5'],grp1_params['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grp2_params['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, grp2_params['modelE_gammaE_5'],grp2_params['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grpn_params['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, grpn_params['modelE_gammaE_5'],grpn_params['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grps_params['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, grps_params['modelE_gammaE_5'],grps_params['modelE_gammaE_95'], alpha = 0.2)

for i in range(2):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[0].set_ylim([0.1,200])
ax[1].set_ylim([0,2])
ax[0].legend(loc = 'lower left', ncols = 3)
plt.show()

fig, ax = plt.subplots(1,2,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, nz_params['modelE_LE'],'.-',label = 'NZ-Wide: All')
ax[0].fill_between(pSA_T, nz_params['modelE_LE_5'],nz_params['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, nz_params_crus['modelE_LE'],'.-',label = 'NZ-Wide: Crus')
ax[0].fill_between(pSA_T, nz_params_crus['modelE_LE_5'],nz_params_crus['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, nz_params_sub['modelE_LE'],'.-',label = 'NZ-Wide: Sub')
ax[0].fill_between(pSA_T, nz_params_sub['modelE_LE_5'],nz_params_sub['modelE_LE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params['modelE_gammaE_5'],nz_params['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params_crus['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params_crus['modelE_gammaE_5'],nz_params_crus['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params_sub['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params_sub['modelE_gammaE_5'],nz_params_sub['modelE_gammaE_95'], alpha = 0.2)

for i in range(2):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[0].set_ylim([1,200])
ax[1].set_ylim([0,1])
ax[0].legend(loc = 'lower center', ncols = 3)
plt.show()

### 6.5.2 Model EA

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, nz_params['modelEA_LE'],'.-',label = 'NZ-Wide')
ax[0].fill_between(pSA_T, nz_params['modelEA_LE_5'],nz_params['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grp1_params['modelEA_LE'],'.-',label = 'Wellington')
ax[0].fill_between(pSA_T, grp1_params['modelEA_LE_5'],grp1_params['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grp2_params['modelEA_LE'],'.-', label='Canterbury')
ax[0].fill_between(pSA_T, grp2_params['modelEA_LE_5'],grp2_params['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grpn_params['modelEA_LE'],'.-', label='North Island')
ax[0].fill_between(pSA_T, grpn_params['modelEA_LE_5'],grpn_params['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grps_params['modelEA_LE'],'.-', label='South Island')
ax[0].fill_between(pSA_T, grps_params['modelEA_LE_5'],grps_params['modelEA_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, nz_params['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params['modelEA_gammaE_5'],nz_params['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grp1_params['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, grp1_params['modelEA_gammaE_5'],grp1_params['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grp2_params['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, grp2_params['modelEA_gammaE_5'],grp2_params['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grpn_params['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, grpn_params['modelEA_gammaE_5'],grpn_params['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grps_params['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, grps_params['modelEA_gammaE_5'],grps_params['modelEA_gammaE_95'], alpha = 0.2)

ax[2].semilogx(pSA_T, nz_params['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params['modelEA_LA_5'],nz_params['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grp1_params['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, grp1_params['modelEA_LA_5'],grp1_params['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grp2_params['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, grp2_params['modelEA_LA_5'],grp2_params['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grpn_params['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, grpn_params['modelEA_LA_5'],grpn_params['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grps_params['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, grps_params['modelEA_LA_5'],grps_params['modelEA_LA_95'], alpha = 0.2)

for i in range(3):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[2].set_ylabel(r'$L_A(\degree)$')
ax[0].set_ylim([0.1,200])
ax[1].set_ylim([0,2])
ax[2].set_ylim([0,45])
ax[0].legend(loc = 'lower left', ncols = 2)
plt.show()


fig, ax = plt.subplots(1,3,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, nz_params['modelEA_LE'],'.-',label = 'NZ-Wide: All')
ax[0].fill_between(pSA_T, nz_params['modelEA_LE_5'],nz_params['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, nz_params_crus['modelEA_LE'],'.-',label = 'NZ-Wide: Crus')
ax[0].fill_between(pSA_T, nz_params_crus['modelEA_LE_5'],nz_params_crus['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, nz_params_sub['modelEA_LE'],'.-', label='NZ-Wide: Sub')
ax[0].fill_between(pSA_T, nz_params_sub['modelEA_LE_5'],nz_params_sub['modelEA_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, nz_params['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params['modelEA_gammaE_5'],nz_params['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params_crus['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params_crus['modelEA_gammaE_5'],nz_params_crus['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params_sub['modelEA_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params_sub['modelEA_gammaE_5'],nz_params_sub['modelEA_gammaE_95'], alpha = 0.2)

ax[2].semilogx(pSA_T, nz_params['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params['modelEA_LA_5'],nz_params['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, nz_params_crus['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params_crus['modelEA_LA_5'],nz_params_crus['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, nz_params_sub['modelEA_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params_sub['modelEA_LA_5'],nz_params_sub['modelEA_LA_95'], alpha = 0.2)

for i in range(3):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[2].set_ylabel(r'$L_A(\degree)$')
ax[0].set_ylim([1,200])
ax[1].set_ylim([0,1])
ax[2].set_ylim([0,45])
ax[0].legend(loc = 'lower left')
plt.show()

### 6.5.3 Model EAS

In [ ]:
fig = plt.figure(figsize=(12,6), dpi=200, constrained_layout=True)
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1,1])
gs_top = gs[0].subgridspec(1, 3)
ax0 = fig.add_subplot(gs_top[0])
ax1 = fig.add_subplot(gs_top[1])
ax2 = fig.add_subplot(gs_top[2])

gs_bottom = gs[1].subgridspec(1, 2)
ax3 = fig.add_subplot(gs_bottom[0])
ax4 = fig.add_subplot(gs_bottom[1])

ax = [ax0, ax1, ax2, ax3, ax4]

ax[0].loglog(pSA_T, nz_params['modelEAS_LE'],'.-',label = 'NZ-Wide')
ax[0].fill_between(pSA_T, nz_params['modelEAS_LE_5'],nz_params['modelEAS_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grp1_params['modelEAS_LE'],'.-',label = 'Wellington')
ax[0].fill_between(pSA_T, grp1_params['modelEAS_LE_5'],grp1_params['modelEAS_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grp2_params['modelEAS_LE'],'.-', label='Canterbury')
ax[0].fill_between(pSA_T, grp2_params['modelEAS_LE_5'],grp2_params['modelEAS_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grpn_params['modelEAS_LE'],'.-', label='North Island')
ax[0].fill_between(pSA_T, grpn_params['modelEAS_LE_5'],grpn_params['modelEAS_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, grps_params['modelEAS_LE'],'.-', label='South Island')
ax[0].fill_between(pSA_T, grps_params['modelEAS_LE_5'],grps_params['modelEAS_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, nz_params['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params['modelEAS_gammaE_5'],nz_params['modelEAS_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grp1_params['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, grp1_params['modelEAS_gammaE_5'],grp1_params['modelEAS_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grp2_params['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, grp2_params['modelEAS_gammaE_5'],grp2_params['modelEAS_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grpn_params['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, grpn_params['modelEAS_gammaE_5'],grpn_params['modelEAS_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, grps_params['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, grps_params['modelEAS_gammaE_5'],grps_params['modelEAS_gammaE_95'], alpha = 0.2)

ax[2].semilogx(pSA_T, nz_params['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params['modelEAS_LA_5'],nz_params['modelEAS_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grp1_params['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, grp1_params['modelEAS_LA_5'],grp1_params['modelEAS_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grp2_params['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, grp2_params['modelEAS_LA_5'],grp2_params['modelEAS_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grpn_params['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, grpn_params['modelEAS_LA_5'],grpn_params['modelEAS_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, grps_params['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, grps_params['modelEAS_LA_5'],grps_params['modelEAS_LA_95'], alpha = 0.2)

ax[3].loglog(pSA_T, nz_params['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, nz_params['modelEAS_LS_5'],nz_params['modelEAS_LS_95'], alpha = 0.2)
ax[3].loglog(pSA_T, grp1_params['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, grp1_params['modelEAS_LS_5'],grp1_params['modelEAS_LS_95'], alpha = 0.2)
ax[3].loglog(pSA_T, grp2_params['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, grp2_params['modelEAS_LS_5'],grp2_params['modelEAS_LS_95'], alpha = 0.2)
ax[3].loglog(pSA_T, grpn_params['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, grpn_params['modelEAS_LS_5'],grpn_params['modelEAS_LS_95'], alpha = 0.2)
ax[3].loglog(pSA_T, grps_params['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, grps_params['modelEAS_LS_5'],grps_params['modelEAS_LS_95'], alpha = 0.2)

ax[4].semilogx(pSA_T, nz_params['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, nz_params['modelEAS_w_5'],nz_params['modelEAS_w_95'], alpha = 0.2)
ax[4].semilogx(pSA_T, grp1_params['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, grp1_params['modelEAS_w_5'],grp1_params['modelEAS_w_95'], alpha = 0.2)
ax[4].semilogx(pSA_T, grp2_params['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, grp2_params['modelEAS_w_5'],grp2_params['modelEAS_w_95'], alpha = 0.2)
ax[4].semilogx(pSA_T, grpn_params['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, grpn_params['modelEAS_w_5'],grpn_params['modelEAS_w_95'], alpha = 0.2)
ax[4].semilogx(pSA_T, grps_params['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, grps_params['modelEAS_w_5'],grps_params['modelEAS_w_95'], alpha = 0.2)

for a in ax:
    a.set_xlabel('Period, T(s)')
    a.set_xlim(0.01,6)
    a.grid(which='both')

ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[2].set_ylabel(r'$L_A(\degree)$')
ax[3].set_ylabel(r'$L_S$ (m/s)')
ax[4].set_ylabel(r'Weight, w')
ax[0].set_ylim(0.1,500)
ax[1].set_ylim(0,2)
ax[2].set_ylim(0,45)
ax[3].set_ylim(1,5000)
ax[4].set_ylim(0,1)
ax[0].legend(loc = 'lower left', ncols = 2)
plt.show()

fig = plt.figure(figsize=(12,6), dpi=200, constrained_layout=True)
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1,1])
gs_top = gs[0].subgridspec(1, 3)
ax0 = fig.add_subplot(gs_top[0])
ax1 = fig.add_subplot(gs_top[1])
ax2 = fig.add_subplot(gs_top[2])
gs_bottom = gs[1].subgridspec(1, 2)
ax3 = fig.add_subplot(gs_bottom[0])
ax4 = fig.add_subplot(gs_bottom[1])
ax = [ax0, ax1, ax2, ax3, ax4]

ax[0].loglog(pSA_T, nz_params['modelEAS_LE'],'.-',label = 'NZ-Wide: All')
ax[0].fill_between(pSA_T, nz_params['modelEAS_LE_5'],nz_params['modelEAS_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, nz_params_crus['modelEAS_LE'],'.-',label = 'NZ-Wide: Crus')
ax[0].fill_between(pSA_T, nz_params_crus['modelEAS_LE_5'],nz_params_crus['modelEAS_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, nz_params_sub['modelEAS_LE'],'.-', label='NZ-Wide: Sub')
ax[0].fill_between(pSA_T, nz_params_sub['modelEAS_LE_5'],nz_params_sub['modelEAS_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, nz_params['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params['modelEAS_gammaE_5'],nz_params['modelEAS_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params_crus['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params_crus['modelEAS_gammaE_5'],nz_params_crus['modelEAS_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, nz_params_sub['modelEAS_gammaE'],'.-')
ax[1].fill_between(pSA_T, nz_params_sub['modelEAS_gammaE_5'],nz_params_sub['modelEAS_gammaE_95'], alpha = 0.2)

ax[2].semilogx(pSA_T, nz_params['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params['modelEAS_LA_5'],nz_params['modelEAS_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, nz_params_crus['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params_crus['modelEAS_LA_5'],nz_params_crus['modelEAS_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, nz_params_sub['modelEAS_LA'],'.-')
ax[2].fill_between(pSA_T, nz_params_sub['modelEAS_LA_5'],nz_params_sub['modelEAS_LA_95'], alpha = 0.2)

ax[3].loglog(pSA_T, nz_params['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, nz_params['modelEAS_LS_5'],nz_params['modelEAS_LS_95'], alpha = 0.2)
ax[3].loglog(pSA_T, nz_params_crus['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, nz_params_crus['modelEAS_LS_5'],nz_params_crus['modelEAS_LS_95'], alpha = 0.2)
ax[3].loglog(pSA_T, nz_params_sub['modelEAS_LS'],'.-')
ax[3].fill_between(pSA_T, nz_params_sub['modelEAS_LS_5'],nz_params_sub['modelEAS_LS_95'], alpha = 0.2)

ax[4].semilogx(pSA_T, nz_params['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, nz_params['modelEAS_w_5'],nz_params['modelEAS_w_95'], alpha = 0.2)
ax[4].semilogx(pSA_T, nz_params_crus['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, nz_params_crus['modelEAS_w_5'],nz_params_crus['modelEAS_w_95'], alpha = 0.2)
ax[4].semilogx(pSA_T, nz_params_sub['modelEAS_w'],'.-')
ax[4].fill_between(pSA_T, nz_params_sub['modelEAS_w_5'],nz_params_sub['modelEAS_w_95'], alpha = 0.2)

for a in ax:
    a.set_xlabel('Period, T(s)')
    a.set_xlim(0.01,6)
    a.grid(which='both')

ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[2].set_ylabel(r'$L_A(\degree)$')
ax[3].set_ylabel(r'$L_S$ (m/s)')
ax[4].set_ylabel(r'Weight, w')
ax[0].set_ylim(1,200)
ax[1].set_ylim(0,1)
ax[2].set_ylim(0,45)
ax[3].set_ylim(1,5000)
ax[4].set_ylim(0,1)
ax[0].legend(loc = 'lower left')
plt.show()

### 6.5.4 Model EASZ

In [ ]:
fig, ax = plt.subplots(2,3,figsize=(12,6),dpi=200, constrained_layout=True)
ax[0,0].loglog(pSA_T, nz_params['modelEASZ_LE'],'.-',label = 'NZ-Wide')
ax[0,0].fill_between(pSA_T, nz_params['modelEASZ_LE_5'],nz_params['modelEASZ_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp1_params['modelEASZ_LE'],'.-',label = 'Wellington')
ax[0,0].fill_between(pSA_T, grp1_params['modelEASZ_LE_5'],grp1_params['modelEASZ_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp2_params['modelEASZ_LE'],'.-', label='Canterbury')
ax[0,0].fill_between(pSA_T, grp2_params['modelEASZ_LE_5'],grp2_params['modelEASZ_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grpn_params['modelEASZ_LE'],'.-', label='North Island')
ax[0,0].fill_between(pSA_T, grpn_params['modelEASZ_LE_5'],grpn_params['modelEASZ_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grps_params['modelEASZ_LE'],'.-', label='South Island')
ax[0,0].fill_between(pSA_T, grps_params['modelEASZ_LE_5'],grps_params['modelEASZ_LE_95'], alpha = 0.2)

ax[0,1].semilogx(pSA_T, nz_params['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params['modelEASZ_gammaE_5'],nz_params['modelEASZ_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grp1_params['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grp1_params['modelEASZ_gammaE_5'],grp1_params['modelEASZ_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grp2_params['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grp2_params['modelEASZ_gammaE_5'],grp2_params['modelEASZ_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grpn_params['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grpn_params['modelEASZ_gammaE_5'],grpn_params['modelEASZ_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grps_params['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grps_params['modelEASZ_gammaE_5'],grps_params['modelEASZ_gammaE_95'], alpha = 0.2)

ax[0,2].semilogx(pSA_T, nz_params['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params['modelEASZ_LA_5'],nz_params['modelEASZ_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grp1_params['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, grp1_params['modelEASZ_LA_5'],grp1_params['modelEASZ_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grp2_params['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, grp2_params['modelEASZ_LA_5'],grp2_params['modelEASZ_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grpn_params['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, grpn_params['modelEASZ_LA_5'],grpn_params['modelEASZ_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grps_params['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, grps_params['modelEASZ_LA_5'],grps_params['modelEASZ_LA_95'], alpha = 0.2)

ax[1,0].loglog(pSA_T, nz_params['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params['modelEASZ_LS_5'],nz_params['modelEASZ_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grp1_params['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, grp1_params['modelEASZ_LS_5'],grp1_params['modelEASZ_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grp2_params['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, grp2_params['modelEASZ_LS_5'],grp2_params['modelEASZ_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grpn_params['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, grpn_params['modelEASZ_LS_5'],grpn_params['modelEASZ_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grps_params['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, grps_params['modelEASZ_LS_5'],grps_params['modelEASZ_LS_95'], alpha = 0.2)

ax[1,1].loglog(pSA_T, nz_params['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, nz_params['modelEASZ_LZ_5'],nz_params['modelEASZ_LZ_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grp1_params['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, grp1_params['modelEASZ_LZ_5'],grp1_params['modelEASZ_LZ_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grp2_params['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, grp2_params['modelEASZ_LZ_5'],grp2_params['modelEASZ_LZ_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grpn_params['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, grpn_params['modelEASZ_LZ_5'],grpn_params['modelEASZ_LZ_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grps_params['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, grps_params['modelEASZ_LZ_5'],grps_params['modelEASZ_LZ_95'], alpha = 0.2)

ax[1,2].semilogx(pSA_T, nz_params['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params['modelEASZ_w_5'],nz_params['modelEASZ_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grp1_params['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, grp1_params['modelEASZ_w_5'],grp1_params['modelEASZ_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grp2_params['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, grp2_params['modelEASZ_w_5'],grp2_params['modelEASZ_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grpn_params['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, grpn_params['modelEASZ_w_5'],grpn_params['modelEASZ_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grps_params['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, grps_params['modelEASZ_w_5'],grps_params['modelEASZ_w_95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_xlabel('Period, T(s)')
        ax[i,j].set_xlim([min(pSA_T),max(pSA_T)])
        ax[i,j].grid(which = 'both')
        ax[i,j].set_xlim(0.01,6)
ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A(\degree)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$L_Z$ (m)')
ax[1,2].set_ylabel(r'Weight, w')
ax[0,0].set_ylim([0.1,500])
ax[0,1].set_ylim([0,2])
ax[0,2].set_ylim([0,45])
ax[1,0].set_ylim([1,5000])
ax[1,1].set_ylim([1,5000])
ax[1,2].set_ylim([0,1])
ax[0,0].legend(loc = 'lower left', ncols = 2)
plt.show()

fig, ax = plt.subplots(2,3,figsize=(12,6),dpi=200, constrained_layout=True)
ax[0,0].loglog(pSA_T, nz_params['modelEASZ_LE'],'.-',label = 'NZ-Wide: All')
ax[0,0].fill_between(pSA_T, nz_params['modelEASZ_LE_5'],nz_params['modelEASZ_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params_crus['modelEASZ_LE'],'.-',label = 'NZ-Wide: Crus')
ax[0,0].fill_between(pSA_T, nz_params_crus['modelEASZ_LE_5'],nz_params_crus['modelEASZ_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params_sub['modelEASZ_LE'],'.-', label='NZ-Wide: Sub')
ax[0,0].fill_between(pSA_T, nz_params_sub['modelEASZ_LE_5'],nz_params_sub['modelEASZ_LE_95'], alpha = 0.2)

ax[0,1].semilogx(pSA_T, nz_params['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params['modelEASZ_gammaE_5'],nz_params['modelEASZ_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, nz_params_crus['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params_crus['modelEASZ_gammaE_5'],nz_params_crus['modelEASZ_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, nz_params_sub['modelEASZ_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params_sub['modelEASZ_gammaE_5'],nz_params_sub['modelEASZ_gammaE_95'], alpha = 0.2)

ax[0,2].semilogx(pSA_T, nz_params['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params['modelEASZ_LA_5'],nz_params['modelEASZ_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, nz_params_crus['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params_crus['modelEASZ_LA_5'],nz_params_crus['modelEASZ_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, nz_params_sub['modelEASZ_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params_sub['modelEASZ_LA_5'],nz_params_sub['modelEASZ_LA_95'], alpha = 0.2)

ax[1,0].loglog(pSA_T, nz_params['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params['modelEASZ_LS_5'],nz_params['modelEASZ_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, nz_params_crus['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params_crus['modelEASZ_LS_5'],nz_params_crus['modelEASZ_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, nz_params_sub['modelEASZ_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params_sub['modelEASZ_LS_5'],nz_params_sub['modelEASZ_LS_95'], alpha = 0.2)

ax[1,1].loglog(pSA_T, nz_params['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, nz_params['modelEASZ_LZ_5'],nz_params['modelEASZ_LZ_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, nz_params_crus['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, nz_params_crus['modelEASZ_LZ_5'],nz_params_crus['modelEASZ_LZ_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, nz_params_sub['modelEASZ_LZ'],'.-')
ax[1,1].fill_between(pSA_T, nz_params_sub['modelEASZ_LZ_5'],nz_params_sub['modelEASZ_LZ_95'], alpha = 0.2)

ax[1,2].semilogx(pSA_T, nz_params['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params['modelEASZ_w_5'],nz_params['modelEASZ_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, nz_params_crus['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params_crus['modelEASZ_w_5'],nz_params_crus['modelEASZ_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, nz_params_sub['modelEASZ_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params_sub['modelEASZ_w_5'],nz_params_sub['modelEASZ_w_95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_xlabel('Period, T(s)')
        ax[i,j].set_xlim([min(pSA_T),max(pSA_T)])
        ax[i,j].grid(which = 'both')
        ax[i,j].set_xlim(0.01,6)
ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A(\degree)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$L_Z$ (m)')
ax[1,2].set_ylabel(r'Weight, w')
ax[0,0].set_ylim([1,250])
ax[0,1].set_ylim([0,1])
ax[0,2].set_ylim([0,45])
ax[1,0].set_ylim([10,2000])
ax[1,1].set_ylim([10,2000])
ax[1,2].set_ylim([0,1])
ax[0,0].legend(loc = 'lower left')
plt.show()

### 6.5.5 Model EAST

In [ ]:
fig, ax = plt.subplots(2,3,figsize=(12,6),dpi=200, constrained_layout=True)
ax[0,0].loglog(pSA_T, nz_params['modelEAST_LE'],'.-',label = 'NZ-Wide')
ax[0,0].fill_between(pSA_T, nz_params['modelEAST_LE_5'],nz_params['modelEAST_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp1_params['modelEAST_LE'],'.-',label = 'Wellington')
ax[0,0].fill_between(pSA_T, grp1_params['modelEAST_LE_5'],grp1_params['modelEAST_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp2_params['modelEAST_LE'],'.-', label='Canterbury')
ax[0,0].fill_between(pSA_T, grp2_params['modelEAST_LE_5'],grp2_params['modelEAST_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grpn_params['modelEAST_LE'],'.-', label='North Island')
ax[0,0].fill_between(pSA_T, grpn_params['modelEAST_LE_5'],grpn_params['modelEAST_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grps_params['modelEAST_LE'],'.-', label='South Island')
ax[0,0].fill_between(pSA_T, grps_params['modelEAST_LE_5'],grps_params['modelEAST_LE_95'], alpha = 0.2)

ax[0,1].semilogx(pSA_T, nz_params['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params['modelEAST_gammaE_5'],nz_params['modelEAST_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grp1_params['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grp1_params['modelEAST_gammaE_5'],grp1_params['modelEAST_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grp2_params['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grp2_params['modelEAST_gammaE_5'],grp2_params['modelEAST_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grpn_params['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grpn_params['modelEAST_gammaE_5'],grpn_params['modelEAST_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, grps_params['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, grps_params['modelEAST_gammaE_5'],grps_params['modelEAST_gammaE_95'], alpha = 0.2)

ax[0,2].semilogx(pSA_T, nz_params['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params['modelEAST_LA_5'],nz_params['modelEAST_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grp1_params['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, grp1_params['modelEAST_LA_5'],grp1_params['modelEAST_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grp2_params['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, grp2_params['modelEAST_LA_5'],grp2_params['modelEAST_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grpn_params['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, grpn_params['modelEAST_LA_5'],grpn_params['modelEAST_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, grps_params['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, grps_params['modelEAST_LA_5'],grps_params['modelEAST_LA_95'], alpha = 0.2)

ax[1,0].loglog(pSA_T, nz_params['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params['modelEAST_LS_5'],nz_params['modelEAST_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grp1_params['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, grp1_params['modelEAST_LS_5'],grp1_params['modelEAST_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grp2_params['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, grp2_params['modelEAST_LS_5'],grp2_params['modelEAST_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grpn_params['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, grpn_params['modelEAST_LS_5'],grpn_params['modelEAST_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, grps_params['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, grps_params['modelEAST_LS_5'],grps_params['modelEAST_LS_95'], alpha = 0.2)

ax[1,1].loglog(pSA_T, nz_params['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, nz_params['modelEAST_LT_5'],nz_params['modelEAST_LT_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grp1_params['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, grp1_params['modelEAST_LT_5'],grp1_params['modelEAST_LT_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grp2_params['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, grp2_params['modelEAST_LT_5'],grp2_params['modelEAST_LT_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grpn_params['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, grpn_params['modelEAST_LT_5'],grpn_params['modelEAST_LT_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, grps_params['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, grps_params['modelEAST_LT_5'],grps_params['modelEAST_LT_95'], alpha = 0.2)

ax[1,2].semilogx(pSA_T, nz_params['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params['modelEAST_w_5'],nz_params['modelEAST_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grp1_params['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, grp1_params['modelEAST_w_5'],grp1_params['modelEAST_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grp2_params['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, grp2_params['modelEAST_w_5'],grp2_params['modelEAST_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grpn_params['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, grpn_params['modelEAST_w_5'],grpn_params['modelEAST_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, grps_params['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, grps_params['modelEAST_w_5'],grps_params['modelEAST_w_95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_xlabel('Period, T(s)')
        ax[i,j].set_xlim([min(pSA_T),max(pSA_T)])
        ax[i,j].grid(which = 'both')
        ax[i,j].set_xlim(0.01,6)
ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A(\degree)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$L_T$ (m)')
ax[1,2].set_ylabel(r'Weight, w')
ax[0,0].set_ylim([0.1,500])
ax[0,1].set_ylim([0,2])
ax[0,2].set_ylim([0,45])
ax[1,0].set_ylim([1,5000])
ax[1,1].set_ylim([1,500])
ax[1,2].set_ylim([0,1])
ax[0,0].legend(loc = 'lower left', ncols = 2)
plt.show()


fig, ax = plt.subplots(2,3,figsize=(12,6),dpi=200, constrained_layout=True)
ax[0,0].loglog(pSA_T, nz_params['modelEAST_LE'],'.-',label = 'NZ-Wide: All')
ax[0,0].fill_between(pSA_T, nz_params['modelEAST_LE_5'],nz_params['modelEAST_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params_crus['modelEAST_LE'],'.-',label = 'NZ-Wide: Crus')
ax[0,0].fill_between(pSA_T, nz_params_crus['modelEAST_LE_5'],nz_params_crus['modelEAST_LE_95'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params_sub['modelEAST_LE'],'.-', label='NZ-Wide: Sub')
ax[0,0].fill_between(pSA_T, nz_params_sub['modelEAST_LE_5'],nz_params_sub['modelEAST_LE_95'], alpha = 0.2)

ax[0,1].semilogx(pSA_T, nz_params['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params['modelEAST_gammaE_5'],nz_params['modelEAST_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, nz_params_crus['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params_crus['modelEAST_gammaE_5'],nz_params_crus['modelEAST_gammaE_95'], alpha = 0.2)
ax[0,1].semilogx(pSA_T, nz_params_sub['modelEAST_gammaE'],'.-')
ax[0,1].fill_between(pSA_T, nz_params_crus['modelEAST_gammaE_5'],nz_params_crus['modelEAST_gammaE_95'], alpha = 0.2)

ax[0,2].semilogx(pSA_T, nz_params['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params['modelEAST_LA_5'],nz_params['modelEAST_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, nz_params_crus['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params_crus['modelEAST_LA_5'],nz_params_crus['modelEAST_LA_95'], alpha = 0.2)
ax[0,2].semilogx(pSA_T, nz_params_sub['modelEAST_LA'],'.-')
ax[0,2].fill_between(pSA_T, nz_params_sub['modelEAST_LA_5'],nz_params_sub['modelEAST_LA_95'], alpha = 0.2)

ax[1,0].loglog(pSA_T, nz_params['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params['modelEAST_LS_5'],nz_params['modelEAST_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, nz_params_crus['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params_crus['modelEAST_LS_5'],nz_params_crus['modelEAST_LS_95'], alpha = 0.2)
ax[1,0].loglog(pSA_T, nz_params_sub['modelEAST_LS'],'.-')
ax[1,0].fill_between(pSA_T, nz_params_sub['modelEAST_LS_5'],nz_params_sub['modelEAST_LS_95'], alpha = 0.2)

ax[1,1].loglog(pSA_T, nz_params['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, nz_params['modelEAST_LT_5'],nz_params['modelEAST_LT_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, nz_params_crus['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, nz_params_crus['modelEAST_LT_5'],nz_params_crus['modelEAST_LT_95'], alpha = 0.2)
ax[1,1].loglog(pSA_T, nz_params_sub['modelEAST_LT'],'.-')
ax[1,1].fill_between(pSA_T, nz_params_sub['modelEAST_LT_5'],nz_params_sub['modelEAST_LT_95'], alpha = 0.2)

ax[1,2].semilogx(pSA_T, nz_params['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params['modelEAST_w_5'],nz_params['modelEAST_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, nz_params_crus['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params_crus['modelEAST_w_5'],nz_params_crus['modelEAST_w_95'], alpha = 0.2)
ax[1,2].semilogx(pSA_T, nz_params_sub['modelEAST_w'],'.-')
ax[1,2].fill_between(pSA_T, nz_params_sub['modelEAST_w_5'],nz_params_sub['modelEAST_w_95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_xlabel('Period, T(s)')
        ax[i,j].set_xlim([min(pSA_T),max(pSA_T)])
        ax[i,j].grid(which = 'both')
        ax[i,j].set_xlim(0.01,6)
ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A(\degree)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$L_T$ (m)')
ax[1,2].set_ylabel(r'Weight, w')
ax[0,0].set_ylim([1,250])
ax[0,1].set_ylim([0,1])
ax[0,2].set_ylim([0,45])
ax[1,0].set_ylim([10,2000])
ax[1,1].set_ylim([1,200])
ax[1,2].set_ylim([0,1])
ax[0,0].legend(loc = 'lower left')
plt.show()

### 6.5.6 Model EASZT

In [ ]:
fig = plt.figure(figsize=(12,9),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(4, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax20 = fig.add_subplot(gs[2,0])
ax21 = fig.add_subplot(gs[2,1])
ax22 = fig.add_subplot(gs[2,2])
ax30 = fig.add_subplot(gs[3,0:3])

ax00.loglog(pSA_T, nz_params['modelEASZT_LE'],'.-',label = 'NZ-Wide')
ax00.fill_between(pSA_T, nz_params['modelEASZT_LE_5'],nz_params['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, grp1_params['modelEASZT_LE'],'.-',label = 'Wellington')
ax00.fill_between(pSA_T, grp1_params['modelEASZT_LE_5'],grp1_params['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, grp2_params['modelEASZT_LE'],'.-', label='Canterbury')
ax00.fill_between(pSA_T, grp2_params['modelEASZT_LE_5'],grp2_params['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, grpn_params['modelEASZT_LE'],'.-', label='North Island')
ax00.fill_between(pSA_T, grpn_params['modelEASZT_LE_5'],grpn_params['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, grps_params['modelEASZT_LE'],'.-', label='South Island')
ax00.fill_between(pSA_T, grps_params['modelEASZT_LE_5'],grps_params['modelEASZT_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, nz_params['modelEASZT_gammaE'],'.-',label = 'NZ-Wide', lw = 1)
ax01.fill_between(pSA_T, nz_params['modelEASZT_gammaE_5'], nz_params['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, grp1_params['modelEASZT_gammaE'],'.-',label = 'Wellington')
ax01.fill_between(pSA_T, grp1_params['modelEASZT_gammaE_5'], grp1_params['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, grp2_params['modelEASZT_gammaE'],'.-', label='Canterbury')
ax01.fill_between(pSA_T, grp2_params['modelEASZT_gammaE_5'], grp2_params['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, grpn_params['modelEASZT_gammaE'],'.-', label='North Island')
ax01.fill_between(pSA_T, grpn_params['modelEASZT_gammaE_5'], grpn_params['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, grps_params['modelEAST_gammaE'],'.-', label='South Island')
ax01.fill_between(pSA_T, grps_params['modelEASZT_gammaE_5'], grps_params['modelEASZT_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, nz_params['modelEASZT_LA'],'.-',label = 'NZ-Wide', lw = 1)
ax02.fill_between(pSA_T, nz_params['modelEASZT_LA_5'], nz_params['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, grp1_params['modelEASZT_LA'],'.-',label = 'Wellington')
ax02.fill_between(pSA_T, grp1_params['modelEASZT_LA_5'], grp1_params['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, grp2_params['modelEASZT_LA'],'.-', label='Canterbury')
ax02.fill_between(pSA_T, grp2_params['modelEASZT_LA_5'], grp2_params['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, grpn_params['modelEASZT_LA'],'.-', label='North Island')
ax02.fill_between(pSA_T, grpn_params['modelEASZT_LA_5'], grpn_params['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, grps_params['modelEASZT_LA'],'.-', label='South Island')
ax02.fill_between(pSA_T, grps_params['modelEASZT_LA_5'], grps_params['modelEASZT_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, nz_params['modelEASZT_LS'],'.-',label = 'NZ-Wide', lw = 1)
ax10.fill_between(pSA_T, nz_params['modelEASZT_LS_5'], nz_params['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, grp1_params['modelEASZT_LS'],'.-',label = 'Wellington')
ax10.fill_between(pSA_T, grp1_params['modelEASZT_LS_5'], grp1_params['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, grp2_params['modelEASZT_LS'],'.-', label='Canterbury')
ax10.fill_between(pSA_T, grp2_params['modelEASZT_LS_5'], grp2_params['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, grpn_params['modelEASZT_LS'],'.-', label='North Island')
ax10.fill_between(pSA_T, grpn_params['modelEASZT_LS_5'], grpn_params['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, grps_params['modelEASZT_LS'],'.-', label='South Island')
ax10.fill_between(pSA_T, grps_params['modelEASZT_LS_5'], grps_params['modelEASZT_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, nz_params['modelEASZT_LZ'],'.-',label = 'NZ-Wide', lw = 1)
ax11.fill_between(pSA_T, nz_params['modelEASZT_LZ_5'], nz_params['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, grp1_params['modelEASZT_LZ'],'.-',label = 'Wellington')
ax11.fill_between(pSA_T, grp1_params['modelEASZT_LZ_5'], grp1_params['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, grp2_params['modelEASZT_LZ'],'.-', label='Canterbury')
ax11.fill_between(pSA_T, grp2_params['modelEASZT_LZ_5'], grp2_params['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, grpn_params['modelEASZT_LZ'],'.-', label='North Island')
ax11.fill_between(pSA_T, grpn_params['modelEASZT_LZ_5'], grpn_params['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, grps_params['modelEASZT_LZ'],'.-', label='South Island')
ax11.fill_between(pSA_T, grps_params['modelEASZT_LZ_5'], grps_params['modelEASZT_LZ_95'], alpha = 0.2)

ax12.loglog(pSA_T, nz_params['modelEASZT_LT'],'.-',label = 'NZ-Wide', lw = 1)
ax12.fill_between(pSA_T, nz_params['modelEASZT_LT_5'], nz_params['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, grp1_params['modelEASZT_LT'],'.-',label = 'Wellington')
ax12.fill_between(pSA_T, grp1_params['modelEASZT_LT_5'], grp1_params['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, grp2_params['modelEASZT_LT'],'.-', label='Canterbury')
ax12.fill_between(pSA_T, grp2_params['modelEASZT_LT_5'], grp2_params['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, grpn_params['modelEASZT_LT'],'.-', label='North Island')
ax12.fill_between(pSA_T, grpn_params['modelEASZT_LT_5'], grpn_params['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, grps_params['modelEASZT_LT'],'.-', label='South Island')
ax12.fill_between(pSA_T, grps_params['modelEASZT_LT_5'], grps_params['modelEASZT_LT_95'], alpha = 0.2)

ax20.semilogx(pSA_T, nz_params['modelEASZT_wS'],'.-',label = 'NZ-Wide', lw = 1)
ax20.fill_between(pSA_T, nz_params['modelEASZT_wS_5'], nz_params['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, grp1_params['modelEASZT_wS'],'.-',label = 'Wellington')
ax20.fill_between(pSA_T, grp1_params['modelEASZT_wS_5'], grp1_params['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, grp2_params['modelEASZT_wS'],'.-', label='Canterbury')
ax20.fill_between(pSA_T, grp2_params['modelEASZT_wS_5'], grp2_params['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, grpn_params['modelEASZT_wS'],'.-', label='North Island')
ax20.fill_between(pSA_T, grpn_params['modelEASZT_wS_5'], grpn_params['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, grps_params['modelEASZT_wS'],'.-', label='South Island')
ax20.fill_between(pSA_T, grps_params['modelEASZT_wS_5'], grps_params['modelEASZT_wS_95'], alpha = 0.2)

ax21.semilogx(pSA_T, nz_params['modelEASZT_wZ'],'.-',label = 'NZ-Wide', lw = 1)
ax21.fill_between(pSA_T, nz_params['modelEASZT_wZ_5'], nz_params['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, grp1_params['modelEASZT_wZ'],'.-',label = 'Wellington')
ax21.fill_between(pSA_T, grp1_params['modelEASZT_wZ_5'], grp1_params['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, grp2_params['modelEASZT_wZ'],'.-', label='Canterbury')
ax21.fill_between(pSA_T, grp2_params['modelEASZT_wZ_5'], grp2_params['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, grpn_params['modelEASZT_wZ'],'.-', label='North Island')
ax21.fill_between(pSA_T, grpn_params['modelEASZT_wZ_5'], grpn_params['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, grps_params['modelEASZT_wZ'],'.-', label='South Island')
ax21.fill_between(pSA_T, grps_params['modelEASZT_wZ_5'], grps_params['modelEASZT_wZ_95'], alpha = 0.2)

ax22.semilogx(pSA_T, nz_params['modelEASZT_wT'],'.-',label = 'NZ-Wide', lw = 1)
ax22.fill_between(pSA_T, nz_params['modelEASZT_wT_5'], nz_params['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, grp1_params['modelEASZT_wT'],'.-',label = 'Wellington')
ax22.fill_between(pSA_T, grp1_params['modelEASZT_wT_5'], grp1_params['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, grp2_params['modelEASZT_wT'],'.-', label='Canterbury')
ax22.fill_between(pSA_T, grp2_params['modelEASZT_wT_5'], grp2_params['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, grpn_params['modelEASZT_wT'],'.-', label='North Island')
ax22.fill_between(pSA_T, grpn_params['modelEASZT_wT_5'], grpn_params['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, grps_params['modelEASZT_wT'],'.-', label='South Island')
ax22.fill_between(pSA_T, grps_params['modelEASZT_wT_5'], grps_params['modelEASZT_wT_95'], alpha = 0.2)

ax30.semilogx(pSA_T, nz_params['modelEASZT_w'],'.-',label = 'NZ-Wide', lw = 1)
ax30.fill_between(pSA_T, nz_params['modelEASZT_w_5'], nz_params['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, grp1_params['modelEASZT_w'],'.-',label = 'Wellington')
ax30.fill_between(pSA_T, grp1_params['modelEASZT_w_5'], grp1_params['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, grp2_params['modelEASZT_w'],'.-', label='Canterbury')
ax30.fill_between(pSA_T, grp2_params['modelEASZT_w_5'], grp2_params['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, grpn_params['modelEASZT_w'],'.-', label='North Island')
ax30.fill_between(pSA_T, grpn_params['modelEASZT_w_5'], grpn_params['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, grps_params['modelEASZT_w'],'.-', label='South Island')
ax30.fill_between(pSA_T, grps_params['modelEASZT_w_5'], grps_params['modelEASZT_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax20.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$L_T$ (m)')
ax20.set_ylabel(r'$w_S$')
ax21.set_ylabel(r'$w_Z$')
ax22.set_ylabel(r'$w_T$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax20.set_xlim(0.01, 6)
ax21.set_xlim(0.01, 6)
ax22.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(0.1,500)
ax01.set_ylim(0,2)
ax02.set_ylim(0,45)
ax10.set_ylim(1,5000)
ax11.set_ylim(1,5000)
ax12.set_ylim(1,500)
ax20.set_ylim(0,1)
ax21.set_ylim(0,1)
ax22.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax20, ax21, ax22, ax30]:
    ax.grid(which='both')
ax01.legend(loc='upper center', ncol=2, fontsize = 9.7)
plt.show()

# tectonic class
fig = plt.figure(figsize=(12,9),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(4, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax20 = fig.add_subplot(gs[2,0])
ax21 = fig.add_subplot(gs[2,1])
ax22 = fig.add_subplot(gs[2,2])
ax30 = fig.add_subplot(gs[3,0:3])

ax00.loglog(pSA_T, nz_params['modelEASZT_LE'],'.-',label = 'NZ-Wide: All')
ax00.fill_between(pSA_T, nz_params['modelEASZT_LE_5'],nz_params['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, nz_params_crus['modelEASZT_LE'],'.-',label = 'NZ-Wide: Crus')
ax00.fill_between(pSA_T, nz_params_crus['modelEASZT_LE_5'],nz_params_crus['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, nz_params_sub['modelEASZT_LE'],'.-', label='NZ-Wide: Sub')
ax00.fill_between(pSA_T, nz_params_sub['modelEASZT_LE_5'],nz_params_sub['modelEASZT_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, nz_params['modelEASZT_gammaE'],'.-',label = 'NZ-Wide: All', lw = 1)
ax01.fill_between(pSA_T, nz_params['modelEASZT_gammaE_5'], nz_params['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, nz_params_crus['modelEASZT_gammaE'],'.-',label = 'NZ-Wide: Crus')
ax01.fill_between(pSA_T, nz_params_crus['modelEASZT_gammaE_5'], nz_params_crus['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, nz_params_sub['modelEASZT_gammaE'],'.-', label='NZ-Wide: Sub')
ax01.fill_between(pSA_T, nz_params_sub['modelEASZT_gammaE_5'], nz_params_sub['modelEASZT_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, nz_params['modelEASZT_LA'],'.-',label = 'NZ-Wide: All', lw = 1)
ax02.fill_between(pSA_T, nz_params['modelEASZT_LA_5'], nz_params['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, nz_params_crus['modelEASZT_LA'],'.-',label = 'NZ-Wide: Crus')
ax02.fill_between(pSA_T, nz_params_crus['modelEASZT_LA_5'], nz_params_crus['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, nz_params_sub['modelEASZT_LA'],'.-', label='NZ-Wide: Sub')
ax02.fill_between(pSA_T, nz_params_sub['modelEASZT_LA_5'], nz_params_sub['modelEASZT_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, nz_params['modelEASZT_LS'],'.-',label = 'NZ-Wide: All', lw = 1)
ax10.fill_between(pSA_T, nz_params['modelEASZT_LS_5'], nz_params['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, nz_params_crus['modelEASZT_LS'],'.-',label = 'NZ-Wide: Crus')
ax10.fill_between(pSA_T, nz_params_crus['modelEASZT_LS_5'], nz_params_crus['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, nz_params_sub['modelEASZT_LS'],'.-', label='NZ-Wide: Sub')
ax10.fill_between(pSA_T, nz_params_sub['modelEASZT_LS_5'], nz_params_sub['modelEASZT_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, nz_params['modelEASZT_LZ'],'.-',label = 'NZ-Wide: All', lw = 1)
ax11.fill_between(pSA_T, nz_params['modelEASZT_LZ_5'], nz_params['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, nz_params_crus['modelEASZT_LZ'],'.-',label = 'NZ-Wide: Crus')
ax11.fill_between(pSA_T, nz_params_crus['modelEASZT_LZ_5'], nz_params_crus['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, nz_params_sub['modelEASZT_LZ'],'.-', label='NZ-Wide: Sub')
ax11.fill_between(pSA_T, nz_params_sub['modelEASZT_LZ_5'], nz_params_sub['modelEASZT_LZ_95'], alpha = 0.2)

ax12.loglog(pSA_T, nz_params['modelEASZT_LT'],'.-',label = 'NZ-Wide: All', lw = 1)
ax12.fill_between(pSA_T, nz_params['modelEASZT_LT_5'], nz_params['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, nz_params_crus['modelEASZT_LT'],'.-',label = 'NZ-Wide: Crus')
ax12.fill_between(pSA_T, nz_params_crus['modelEASZT_LT_5'], nz_params_crus['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, nz_params_sub['modelEASZT_LT'],'.-', label='NZ-Wide: Sub')
ax12.fill_between(pSA_T, nz_params_sub['modelEASZT_LT_5'], nz_params_sub['modelEASZT_LT_95'], alpha = 0.2)

ax20.semilogx(pSA_T, nz_params['modelEASZT_wS'],'.-',label = 'NZ-Wide: All', lw = 1)
ax20.fill_between(pSA_T, nz_params['modelEASZT_wS_5'], nz_params['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, nz_params_crus['modelEASZT_wS'],'.-',label = 'NZ-Wide: Crus')
ax20.fill_between(pSA_T, nz_params_crus['modelEASZT_wS_5'], nz_params_crus['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, nz_params_sub['modelEASZT_wS'],'.-', label='NZ-Wide: Sub')
ax20.fill_between(pSA_T, nz_params_sub['modelEASZT_wS_5'], nz_params_sub['modelEASZT_wS_95'], alpha = 0.2)

ax21.semilogx(pSA_T, nz_params['modelEASZT_wZ'],'.-',label = 'NZ-Wide: All', lw = 1)
ax21.fill_between(pSA_T, nz_params['modelEASZT_wZ_5'], nz_params['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, nz_params_crus['modelEASZT_wZ'],'.-',label = 'NZ-Wide: Crus')
ax21.fill_between(pSA_T, nz_params_crus['modelEASZT_wZ_5'], nz_params_crus['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, nz_params_sub['modelEASZT_wZ'],'.-', label='NZ-Wide: Sub')
ax21.fill_between(pSA_T, nz_params_sub['modelEASZT_wZ_5'], nz_params_sub['modelEASZT_wZ_95'], alpha = 0.2)

ax22.semilogx(pSA_T, nz_params['modelEASZT_wT'],'.-',label = 'NZ-Wide: All', lw = 1)
ax22.fill_between(pSA_T, nz_params['modelEASZT_wT_5'], nz_params['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, nz_params_crus['modelEASZT_wT'],'.-',label = 'NZ-Wide: Crus')
ax22.fill_between(pSA_T, nz_params_crus['modelEASZT_wT_5'], nz_params_crus['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, nz_params_sub['modelEASZT_wT'],'.-', label='NZ-Wide: Sub')
ax22.fill_between(pSA_T, nz_params_sub['modelEASZT_wT_5'], nz_params_sub['modelEASZT_wT_95'], alpha = 0.2)

ax30.semilogx(pSA_T, nz_params['modelEASZT_w'],'.-',label = 'NZ-Wide: All', lw = 1)
ax30.fill_between(pSA_T, nz_params['modelEASZT_w_5'], nz_params['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, nz_params_crus['modelEASZT_w'],'.-',label = 'NZ-Wide: Crus')
ax30.fill_between(pSA_T, nz_params_crus['modelEASZT_w_5'], nz_params_crus['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, nz_params_sub['modelEASZT_w'],'.-', label='NZ-Wide: Sub')
ax30.fill_between(pSA_T, nz_params_sub['modelEASZT_w_5'], nz_params_sub['modelEASZT_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax20.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$L_T$ (m)')
ax20.set_ylabel(r'$w_S$')
ax21.set_ylabel(r'$w_Z$')
ax22.set_ylabel(r'$w_T$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax20.set_xlim(0.01, 6)
ax21.set_xlim(0.01, 6)
ax22.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(0.1,500)
ax01.set_ylim(0,2)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(10,5000)
ax12.set_ylim(1,500)
ax20.set_ylim(0,1)
ax21.set_ylim(0,1)
ax22.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax20, ax21, ax22, ax30]:
    ax.grid(which='both')
ax01.legend(loc='upper center', fontsize = 9.7)
plt.show()

### 6.5.7 NZ-wide SCM

In [ ]:
models = [
    'modelE', 'modelEA', 'modelEAS', 'modelEASZ',
    'modelEAST', 'modelEASZT'
]

cmap = plt.get_cmap('tab10')
model_colors = {m: cmap(i) for i, m in enumerate(models)}


fig, ax = plt.subplots(3,3, figsize = (12,8), dpi = 200, constrained_layout = True)
ax[0,0].loglog(pSA_T, nz_params['modelE_LE'],'.-',color=model_colors['modelE'], label = 'Model E')
ax[0,0].fill_between(pSA_T, nz_params['modelE_LE_5'],nz_params['modelE_LE_95'], color=model_colors['modelE'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEA_LE'],'.-', color=model_colors['modelEA'], label = 'Model EA')
ax[0,0].fill_between(pSA_T, nz_params['modelEA_LE_5'],nz_params['modelEA_LE_95'], color=model_colors['modelEA'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEAS_LE'],'.-', color=model_colors['modelEAS'], label = 'Model EAS')
ax[0,0].fill_between(pSA_T, nz_params['modelEAS_LE_5'],nz_params['modelEAS_LE_95'], color=model_colors['modelEAS'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEASZ_LE'],'.-',color=model_colors['modelEASZ'], label = 'Model EASZ')
ax[0,0].fill_between(pSA_T, nz_params['modelEASZ_LE_5'],nz_params['modelEASZ_LE_95'], color=model_colors['modelEASZ'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEAST_LE'],'.-',color=model_colors['modelEAST'], label = 'Model EAST')
ax[0,0].fill_between(pSA_T, nz_params['modelEAST_LE_5'],nz_params['modelEAST_LE_95'], color=model_colors['modelEAST'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEASZT_LE'],'.-',color=model_colors['modelEASZT'], label = 'Model EASZT')
ax[0,0].fill_between(pSA_T, nz_params['modelEASZT_LE_5'],nz_params['modelEASZT_LE_95'], color=model_colors['modelEASZT'], alpha = 0.2)
ax[0,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LE'],'.-',color='black', label = 'BM23EAS')
ax[0,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LE_5'],BM23_parameters['modelEAS_LE_95'], color='black', alpha = 0.2)

ax[0,1].semilogx(pSA_T, nz_params['modelE_gammaE'],'.-',color=model_colors['modelE'],label='Model E')
ax[0,1].fill_between(pSA_T, nz_params['modelE_gammaE_5'],nz_params['modelE_gammaE_95'], color=model_colors['modelE'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEA_gammaE'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,1].fill_between(pSA_T, nz_params['modelEA_gammaE_5'],nz_params['modelEA_gammaE_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEAS_gammaE'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,1].fill_between(pSA_T, nz_params['modelEAS_gammaE_5'],nz_params['modelEAS_gammaE_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEASZ_gammaE'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,1].fill_between(pSA_T, nz_params['modelEASZ_gammaE_5'],nz_params['modelEASZ_gammaE_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEAST_gammaE'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,1].fill_between(pSA_T, nz_params['modelEAST_gammaE_5'],nz_params['modelEAST_gammaE_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEASZT_gammaE'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,1].fill_between(pSA_T, nz_params['modelEASZT_gammaE_5'],nz_params['modelEASZT_gammaE_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE'],'.-',color='black', label = 'BM23EAS')
ax[0,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE_5'],BM23_parameters['modelEAS_gammaE_95'], color='black', alpha = 0.2)

ax[0,2].semilogx(pSA_T, nz_params['modelEA_LA'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,2].fill_between(pSA_T, nz_params['modelEA_LA_5'],nz_params['modelEA_LA_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,2].semilogx(pSA_T, nz_params['modelEAS_LA'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,2].fill_between(pSA_T, nz_params['modelEAS_LA_5'],nz_params['modelEAS_LA_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,2].semilogx(pSA_T, nz_params['modelEASZ_LA'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,2].fill_between(pSA_T, nz_params['modelEASZ_LA_5'],nz_params['modelEASZ_LA_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,2].semilogx(pSA_T, nz_params['modelEAST_LA'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,2].fill_between(pSA_T, nz_params['modelEAST_LA_5'],nz_params['modelEAST_LA_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,2].semilogx(pSA_T, nz_params['modelEASZT_LA'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,2].fill_between(pSA_T, nz_params['modelEASZT_LA_5'],nz_params['modelEASZT_LA_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,2].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_LA'],'.-',color='black', label = 'BM23EAS')
ax[0,2].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LA_5'],BM23_parameters['modelEAS_LA_95'], color='black', alpha = 0.2)

ax[1,0].loglog(pSA_T, nz_params['modelEAS_LS'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,0].fill_between(pSA_T, nz_params['modelEAS_LS_5'],nz_params['modelEAS_LS_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,0].loglog(pSA_T, nz_params['modelEASZ_LS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,0].fill_between(pSA_T, nz_params['modelEASZ_LS_5'],nz_params['modelEASZ_LS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,0].loglog(pSA_T, nz_params['modelEAST_LS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,0].fill_between(pSA_T, nz_params['modelEAST_LS_5'],nz_params['modelEAST_LS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,0].loglog(pSA_T, nz_params['modelEASZT_LS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,0].fill_between(pSA_T, nz_params['modelEASZT_LS_5'],nz_params['modelEASZT_LS_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LS'],'.-',color='black', label = 'BM23EAS')
ax[1,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LS_5'],BM23_parameters['modelEAS_LS_95'], color='black', alpha = 0.2)

ax[1,1].semilogx(pSA_T, nz_params['modelEAS_w'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,1].fill_between(pSA_T, nz_params['modelEAS_w_5'],nz_params['modelEAS_w_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,1].semilogx(pSA_T, nz_params['modelEASZ_w'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,1].fill_between(pSA_T, nz_params['modelEASZ_w_5'],nz_params['modelEASZ_w_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,1].semilogx(pSA_T, nz_params['modelEAST_w'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,1].fill_between(pSA_T, nz_params['modelEAST_w_5'],nz_params['modelEAST_w_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,1].semilogx(pSA_T, nz_params['modelEASZT_w'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,1].fill_between(pSA_T, nz_params['modelEASZT_w_5'],nz_params['modelEASZT_w_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_w'],'.-',color='black', label = 'BM23EAS')
ax[1,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_w_5'],BM23_parameters['modelEAS_w_95'], color='black', alpha = 0.2)

ax[1,2].loglog(pSA_T, nz_params['modelEASZ_LZ'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,2].fill_between(pSA_T, nz_params['modelEASZ_LZ_5'],nz_params['modelEASZ_LZ_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,2].loglog(pSA_T, nz_params['modelEASZT_LZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,2].fill_between(pSA_T, nz_params['modelEASZT_LZ_5'],nz_params['modelEASZT_LZ_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,0].semilogx(pSA_T, nz_params['modelEASZ_wS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[2,0].fill_between(pSA_T, nz_params['modelEASZ_wS_5'],nz_params['modelEASZ_wS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[2,0].semilogx(pSA_T, nz_params['modelEAST_wS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,0].fill_between(pSA_T, nz_params['modelEAST_wS_5'],nz_params['modelEAST_wS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,0].semilogx(pSA_T, nz_params['modelEASZT_wS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,0].fill_between(pSA_T, nz_params['modelEASZT_wS_5'],nz_params['modelEASZT_wS_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,1].loglog(pSA_T, nz_params['modelEAST_LT'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,1].fill_between(pSA_T, nz_params['modelEAST_LT_5'],nz_params['modelEAST_LT_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,1].loglog(pSA_T, nz_params['modelEASZT_LT'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,1].fill_between(pSA_T, nz_params['modelEASZT_LT_5'],nz_params['modelEASZT_LT_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,2].semilogx(pSA_T, nz_params['modelEASZT_wZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT: wZ')
ax[2,2].fill_between(pSA_T, nz_params['modelEASZT_wZ_5'],nz_params['modelEASZT_wZ_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[2,2].semilogx(pSA_T, nz_params['modelEASZT_wT'],'--',color=model_colors['modelEASZT'],label='Model EASZT: wT')
ax[2,2].fill_between(pSA_T, nz_params['modelEASZT_wT_5'],nz_params['modelEASZT_wT_95'], color=model_colors['modelEASZT'], alpha=0.2)

for i in range(3):
    for j in range(3):
        ax[i,j].grid(which='both', linestyle = '--', alpha = 0.3)
        ax[i,j].set_xlim(0.01,6)
        ax[i,j].set_xlabel('Period, T(s)')
        if ax[i,j].get_yscale() == 'log':
            # Log axis → use LogFormatter, no ticklabel_format allowed
            ax[i,j].yaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))

ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A\ (^\circ)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$w$')
ax[1,2].set_ylabel(r'$L_Z$ (m)')
ax[2,0].set_ylabel(r'$w_S$')
ax[2,1].set_ylabel(r'$L_T$ (m)')
ax[2,2].set_ylabel(r'$w_{Z},\; w_{T}$')

ax[0,1].set_ylim(0,2)
ax[0,2].set_ylim(0,45)
ax[1,1].set_ylim(0,1)
ax[2,0].set_ylim(0,1)
ax[2,2].set_ylim(0,1)
ax[0,1].legend(loc="upper center", ncols = 2)
ax[2,2].legend(loc="upper right")
plt.savefig(FIGURES_DIR / "nz_wide_parameters.png", dpi = 300)
plt.show(fig)

In [ ]:
models = [
    'modelE', 'modelEA', 'modelEAS', 'modelEASZ',
    'modelEAST', 'modelEASZT'
]

cmap = plt.get_cmap('tab10')
model_colors = {m: cmap(i) for i, m in enumerate(models)}


fig, ax = plt.subplots(3,4, figsize = (12,8), dpi = 200, constrained_layout = True)
ax[0,0].loglog(pSA_T, nz_params['modelE_LE'],'.-',color=model_colors['modelE'], label = 'Model E')
ax[0,0].fill_between(pSA_T, nz_params['modelE_LE_5'],nz_params['modelE_LE_95'], color=model_colors['modelE'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEA_LE'],'.-', color=model_colors['modelEA'], label = 'Model EA')
ax[0,0].fill_between(pSA_T, nz_params['modelEA_LE_5'],nz_params['modelEA_LE_95'], color=model_colors['modelEA'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEAS_LE'],'.-', color=model_colors['modelEAS'], label = 'Model EAS')
ax[0,0].fill_between(pSA_T, nz_params['modelEAS_LE_5'],nz_params['modelEAS_LE_95'], color=model_colors['modelEAS'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEASZ_LE'],'.-',color=model_colors['modelEASZ'], label = 'Model EASZ')
ax[0,0].fill_between(pSA_T, nz_params['modelEASZ_LE_5'],nz_params['modelEASZ_LE_95'], color=model_colors['modelEASZ'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEAST_LE'],'.-',color=model_colors['modelEAST'], label = 'Model EAST')
ax[0,0].fill_between(pSA_T, nz_params['modelEAST_LE_5'],nz_params['modelEAST_LE_95'], color=model_colors['modelEAST'], alpha = 0.2)
ax[0,0].loglog(pSA_T, nz_params['modelEASZT_LE'],'.-',color=model_colors['modelEASZT'], label = 'Model EASZT')
ax[0,0].fill_between(pSA_T, nz_params['modelEASZT_LE_5'],nz_params['modelEASZT_LE_95'], color=model_colors['modelEASZT'], alpha = 0.2)

ax[0,1].semilogx(pSA_T, nz_params['modelE_gammaE'],'.-',color=model_colors['modelE'],label='Model E')
ax[0,1].fill_between(pSA_T, nz_params['modelE_gammaE_5'],nz_params['modelE_gammaE_95'], color=model_colors['modelE'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEA_gammaE'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,1].fill_between(pSA_T, nz_params['modelEA_gammaE_5'],nz_params['modelEA_gammaE_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEAS_gammaE'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,1].fill_between(pSA_T, nz_params['modelEAS_gammaE_5'],nz_params['modelEAS_gammaE_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEASZ_gammaE'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,1].fill_between(pSA_T, nz_params['modelEASZ_gammaE_5'],nz_params['modelEASZ_gammaE_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEAST_gammaE'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,1].fill_between(pSA_T, nz_params['modelEAST_gammaE_5'],nz_params['modelEAST_gammaE_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,1].semilogx(pSA_T, nz_params['modelEASZT_gammaE'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,1].fill_between(pSA_T, nz_params['modelEASZT_gammaE_5'],nz_params['modelEASZT_gammaE_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[0,2].loglog(pSA_T, nz_params['modelEA_LA'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,2].fill_between(pSA_T, nz_params['modelEA_LA_5'],nz_params['modelEA_LA_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,2].loglog(pSA_T, nz_params['modelEAS_LA'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,2].fill_between(pSA_T, nz_params['modelEAS_LA_5'],nz_params['modelEAS_LA_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,2].loglog(pSA_T, nz_params['modelEASZ_LA'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,2].fill_between(pSA_T, nz_params['modelEASZ_LA_5'],nz_params['modelEASZ_LA_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,2].loglog(pSA_T, nz_params['modelEAST_LA'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,2].fill_between(pSA_T, nz_params['modelEAST_LA_5'],nz_params['modelEAST_LA_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,2].loglog(pSA_T, nz_params['modelEASZT_LA'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,2].fill_between(pSA_T, nz_params['modelEASZT_LA_5'],nz_params['modelEASZT_LA_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[0,3].loglog(pSA_T, nz_params['modelEAS_LS'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,3].fill_between(pSA_T, nz_params['modelEAS_LS_5'],nz_params['modelEAS_LS_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,3].loglog(pSA_T, nz_params['modelEASZ_LS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,3].fill_between(pSA_T, nz_params['modelEASZ_LS_5'],nz_params['modelEASZ_LS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,3].loglog(pSA_T, nz_params['modelEAST_LS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,3].fill_between(pSA_T, nz_params['modelEAST_LS_5'],nz_params['modelEAST_LS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,3].loglog(pSA_T, nz_params['modelEASZT_LS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,3].fill_between(pSA_T, nz_params['modelEASZT_LS_5'],nz_params['modelEASZT_LS_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[1,0].semilogx(pSA_T, nz_params['modelEAS_w'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,0].fill_between(pSA_T, nz_params['modelEAS_w_5'],nz_params['modelEAS_w_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,0].semilogx(pSA_T, nz_params['modelEASZ_w'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,0].fill_between(pSA_T, nz_params['modelEASZ_w_5'],nz_params['modelEASZ_w_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,0].semilogx(pSA_T, nz_params['modelEAST_w'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,0].fill_between(pSA_T, nz_params['modelEAST_w_5'],nz_params['modelEAST_w_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,0].semilogx(pSA_T, nz_params['modelEASZT_w'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,0].fill_between(pSA_T, nz_params['modelEASZT_w_5'],nz_params['modelEASZT_w_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[1,1].loglog(pSA_T, nz_params['modelEASZ_LZ'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,1].fill_between(pSA_T, nz_params['modelEASZ_LZ_5'],nz_params['modelEASZ_LZ_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,1].loglog(pSA_T, nz_params['modelEASZT_LZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,1].fill_between(pSA_T, nz_params['modelEASZT_LZ_5'],nz_params['modelEASZT_LZ_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[1,2].semilogx(pSA_T, nz_params['modelEASZ_wS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,2].fill_between(pSA_T, nz_params['modelEASZ_wS_5'],nz_params['modelEASZ_wS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,2].semilogx(pSA_T, nz_params['modelEAST_wS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,2].fill_between(pSA_T, nz_params['modelEAST_wS_5'],nz_params['modelEAST_wS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,2].semilogx(pSA_T, nz_params['modelEASZT_wS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,2].fill_between(pSA_T, nz_params['modelEASZT_wS_5'],nz_params['modelEASZT_wS_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[1,3].loglog(pSA_T, nz_params['modelEAST_LT'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,3].fill_between(pSA_T, nz_params['modelEAST_LT_5'],nz_params['modelEAST_LT_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,3].loglog(pSA_T, nz_params['modelEASZT_LT'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,3].fill_between(pSA_T, nz_params['modelEASZT_LT_5'],nz_params['modelEASZT_LT_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,0].semilogx(pSA_T, nz_params['modelEASZT_wZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,0].fill_between(pSA_T, nz_params['modelEASZT_wZ_5'],nz_params['modelEASZT_wZ_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,1].semilogx(pSA_T, nz_params['modelEASZT_wT'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,1].fill_between(pSA_T, nz_params['modelEASZT_wT_5'],nz_params['modelEASZT_wT_95'], color=model_colors['modelEASZT'], alpha=0.2)

for i in range(3):
    for j in range(4):
        ax[i,j].grid(which='both', linestyle = '--', alpha = 0.3)
        ax[i,j].set_xlim(0.01,6)
        ax[i,j].set_xlabel('Period, T(s)')
        if ax[i,j].get_yscale() == 'log':
            # Log axis → use LogFormatter, no ticklabel_format allowed
            ax[i,j].yaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))

ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A\ (^\circ)$')
ax[0,3].set_ylabel(r'$L_S$ (m/s)')
ax[1,0].set_ylabel(r'$w$')
ax[1,1].set_ylabel(r'$L_Z$ (m)')
ax[1,2].set_ylabel(r'$w_S$')
ax[1,3].set_ylabel(r'$L_T$ (m)')
ax[2,0].set_ylabel(r'$w_Z$')
ax[2,1].set_ylabel(r'$w_T$')

ax[0,1].set_ylim(0,2)
ax[1,0].set_ylim(0,1)
ax[1,2].set_ylim(0,1)
ax[2,0].set_ylim(0,1)
ax[2,1].set_ylim(0,1)
ax[2,2].set_axis_off()
ax[2,3].set_visible(False)
ax[2,2].legend(*ax[0,0].get_legend_handles_labels(),loc='center',fontsize=12,frameon=True)
plt.show(fig)

### 6.5.8 Wellington SCM

In [ ]:
models = [
    'modelE', 'modelEA', 'modelEAS', 'modelEASZ',
    'modelEAST', 'modelEASZT'
]

cmap = plt.get_cmap('tab10')
model_colors = {m: cmap(i) for i, m in enumerate(models)}


fig, ax = plt.subplots(3,3, figsize = (12,8), dpi = 200, constrained_layout = True)
ax[0,0].loglog(pSA_T, grp1_params['modelE_LE'],'.-',color=model_colors['modelE'], label = 'Model E')
ax[0,0].fill_between(pSA_T, grp1_params['modelE_LE_5'],grp1_params['modelE_LE_95'], color=model_colors['modelE'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp1_params['modelEA_LE'],'.-', color=model_colors['modelEA'], label = 'Model EA')
ax[0,0].fill_between(pSA_T, grp1_params['modelEA_LE_5'],grp1_params['modelEA_LE_95'], color=model_colors['modelEA'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp1_params['modelEAS_LE'],'.-', color=model_colors['modelEAS'], label = 'Model EAS')
ax[0,0].fill_between(pSA_T, grp1_params['modelEAS_LE_5'],grp1_params['modelEAS_LE_95'], color=model_colors['modelEAS'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp1_params['modelEASZ_LE'],'.-',color=model_colors['modelEASZ'], label = 'Model EASZ')
ax[0,0].fill_between(pSA_T, grp1_params['modelEASZ_LE_5'],grp1_params['modelEASZ_LE_95'], color=model_colors['modelEASZ'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp1_params['modelEAST_LE'],'.-',color=model_colors['modelEAST'], label = 'Model EAST')
ax[0,0].fill_between(pSA_T, grp1_params['modelEAST_LE_5'],grp1_params['modelEAST_LE_95'], color=model_colors['modelEAST'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp1_params['modelEASZT_LE'],'.-',color=model_colors['modelEASZT'], label = 'Model EASZT')
ax[0,0].fill_between(pSA_T, grp1_params['modelEASZT_LE_5'],grp1_params['modelEASZT_LE_95'], color=model_colors['modelEASZT'], alpha = 0.2)
ax[0,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LE'],'.-',color='black', label = 'BM23EAS')
ax[0,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LE_5'],BM23_parameters['modelEAS_LE_95'], color='black', alpha = 0.2)

ax[0,1].semilogx(pSA_T, grp1_params['modelE_gammaE'],'.-',color=model_colors['modelE'],label='Model E')
ax[0,1].fill_between(pSA_T, grp1_params['modelE_gammaE_5'],grp1_params['modelE_gammaE_95'], color=model_colors['modelE'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp1_params['modelEA_gammaE'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,1].fill_between(pSA_T, grp1_params['modelEA_gammaE_5'],grp1_params['modelEA_gammaE_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp1_params['modelEAS_gammaE'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,1].fill_between(pSA_T, grp1_params['modelEAS_gammaE_5'],grp1_params['modelEAS_gammaE_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp1_params['modelEASZ_gammaE'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,1].fill_between(pSA_T, grp1_params['modelEASZ_gammaE_5'],grp1_params['modelEASZ_gammaE_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp1_params['modelEAST_gammaE'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,1].fill_between(pSA_T, grp1_params['modelEAST_gammaE_5'],grp1_params['modelEAST_gammaE_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp1_params['modelEASZT_gammaE'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,1].fill_between(pSA_T, grp1_params['modelEASZT_gammaE_5'],grp1_params['modelEASZT_gammaE_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE'],'.-',color='black', label = 'BM23EAS')
ax[0,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE_5'],BM23_parameters['modelEAS_gammaE_95'], color='black', alpha = 0.2)

ax[0,2].semilogx(pSA_T, grp1_params['modelEA_LA'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,2].fill_between(pSA_T, grp1_params['modelEA_LA_5'],grp1_params['modelEA_LA_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp1_params['modelEAS_LA'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,2].fill_between(pSA_T, grp1_params['modelEAS_LA_5'],grp1_params['modelEAS_LA_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp1_params['modelEASZ_LA'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,2].fill_between(pSA_T, grp1_params['modelEASZ_LA_5'],grp1_params['modelEASZ_LA_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp1_params['modelEAST_LA'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,2].fill_between(pSA_T, grp1_params['modelEAST_LA_5'],grp1_params['modelEAST_LA_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp1_params['modelEASZT_LA'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,2].fill_between(pSA_T, grp1_params['modelEASZT_LA_5'],grp1_params['modelEASZT_LA_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,2].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_LA'],'.-',color='black', label = 'BM23EAS')
ax[0,2].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LA_5'],BM23_parameters['modelEAS_LA_95'], color='black', alpha = 0.2)

ax[1,0].loglog(pSA_T, grp1_params['modelEAS_LS'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,0].fill_between(pSA_T, grp1_params['modelEAS_LS_5'],grp1_params['modelEAS_LS_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,0].loglog(pSA_T, grp1_params['modelEASZ_LS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,0].fill_between(pSA_T, grp1_params['modelEASZ_LS_5'],grp1_params['modelEASZ_LS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,0].loglog(pSA_T, grp1_params['modelEAST_LS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,0].fill_between(pSA_T, grp1_params['modelEAST_LS_5'],grp1_params['modelEAST_LS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,0].loglog(pSA_T, grp1_params['modelEASZT_LS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,0].fill_between(pSA_T, grp1_params['modelEASZT_LS_5'],grp1_params['modelEASZT_LS_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LS'],'.-',color='black', label = 'BM23EAS')
ax[1,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LS_5'],BM23_parameters['modelEAS_LS_95'], color='black', alpha = 0.2)

ax[1,1].semilogx(pSA_T, grp1_params['modelEAS_w'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,1].fill_between(pSA_T, grp1_params['modelEAS_w_5'],grp1_params['modelEAS_w_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grp1_params['modelEASZ_w'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,1].fill_between(pSA_T, grp1_params['modelEASZ_w_5'],grp1_params['modelEASZ_w_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grp1_params['modelEAST_w'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,1].fill_between(pSA_T, grp1_params['modelEAST_w_5'],grp1_params['modelEAST_w_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grp1_params['modelEASZT_w'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,1].fill_between(pSA_T, grp1_params['modelEASZT_w_5'],grp1_params['modelEASZT_w_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_w'],'.-',color='black', label = 'BM23EAS')
ax[1,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_w_5'],BM23_parameters['modelEAS_w_95'], color='black', alpha = 0.2)

ax[1,2].loglog(pSA_T, grp1_params['modelEASZ_LZ'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,2].fill_between(pSA_T, grp1_params['modelEASZ_LZ_5'],grp1_params['modelEASZ_LZ_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,2].loglog(pSA_T, grp1_params['modelEASZT_LZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,2].fill_between(pSA_T, grp1_params['modelEASZT_LZ_5'],grp1_params['modelEASZT_LZ_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,0].semilogx(pSA_T, grp1_params['modelEASZ_wS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[2,0].fill_between(pSA_T, grp1_params['modelEASZ_wS_5'],grp1_params['modelEASZ_wS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grp1_params['modelEAST_wS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,0].fill_between(pSA_T, grp1_params['modelEAST_wS_5'],grp1_params['modelEAST_wS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grp1_params['modelEASZT_wS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,0].fill_between(pSA_T, grp1_params['modelEASZT_wS_5'],grp1_params['modelEASZT_wS_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,1].loglog(pSA_T, grp1_params['modelEAST_LT'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,1].fill_between(pSA_T, grp1_params['modelEAST_LT_5'],grp1_params['modelEAST_LT_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,1].loglog(pSA_T, grp1_params['modelEASZT_LT'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,1].fill_between(pSA_T, grp1_params['modelEASZT_LT_5'],grp1_params['modelEASZT_LT_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,2].semilogx(pSA_T, grp1_params['modelEASZT_wZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT: wZ')
ax[2,2].fill_between(pSA_T, grp1_params['modelEASZT_wZ_5'],grp1_params['modelEASZT_wZ_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[2,2].semilogx(pSA_T, grp1_params['modelEASZT_wT'],'--',color=model_colors['modelEASZT'],label='Model EASZT: wT')
ax[2,2].fill_between(pSA_T, grp1_params['modelEASZT_wT_5'],grp1_params['modelEASZT_wT_95'], color=model_colors['modelEASZT'], alpha=0.2)

for i in range(3):
    for j in range(3):
        ax[i,j].grid(which='both', linestyle = '--', alpha = 0.3)
        ax[i,j].set_xlim(0.01,6)
        ax[i,j].set_xlabel('Period, T(s)')
        if ax[i,j].get_yscale() == 'log':
            # Log axis → use LogFormatter, no ticklabel_format allowed
            ax[i,j].yaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))

ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A\ (^\circ)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$w$')
ax[1,2].set_ylabel(r'$L_Z$ (m)')
ax[2,0].set_ylabel(r'$w_S$')
ax[2,1].set_ylabel(r'$L_T$ (m)')
ax[2,2].set_ylabel(r'$w_{Z},\; w_{T}$')

ax[0,1].set_ylim(0,2)
ax[0,2].set_ylim(0,45)
ax[1,1].set_ylim(0,1)
ax[2,0].set_ylim(0,1)
ax[2,2].set_ylim(0,1)
ax[0,1].legend(loc="upper center", ncols = 2)
ax[2,2].legend(loc="upper right")
plt.show(fig)

### 6.5.9 Canterbury SCM

In [ ]:
models = [
    'modelE', 'modelEA', 'modelEAS', 'modelEASZ',
    'modelEAST', 'modelEASZT'
]

cmap = plt.get_cmap('tab10')
model_colors = {m: cmap(i) for i, m in enumerate(models)}


fig, ax = plt.subplots(3,3, figsize = (12,8), dpi = 200, constrained_layout = True)
ax[0,0].loglog(pSA_T, grp2_params['modelE_LE'],'.-',color=model_colors['modelE'], label = 'Model E')
ax[0,0].fill_between(pSA_T, grp2_params['modelE_LE_5'],grp2_params['modelE_LE_95'], color=model_colors['modelE'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp2_params['modelEA_LE'],'.-', color=model_colors['modelEA'], label = 'Model EA')
ax[0,0].fill_between(pSA_T, grp2_params['modelEA_LE_5'],grp2_params['modelEA_LE_95'], color=model_colors['modelEA'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp2_params['modelEAS_LE'],'.-', color=model_colors['modelEAS'], label = 'Model EAS')
ax[0,0].fill_between(pSA_T, grp2_params['modelEAS_LE_5'],grp2_params['modelEAS_LE_95'], color=model_colors['modelEAS'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp2_params['modelEASZ_LE'],'.-',color=model_colors['modelEASZ'], label = 'Model EASZ')
ax[0,0].fill_between(pSA_T, grp2_params['modelEASZ_LE_5'],grp2_params['modelEASZ_LE_95'], color=model_colors['modelEASZ'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp2_params['modelEAST_LE'],'.-',color=model_colors['modelEAST'], label = 'Model EAST')
ax[0,0].fill_between(pSA_T, grp2_params['modelEAST_LE_5'],grp2_params['modelEAST_LE_95'], color=model_colors['modelEAST'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grp2_params['modelEASZT_LE'],'.-',color=model_colors['modelEASZT'], label = 'Model EASZT')
ax[0,0].fill_between(pSA_T, grp2_params['modelEASZT_LE_5'],grp2_params['modelEASZT_LE_95'], color=model_colors['modelEASZT'], alpha = 0.2)
ax[0,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LE'],'.-',color='black', label = 'BM23EAS')
ax[0,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LE_5'],BM23_parameters['modelEAS_LE_95'], color='black', alpha = 0.2)

ax[0,1].semilogx(pSA_T, grp2_params['modelE_gammaE'],'.-',color=model_colors['modelE'],label='Model E')
ax[0,1].fill_between(pSA_T, grp2_params['modelE_gammaE_5'],grp2_params['modelE_gammaE_95'], color=model_colors['modelE'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp2_params['modelEA_gammaE'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,1].fill_between(pSA_T, grp2_params['modelEA_gammaE_5'],grp2_params['modelEA_gammaE_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp2_params['modelEAS_gammaE'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,1].fill_between(pSA_T, grp2_params['modelEAS_gammaE_5'],grp2_params['modelEAS_gammaE_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp2_params['modelEASZ_gammaE'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,1].fill_between(pSA_T, grp2_params['modelEASZ_gammaE_5'],grp2_params['modelEASZ_gammaE_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp2_params['modelEAST_gammaE'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,1].fill_between(pSA_T, grp2_params['modelEAST_gammaE_5'],grp2_params['modelEAST_gammaE_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grp2_params['modelEASZT_gammaE'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,1].fill_between(pSA_T, grp2_params['modelEASZT_gammaE_5'],grp2_params['modelEASZT_gammaE_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE'],'.-',color='black', label = 'BM23EAS')
ax[0,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE_5'],BM23_parameters['modelEAS_gammaE_95'], color='black', alpha = 0.2)

ax[0,2].semilogx(pSA_T, grp2_params['modelEA_LA'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,2].fill_between(pSA_T, grp2_params['modelEA_LA_5'],grp2_params['modelEA_LA_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp2_params['modelEAS_LA'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,2].fill_between(pSA_T, grp2_params['modelEAS_LA_5'],grp2_params['modelEAS_LA_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp2_params['modelEASZ_LA'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,2].fill_between(pSA_T, grp2_params['modelEASZ_LA_5'],grp2_params['modelEASZ_LA_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp2_params['modelEAST_LA'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,2].fill_between(pSA_T, grp2_params['modelEAST_LA_5'],grp2_params['modelEAST_LA_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grp2_params['modelEASZT_LA'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,2].fill_between(pSA_T, grp2_params['modelEASZT_LA_5'],grp2_params['modelEASZT_LA_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,2].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_LA'],'.-',color='black', label = 'BM23EAS')
ax[0,2].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LA_5'],BM23_parameters['modelEAS_LA_95'], color='black', alpha = 0.2)

ax[1,0].loglog(pSA_T, grp2_params['modelEAS_LS'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,0].fill_between(pSA_T, grp2_params['modelEAS_LS_5'],grp2_params['modelEAS_LS_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,0].loglog(pSA_T, grp2_params['modelEASZ_LS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,0].fill_between(pSA_T, grp2_params['modelEASZ_LS_5'],grp2_params['modelEASZ_LS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,0].loglog(pSA_T, grp2_params['modelEAST_LS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,0].fill_between(pSA_T, grp2_params['modelEAST_LS_5'],grp2_params['modelEAST_LS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,0].loglog(pSA_T, grp2_params['modelEASZT_LS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,0].fill_between(pSA_T, grp2_params['modelEASZT_LS_5'],grp2_params['modelEASZT_LS_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LS'],'.-',color='black', label = 'BM23EAS')
ax[1,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LS_5'],BM23_parameters['modelEAS_LS_95'], color='black', alpha = 0.2)

ax[1,1].semilogx(pSA_T, grp2_params['modelEAS_w'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,1].fill_between(pSA_T, grp2_params['modelEAS_w_5'],grp2_params['modelEAS_w_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grp2_params['modelEASZ_w'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,1].fill_between(pSA_T, grp2_params['modelEASZ_w_5'],grp2_params['modelEASZ_w_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grp2_params['modelEAST_w'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,1].fill_between(pSA_T, grp2_params['modelEAST_w_5'],grp2_params['modelEAST_w_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grp2_params['modelEASZT_w'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,1].fill_between(pSA_T, grp2_params['modelEASZT_w_5'],grp2_params['modelEASZT_w_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_w'],'.-',color='black', label = 'BM23EAS')
ax[1,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_w_5'],BM23_parameters['modelEAS_w_95'], color='black', alpha = 0.2)

ax[1,2].loglog(pSA_T, grp2_params['modelEASZ_LZ'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,2].fill_between(pSA_T, grp2_params['modelEASZ_LZ_5'],grp2_params['modelEASZ_LZ_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,2].loglog(pSA_T, grp2_params['modelEASZT_LZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,2].fill_between(pSA_T, grp2_params['modelEASZT_LZ_5'],grp2_params['modelEASZT_LZ_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,0].semilogx(pSA_T, grp2_params['modelEASZ_wS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[2,0].fill_between(pSA_T, grp2_params['modelEASZ_wS_5'],grp2_params['modelEASZ_wS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grp2_params['modelEAST_wS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,0].fill_between(pSA_T, grp2_params['modelEAST_wS_5'],grp2_params['modelEAST_wS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grp2_params['modelEASZT_wS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,0].fill_between(pSA_T, grp2_params['modelEASZT_wS_5'],grp2_params['modelEASZT_wS_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,1].loglog(pSA_T, grp2_params['modelEAST_LT'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,1].fill_between(pSA_T, grp2_params['modelEAST_LT_5'],grp2_params['modelEAST_LT_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,1].loglog(pSA_T, grp2_params['modelEASZT_LT'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,1].fill_between(pSA_T, grp2_params['modelEASZT_LT_5'],grp2_params['modelEASZT_LT_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,2].semilogx(pSA_T, grp2_params['modelEASZT_wZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT: wZ')
ax[2,2].fill_between(pSA_T, grp2_params['modelEASZT_wZ_5'],grp2_params['modelEASZT_wZ_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[2,2].semilogx(pSA_T, grp2_params['modelEASZT_wT'],'--',color=model_colors['modelEASZT'],label='Model EASZT: wT')
ax[2,2].fill_between(pSA_T, grp2_params['modelEASZT_wT_5'],grp2_params['modelEASZT_wT_95'], color=model_colors['modelEASZT'], alpha=0.2)

for i in range(3):
    for j in range(3):
        ax[i,j].grid(which='both', linestyle = '--', alpha = 0.3)
        ax[i,j].set_xlim(0.01,6)
        ax[i,j].set_xlabel('Period, T(s)')
        if ax[i,j].get_yscale() == 'log':
            # Log axis → use LogFormatter, no ticklabel_format allowed
            ax[i,j].yaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))

ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A\ (^\circ)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$w$')
ax[1,2].set_ylabel(r'$L_Z$ (m)')
ax[2,0].set_ylabel(r'$w_S$')
ax[2,1].set_ylabel(r'$L_T$ (m)')
ax[2,2].set_ylabel(r'$w_{Z},\; w_{T}$')

ax[0,1].set_ylim(0,2)
ax[0,2].set_ylim(0,45)
ax[1,1].set_ylim(0,1)
ax[2,0].set_ylim(0,1)
ax[2,2].set_ylim(0,1)
ax[0,1].legend(loc="upper center", ncols = 2)
ax[2,2].legend(loc="upper right")
plt.show(fig)

### 6.5.10 North Island

In [ ]:
models = [
    'modelE', 'modelEA', 'modelEAS', 'modelEASZ',
    'modelEAST', 'modelEASZT'
]

cmap = plt.get_cmap('tab10')
model_colors = {m: cmap(i) for i, m in enumerate(models)}


fig, ax = plt.subplots(3,3, figsize = (12,8), dpi = 200, constrained_layout = True)
ax[0,0].loglog(pSA_T, grpn_params['modelE_LE'],'.-',color=model_colors['modelE'], label = 'Model E')
ax[0,0].fill_between(pSA_T, grpn_params['modelE_LE_5'],grpn_params['modelE_LE_95'], color=model_colors['modelE'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grpn_params['modelEA_LE'],'.-', color=model_colors['modelEA'], label = 'Model EA')
ax[0,0].fill_between(pSA_T, grpn_params['modelEA_LE_5'],grpn_params['modelEA_LE_95'], color=model_colors['modelEA'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grpn_params['modelEAS_LE'],'.-', color=model_colors['modelEAS'], label = 'Model EAS')
ax[0,0].fill_between(pSA_T, grpn_params['modelEAS_LE_5'],grpn_params['modelEAS_LE_95'], color=model_colors['modelEAS'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grpn_params['modelEASZ_LE'],'.-',color=model_colors['modelEASZ'], label = 'Model EASZ')
ax[0,0].fill_between(pSA_T, grpn_params['modelEASZ_LE_5'],grpn_params['modelEASZ_LE_95'], color=model_colors['modelEASZ'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grpn_params['modelEAST_LE'],'.-',color=model_colors['modelEAST'], label = 'Model EAST')
ax[0,0].fill_between(pSA_T, grpn_params['modelEAST_LE_5'],grpn_params['modelEAST_LE_95'], color=model_colors['modelEAST'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grpn_params['modelEASZT_LE'],'.-',color=model_colors['modelEASZT'], label = 'Model EASZT')
ax[0,0].fill_between(pSA_T, grpn_params['modelEASZT_LE_5'],grpn_params['modelEASZT_LE_95'], color=model_colors['modelEASZT'], alpha = 0.2)
ax[0,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LE'],'.-',color='black', label = 'BM23EAS')
ax[0,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LE_5'],BM23_parameters['modelEAS_LE_95'], color='black', alpha = 0.2)

ax[0,1].semilogx(pSA_T, grpn_params['modelE_gammaE'],'.-',color=model_colors['modelE'],label='Model E')
ax[0,1].fill_between(pSA_T, grpn_params['modelE_gammaE_5'],grpn_params['modelE_gammaE_95'], color=model_colors['modelE'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grpn_params['modelEA_gammaE'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,1].fill_between(pSA_T, grpn_params['modelEA_gammaE_5'],grpn_params['modelEA_gammaE_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grpn_params['modelEAS_gammaE'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,1].fill_between(pSA_T, grpn_params['modelEAS_gammaE_5'],grpn_params['modelEAS_gammaE_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grpn_params['modelEASZ_gammaE'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,1].fill_between(pSA_T, grpn_params['modelEASZ_gammaE_5'],grpn_params['modelEASZ_gammaE_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grpn_params['modelEAST_gammaE'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,1].fill_between(pSA_T, grpn_params['modelEAST_gammaE_5'],grpn_params['modelEAST_gammaE_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grpn_params['modelEASZT_gammaE'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,1].fill_between(pSA_T, grpn_params['modelEASZT_gammaE_5'],grpn_params['modelEASZT_gammaE_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE'],'.-',color='black', label = 'BM23EAS')
ax[0,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE_5'],BM23_parameters['modelEAS_gammaE_95'], color='black', alpha = 0.2)

ax[0,2].semilogx(pSA_T, grpn_params['modelEA_LA'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,2].fill_between(pSA_T, grpn_params['modelEA_LA_5'],grpn_params['modelEA_LA_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grpn_params['modelEAS_LA'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,2].fill_between(pSA_T, grpn_params['modelEAS_LA_5'],grpn_params['modelEAS_LA_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grpn_params['modelEASZ_LA'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,2].fill_between(pSA_T, grpn_params['modelEASZ_LA_5'],grpn_params['modelEASZ_LA_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grpn_params['modelEAST_LA'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,2].fill_between(pSA_T, grpn_params['modelEAST_LA_5'],grpn_params['modelEAST_LA_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grpn_params['modelEASZT_LA'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,2].fill_between(pSA_T, grpn_params['modelEASZT_LA_5'],grpn_params['modelEASZT_LA_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,2].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_LA'],'.-',color='black', label = 'BM23EAS')
ax[0,2].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LA_5'],BM23_parameters['modelEAS_LA_95'], color='black', alpha = 0.2)

ax[1,0].loglog(pSA_T, grpn_params['modelEAS_LS'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,0].fill_between(pSA_T, grpn_params['modelEAS_LS_5'],grpn_params['modelEAS_LS_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,0].loglog(pSA_T, grpn_params['modelEASZ_LS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,0].fill_between(pSA_T, grpn_params['modelEASZ_LS_5'],grpn_params['modelEASZ_LS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,0].loglog(pSA_T, grpn_params['modelEAST_LS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,0].fill_between(pSA_T, grpn_params['modelEAST_LS_5'],grpn_params['modelEAST_LS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,0].loglog(pSA_T, grpn_params['modelEASZT_LS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,0].fill_between(pSA_T, grpn_params['modelEASZT_LS_5'],grpn_params['modelEASZT_LS_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LS'],'.-',color='black', label = 'BM23EAS')
ax[1,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LS_5'],BM23_parameters['modelEAS_LS_95'], color='black', alpha = 0.2)

ax[1,1].semilogx(pSA_T, grpn_params['modelEAS_w'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,1].fill_between(pSA_T, grpn_params['modelEAS_w_5'],grpn_params['modelEAS_w_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grpn_params['modelEASZ_w'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,1].fill_between(pSA_T, grpn_params['modelEASZ_w_5'],grpn_params['modelEASZ_w_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grpn_params['modelEAST_w'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,1].fill_between(pSA_T, grpn_params['modelEAST_w_5'],grpn_params['modelEAST_w_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grpn_params['modelEASZT_w'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,1].fill_between(pSA_T, grpn_params['modelEASZT_w_5'],grpn_params['modelEASZT_w_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_w'],'.-',color='black', label = 'BM23EAS')
ax[1,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_w_5'],BM23_parameters['modelEAS_w_95'], color='black', alpha = 0.2)

ax[1,2].loglog(pSA_T, grpn_params['modelEASZ_LZ'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,2].fill_between(pSA_T, grpn_params['modelEASZ_LZ_5'],grpn_params['modelEASZ_LZ_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,2].loglog(pSA_T, grpn_params['modelEASZT_LZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,2].fill_between(pSA_T, grpn_params['modelEASZT_LZ_5'],grpn_params['modelEASZT_LZ_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,0].semilogx(pSA_T, grpn_params['modelEASZ_wS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[2,0].fill_between(pSA_T, grpn_params['modelEASZ_wS_5'],grpn_params['modelEASZ_wS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grpn_params['modelEAST_wS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,0].fill_between(pSA_T, grpn_params['modelEAST_wS_5'],grpn_params['modelEAST_wS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grpn_params['modelEASZT_wS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,0].fill_between(pSA_T, grpn_params['modelEASZT_wS_5'],grpn_params['modelEASZT_wS_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,1].loglog(pSA_T, grpn_params['modelEAST_LT'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,1].fill_between(pSA_T, grpn_params['modelEAST_LT_5'],grpn_params['modelEAST_LT_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,1].loglog(pSA_T, grpn_params['modelEASZT_LT'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,1].fill_between(pSA_T, grpn_params['modelEASZT_LT_5'],grpn_params['modelEASZT_LT_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,2].semilogx(pSA_T, grpn_params['modelEASZT_wZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT: wZ')
ax[2,2].fill_between(pSA_T, grpn_params['modelEASZT_wZ_5'],grpn_params['modelEASZT_wZ_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[2,2].semilogx(pSA_T, grpn_params['modelEASZT_wT'],'--',color=model_colors['modelEASZT'],label='Model EASZT: wT')
ax[2,2].fill_between(pSA_T, grpn_params['modelEASZT_wT_5'],grpn_params['modelEASZT_wT_95'], color=model_colors['modelEASZT'], alpha=0.2)

for i in range(3):
    for j in range(3):
        ax[i,j].grid(which='both', linestyle = '--', alpha = 0.3)
        ax[i,j].set_xlim(0.01,6)
        ax[i,j].set_xlabel('Period, T(s)')
        if ax[i,j].get_yscale() == 'log':
            # Log axis → use LogFormatter, no ticklabel_format allowed
            ax[i,j].yaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))

ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A\ (^\circ)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$w$')
ax[1,2].set_ylabel(r'$L_Z$ (m)')
ax[2,0].set_ylabel(r'$w_S$')
ax[2,1].set_ylabel(r'$L_T$ (m)')
ax[2,2].set_ylabel(r'$w_{Z},\; w_{T}$')

ax[0,1].set_ylim(0,2)
ax[0,2].set_ylim(0,45)
ax[1,1].set_ylim(0,1)
ax[2,0].set_ylim(0,1)
ax[2,2].set_ylim(0,1)
ax[0,1].legend(loc="upper center", ncols = 2)
ax[2,2].legend(loc="upper right")
plt.show(fig)

### 6.5.11 South Island

In [ ]:
models = [
    'modelE', 'modelEA', 'modelEAS', 'modelEASZ',
    'modelEAST', 'modelEASZT'
]

cmap = plt.get_cmap('tab10')
model_colors = {m: cmap(i) for i, m in enumerate(models)}


fig, ax = plt.subplots(3,3, figsize = (12,8), dpi = 200, constrained_layout = True)
ax[0,0].loglog(pSA_T, grps_params['modelE_LE'],'.-',color=model_colors['modelE'], label = 'Model E')
ax[0,0].fill_between(pSA_T, grps_params['modelE_LE_5'],grps_params['modelE_LE_95'], color=model_colors['modelE'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grps_params['modelEA_LE'],'.-', color=model_colors['modelEA'], label = 'Model EA')
ax[0,0].fill_between(pSA_T, grps_params['modelEA_LE_5'],grps_params['modelEA_LE_95'], color=model_colors['modelEA'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grps_params['modelEAS_LE'],'.-', color=model_colors['modelEAS'], label = 'Model EAS')
ax[0,0].fill_between(pSA_T, grps_params['modelEAS_LE_5'],grps_params['modelEAS_LE_95'], color=model_colors['modelEAS'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grps_params['modelEASZ_LE'],'.-',color=model_colors['modelEASZ'], label = 'Model EASZ')
ax[0,0].fill_between(pSA_T, grps_params['modelEASZ_LE_5'],grps_params['modelEASZ_LE_95'], color=model_colors['modelEASZ'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grps_params['modelEAST_LE'],'.-',color=model_colors['modelEAST'], label = 'Model EAST')
ax[0,0].fill_between(pSA_T, grps_params['modelEAST_LE_5'],grps_params['modelEAST_LE_95'], color=model_colors['modelEAST'], alpha = 0.2)
ax[0,0].loglog(pSA_T, grps_params['modelEASZT_LE'],'.-',color=model_colors['modelEASZT'], label = 'Model EASZT')
ax[0,0].fill_between(pSA_T, grps_params['modelEASZT_LE_5'],grps_params['modelEASZT_LE_95'], color=model_colors['modelEASZT'], alpha = 0.2)
ax[0,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LE'],'.-',color='black', label = 'BM23EAS')
ax[0,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LE_5'],BM23_parameters['modelEAS_LE_95'], color='black', alpha = 0.2)

ax[0,1].semilogx(pSA_T, grps_params['modelE_gammaE'],'.-',color=model_colors['modelE'],label='Model E')
ax[0,1].fill_between(pSA_T, grps_params['modelE_gammaE_5'],grps_params['modelE_gammaE_95'], color=model_colors['modelE'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grps_params['modelEA_gammaE'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,1].fill_between(pSA_T, grps_params['modelEA_gammaE_5'],grps_params['modelEA_gammaE_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grps_params['modelEAS_gammaE'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,1].fill_between(pSA_T, grps_params['modelEAS_gammaE_5'],grps_params['modelEAS_gammaE_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grps_params['modelEASZ_gammaE'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,1].fill_between(pSA_T, grps_params['modelEASZ_gammaE_5'],grps_params['modelEASZ_gammaE_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grps_params['modelEAST_gammaE'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,1].fill_between(pSA_T, grps_params['modelEAST_gammaE_5'],grps_params['modelEAST_gammaE_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,1].semilogx(pSA_T, grps_params['modelEASZT_gammaE'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,1].fill_between(pSA_T, grps_params['modelEASZT_gammaE_5'],grps_params['modelEASZT_gammaE_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE'],'.-',color='black', label = 'BM23EAS')
ax[0,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_gammaE_5'],BM23_parameters['modelEAS_gammaE_95'], color='black', alpha = 0.2)

ax[0,2].semilogx(pSA_T, grps_params['modelEA_LA'],'.-',color=model_colors['modelEA'],label='Model EA')
ax[0,2].fill_between(pSA_T, grps_params['modelEA_LA_5'],grps_params['modelEA_LA_95'], color=model_colors['modelEA'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grps_params['modelEAS_LA'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[0,2].fill_between(pSA_T, grps_params['modelEAS_LA_5'],grps_params['modelEAS_LA_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grps_params['modelEASZ_LA'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[0,2].fill_between(pSA_T, grps_params['modelEASZ_LA_5'],grps_params['modelEASZ_LA_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grps_params['modelEAST_LA'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[0,2].fill_between(pSA_T, grps_params['modelEAST_LA_5'],grps_params['modelEAST_LA_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[0,2].semilogx(pSA_T, grps_params['modelEASZT_LA'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[0,2].fill_between(pSA_T, grps_params['modelEASZT_LA_5'],grps_params['modelEASZT_LA_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[0,2].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_LA'],'.-',color='black', label = 'BM23EAS')
ax[0,2].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LA_5'],BM23_parameters['modelEAS_LA_95'], color='black', alpha = 0.2)

ax[1,0].loglog(pSA_T, grps_params['modelEAS_LS'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,0].fill_between(pSA_T, grps_params['modelEAS_LS_5'],grps_params['modelEAS_LS_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,0].loglog(pSA_T, grps_params['modelEASZ_LS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,0].fill_between(pSA_T, grps_params['modelEASZ_LS_5'],grps_params['modelEASZ_LS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,0].loglog(pSA_T, grps_params['modelEAST_LS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,0].fill_between(pSA_T, grps_params['modelEAST_LS_5'],grps_params['modelEAST_LS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,0].loglog(pSA_T, grps_params['modelEASZT_LS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,0].fill_between(pSA_T, grps_params['modelEASZT_LS_5'],grps_params['modelEASZT_LS_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,0].loglog(BM23_parameters['period'], BM23_parameters['modelEAS_LS'],'.-',color='black', label = 'BM23EAS')
ax[1,0].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_LS_5'],BM23_parameters['modelEAS_LS_95'], color='black', alpha = 0.2)

ax[1,1].semilogx(pSA_T, grps_params['modelEAS_w'],'.-',color=model_colors['modelEAS'],label='Model EAS')
ax[1,1].fill_between(pSA_T, grps_params['modelEAS_w_5'],grps_params['modelEAS_w_95'], color=model_colors['modelEAS'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grps_params['modelEASZ_w'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,1].fill_between(pSA_T, grps_params['modelEASZ_w_5'],grps_params['modelEASZ_w_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grps_params['modelEAST_w'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[1,1].fill_between(pSA_T, grps_params['modelEAST_w_5'],grps_params['modelEAST_w_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[1,1].semilogx(pSA_T, grps_params['modelEASZT_w'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,1].fill_between(pSA_T, grps_params['modelEASZT_w_5'],grps_params['modelEASZT_w_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[1,1].semilogx(BM23_parameters['period'], BM23_parameters['modelEAS_w'],'.-',color='black', label = 'BM23EAS')
ax[1,1].fill_between(BM23_parameters['period'], BM23_parameters['modelEAS_w_5'],BM23_parameters['modelEAS_w_95'], color='black', alpha = 0.2)

ax[1,2].loglog(pSA_T, grps_params['modelEASZ_LZ'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[1,2].fill_between(pSA_T, grps_params['modelEASZ_LZ_5'],grps_params['modelEASZ_LZ_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[1,2].loglog(pSA_T, grps_params['modelEASZT_LZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[1,2].fill_between(pSA_T, grps_params['modelEASZT_LZ_5'],grps_params['modelEASZT_LZ_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,0].semilogx(pSA_T, grps_params['modelEASZ_wS'],'.-',color=model_colors['modelEASZ'],label='Model EASZ')
ax[2,0].fill_between(pSA_T, grps_params['modelEASZ_wS_5'],grps_params['modelEASZ_wS_95'], color=model_colors['modelEASZ'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grps_params['modelEAST_wS'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,0].fill_between(pSA_T, grps_params['modelEAST_wS_5'],grps_params['modelEAST_wS_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,0].semilogx(pSA_T, grps_params['modelEASZT_wS'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,0].fill_between(pSA_T, grps_params['modelEASZT_wS_5'],grps_params['modelEASZT_wS_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,1].loglog(pSA_T, grps_params['modelEAST_LT'],'.-',color=model_colors['modelEAST'],label='Model EAST')
ax[2,1].fill_between(pSA_T, grps_params['modelEAST_LT_5'],grps_params['modelEAST_LT_95'], color=model_colors['modelEAST'], alpha=0.2)
ax[2,1].loglog(pSA_T, grps_params['modelEASZT_LT'],'.-',color=model_colors['modelEASZT'],label='Model EASZT')
ax[2,1].fill_between(pSA_T, grps_params['modelEASZT_LT_5'],grps_params['modelEASZT_LT_95'], color=model_colors['modelEASZT'], alpha=0.2)

ax[2,2].semilogx(pSA_T, grps_params['modelEASZT_wZ'],'.-',color=model_colors['modelEASZT'],label='Model EASZT: wZ')
ax[2,2].fill_between(pSA_T, grps_params['modelEASZT_wZ_5'],grps_params['modelEASZT_wZ_95'], color=model_colors['modelEASZT'], alpha=0.2)
ax[2,2].semilogx(pSA_T, grps_params['modelEASZT_wT'],'--',color=model_colors['modelEASZT'],label='Model EASZT: wT')
ax[2,2].fill_between(pSA_T, grps_params['modelEASZT_wT_5'],grps_params['modelEASZT_wT_95'], color=model_colors['modelEASZT'], alpha=0.2)

for i in range(3):
    for j in range(3):
        ax[i,j].grid(which='both', linestyle = '--', alpha = 0.3)
        ax[i,j].set_xlim(0.01,6)
        ax[i,j].set_xlabel('Period, T(s)')
        if ax[i,j].get_yscale() == 'log':
            # Log axis → use LogFormatter, no ticklabel_format allowed
            ax[i,j].yaxis.set_major_formatter(LogFormatter(labelOnlyBase=False))

ax[0,0].set_ylabel(r'$L_E$ (km)')
ax[0,1].set_ylabel(r'$\gamma_E$')
ax[0,2].set_ylabel(r'$L_A\ (^\circ)$')
ax[1,0].set_ylabel(r'$L_S$ (m/s)')
ax[1,1].set_ylabel(r'$w$')
ax[1,2].set_ylabel(r'$L_Z$ (m)')
ax[2,0].set_ylabel(r'$w_S$')
ax[2,1].set_ylabel(r'$L_T$ (m)')
ax[2,2].set_ylabel(r'$w_{Z},\; w_{T}$')

ax[0,1].set_ylim(0,2)
ax[0,2].set_ylim(0,45)
ax[1,1].set_ylim(0,1)
ax[2,0].set_ylim(0,1)
ax[2,2].set_ylim(0,1)
ax[0,1].legend(loc="upper center", ncols = 2)
ax[2,2].legend(loc="upper right")
plt.show(fig)

## 6.6 Correlation deviation mean and standard deviation computation

### 6.6.1 NZ fitted model vs. Regional data

In [ ]:
# compared to NZ dataset
e_NZ = BM23EXT_CBB21(cbb21_NZ, pSA_T, nz_params)
e_NZ_crus = BM23EXT_CBB21(cbb21_NZ, pSA_T, nz_params_crus)
e_NZ_sub = BM23EXT_CBB21(cbb21_NZ, pSA_T, nz_params_sub)

# compared to crustal dataset
e_NZ_allcrus = BM23EXT_CBB21(cbb21_crus, pSA_T, nz_params)
e_NZ_cruscrus = BM23EXT_CBB21(cbb21_crus, pSA_T, nz_params_crus)
e_NZ_subcrus = BM23EXT_CBB21(cbb21_crus, pSA_T, nz_params_sub)

# compared to subduction dataset
e_NZ_allsub = BM23EXT_CBB21(cbb21_sub, pSA_T, nz_params)
e_NZ_crussub = BM23EXT_CBB21(cbb21_sub, pSA_T, nz_params_crus)
e_NZ_subsub = BM23EXT_CBB21(cbb21_sub, pSA_T, nz_params_sub)

# regional groups
e_NZ_grp1 = BM23EXT_CBB21(cbb21_grp1, pSA_T, nz_params)
e_NZ_grp2 = BM23EXT_CBB21(cbb21_grp2, pSA_T, nz_params)
e_NZ_grpn = BM23EXT_CBB21(cbb21_grpn, pSA_T, nz_params)
e_NZ_grps = BM23EXT_CBB21(cbb21_grps, pSA_T, nz_params)

### 6.6.1 Regional fitted models

In [ ]:
e_grp1 = BM23EXT_CBB21(cbb21_grp1, pSA_T, grp1_params)
e_grp2 = BM23EXT_CBB21(cbb21_grp2, pSA_T, grp2_params)
e_grpn = BM23EXT_CBB21(cbb21_grpn, pSA_T, grpn_params)
e_grps = BM23EXT_CBB21(cbb21_grps, pSA_T, grps_params)

## 6.7 Correlation deviation plots

### 6.7.1 Tectonic class SCMs

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 150, constrained_layout = True)
# all tectonic classes
ax[0,0].plot(pSA_T, e_NZ['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EA']['mean5'], e_NZ['EA']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EAS']['mean5'], e_NZ['EAS']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EASZ']['mean5'], e_NZ['EASZ']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EAST']['mean5'], e_NZ['EAST']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EASZT']['mean5'], e_NZ['EASZT']['mean95'], alpha = 0.2)

ax[1,0].plot(pSA_T, e_NZ['E']['std'], '-o', label = 'Model E', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['E']['std5'], e_NZ['E']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EA']['std5'], e_NZ['EA']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EAS']['std5'], e_NZ['EAS']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EASZ']['std5'], e_NZ['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EAST']['std5'], e_NZ['EAST']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EASZT']['std5'], e_NZ['EASZT']['std95'], alpha = 0.2)

# crustal
ax[0,1].plot(pSA_T, e_NZ_crus['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['E']['mean5'], e_NZ_crus['E']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EA']['mean5'], e_NZ_crus['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EAS']['mean5'], e_NZ_crus['EAS']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EASZ']['mean5'], e_NZ_crus['EASZ']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EAST']['mean5'], e_NZ_crus['EAST']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EASZT']['mean5'], e_NZ_crus['EASZT']['mean95'], alpha = 0.2)

ax[1,1].plot(pSA_T, e_NZ_crus['E']['std'], '-o', label = 'Model E', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_crus['E']['std5'], e_NZ_crus['E']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_crus['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_crus['EA']['std5'], e_NZ_crus['EA']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_crus['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_crus['EAS']['std5'], e_NZ_crus['EAS']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_crus['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_crus['EASZ']['std5'], e_NZ_crus['EASZ']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_crus['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_crus['EAST']['std5'], e_NZ_crus['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_crus['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_crus['EASZT']['std5'], e_NZ_crus['EASZT']['std95'], alpha = 0.2)

# subduction
ax[0,2].plot(pSA_T, e_NZ_sub['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['E']['mean5'], e_NZ_sub['E']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EA']['mean5'], e_NZ_sub['EA']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EAS']['mean5'], e_NZ_sub['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EASZ']['mean5'], e_NZ_sub['EASZ']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EAST']['mean5'], e_NZ_sub['EAST']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EASZT']['mean5'], e_NZ_sub['EASZT']['mean95'], alpha = 0.2)

ax[1,2].plot(pSA_T, e_NZ_sub['E']['std'], '-o', label = 'Model E', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_sub['E']['std5'], e_NZ_sub['E']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_sub['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_sub['EA']['std5'], e_NZ_sub['EA']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_sub['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_sub['EAS']['std5'], e_NZ_sub['EAS']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_sub['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_sub['EASZ']['std5'], e_NZ_sub['EASZ']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_sub['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_sub['EAST']['std5'], e_NZ_sub['EAST']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_sub['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_sub['EASZT']['std5'], e_NZ_sub['EASZT']['std95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[0,j].set_ylim([-5,5])
        ax[1,j].set_ylim([1,6])
        ax[i,j].set_xlim([0.01,6])
        ax[0,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[i,j].set_xlabel('Period, T (s)')

ax[0,0].set_ylabel(r'$\mu_e$')
ax[1,0].set_ylabel(r'$\sigma_e$')
ax[0,0].set_title('(A)')
ax[0,1].set_title('(B)')
ax[0,2].set_title('(C)')
ax[0,1].legend(loc = 'upper center', ncol=2, fontsize = 10)
plt.show()

In [ ]:
fig, ax = plt.subplots(3,3,figsize = (12,8), dpi = 150, constrained_layout = True)
## NZ-dataset
# all tectonic classes
ax[0,0].plot(pSA_T, e_NZ['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EA']['mean5'], e_NZ['EA']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EAS']['mean5'], e_NZ['EAS']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EASZ']['mean5'], e_NZ['EASZ']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EAST']['mean5'], e_NZ['EAST']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EASZT']['mean5'], e_NZ['EASZT']['mean95'], alpha = 0.2)
# crustal
ax[0,1].plot(pSA_T, e_NZ_crus['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['E']['mean5'], e_NZ_crus['E']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EA']['mean5'], e_NZ_crus['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EAS']['mean5'], e_NZ_crus['EAS']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EASZ']['mean5'], e_NZ_crus['EASZ']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EAST']['mean5'], e_NZ_crus['EAST']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EASZT']['mean5'], e_NZ_crus['EASZT']['mean95'], alpha = 0.2)
# subduction
ax[0,2].plot(pSA_T, e_NZ_sub['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['E']['mean5'], e_NZ_sub['E']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EA']['mean5'], e_NZ_sub['EA']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EAS']['mean5'], e_NZ_sub['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EASZ']['mean5'], e_NZ_sub['EASZ']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EAST']['mean5'], e_NZ_sub['EAST']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EASZT']['mean5'], e_NZ_sub['EASZT']['mean95'], alpha = 0.2)

## crustal dataset
# all tectonic classes
ax[1,0].plot(pSA_T, e_NZ_allcrus['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['E']['mean5'], e_NZ_allcrus['E']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EA']['mean5'], e_NZ_allcrus['EA']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EAS']['mean5'], e_NZ_allcrus['EAS']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EASZ']['mean5'], e_NZ_allcrus['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EAST']['mean5'], e_NZ_allcrus['EAST']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EASZT']['mean5'], e_NZ_allcrus['EASZT']['mean95'], alpha = 0.2)
# crustal
ax[1,1].plot(pSA_T, e_NZ_cruscrus['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['E']['mean5'], e_NZ_cruscrus['E']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EA']['mean5'], e_NZ_cruscrus['EA']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EAS']['mean5'], e_NZ_cruscrus['EAS']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EASZ']['mean5'], e_NZ_cruscrus['EASZ']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EAST']['mean5'], e_NZ_cruscrus['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EASZT']['mean5'], e_NZ_cruscrus['EASZT']['mean95'], alpha = 0.2)
# subduction
ax[1,2].plot(pSA_T, e_NZ_subcrus['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['E']['mean5'], e_NZ_subcrus['E']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EA']['mean5'], e_NZ_subcrus['EA']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EAS']['mean5'], e_NZ_subcrus['EAS']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EASZ']['mean5'], e_NZ_subcrus['EASZ']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EAST']['mean5'], e_NZ_subcrus['EAST']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EASZT']['mean5'], e_NZ_subcrus['EASZT']['mean95'], alpha = 0.2)

## subduction dataset
# all tectonic classes
ax[2,0].plot(pSA_T, e_NZ_allsub['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['E']['mean5'], e_NZ_allsub['E']['mean95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EA']['mean5'], e_NZ_allsub['EA']['mean95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EAS']['mean5'], e_NZ_allsub['EAS']['mean95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EASZ']['mean5'], e_NZ_allsub['EASZ']['mean95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EAST']['mean5'], e_NZ_allsub['EAST']['mean95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EASZT']['mean5'], e_NZ_allsub['EASZT']['mean95'], alpha = 0.2)
# crustal
ax[2,1].plot(pSA_T, e_NZ_crussub['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['E']['mean5'], e_NZ_crussub['E']['mean95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EA']['mean5'], e_NZ_crussub['EA']['mean95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EAS']['mean5'], e_NZ_crussub['EAS']['mean95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EASZ']['mean5'], e_NZ_crussub['EASZ']['mean95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EAST']['mean5'], e_NZ_crussub['EAST']['mean95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EASZT']['mean5'], e_NZ_crussub['EASZT']['mean95'], alpha = 0.2)
# subduction
ax[2,2].plot(pSA_T, e_NZ_subsub['E']['mean'], '-o', label = 'Model E', lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['E']['mean5'], e_NZ_subsub['E']['mean95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EA']['mean'], '-o', label = 'Model EA',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EA']['mean5'], e_NZ_subsub['EA']['mean95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EAS']['mean'], '-o', label = 'Model EAS', lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EAS']['mean5'], e_NZ_subsub['EAS']['mean95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EASZ']['mean'], '-o', label = 'Model EASZ',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EASZ']['mean5'], e_NZ_subsub['EASZ']['mean95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EAST']['mean'], '-o', label = 'Model EAST',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EAST']['mean5'], e_NZ_subsub['EAST']['mean95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EASZT']['mean'], '-o', label = 'Model EASZT',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EASZT']['mean5'], e_NZ_subsub['EASZT']['mean95'], alpha = 0.2)

for i in range(3):
    for j in range(3):
        ax[i,j].set_ylim([-5,5])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[i,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\mu_e$')

ax[0,0].set_title('(A)')
ax[0,1].set_title('(B)')
ax[0,2].set_title('(C)')
ax[0,1].legend(loc = 'upper center', ncol=2, fontsize = 10)
plt.show()

In [ ]:
fig, ax = plt.subplots(3,3,figsize = (12,8), dpi = 150, constrained_layout = True)
## NZ-dataset
# all tectonic classes
ax[0,0].plot(pSA_T, e_NZ['E']['std'], '-o', label = 'Model E', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['E']['std5'], e_NZ['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EA']['std5'], e_NZ['EA']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EAS']['std5'], e_NZ['EAS']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EASZ']['std5'], e_NZ['EASZ']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EAST']['std5'], e_NZ['EAST']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['EASZT']['std5'], e_NZ['EASZT']['std95'], alpha = 0.2)
# crustal
ax[0,1].plot(pSA_T, e_NZ_crus['E']['std'], '-o', label = 'Model E', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['E']['std5'], e_NZ_crus['E']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EA']['std5'], e_NZ_crus['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EAS']['std5'], e_NZ_crus['EAS']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EASZ']['std5'], e_NZ_crus['EASZ']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EAST']['std5'], e_NZ_crus['EAST']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_crus['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_crus['EASZT']['std5'], e_NZ_crus['EASZT']['std95'], alpha = 0.2)
# subduction
ax[0,2].plot(pSA_T, e_NZ_sub['E']['std'], '-o', label = 'Model E', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['E']['std5'], e_NZ_sub['E']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EA']['std5'], e_NZ_sub['EA']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EAS']['std5'], e_NZ_sub['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EASZ']['std5'], e_NZ_sub['EASZ']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EAST']['std5'], e_NZ_sub['EAST']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_sub['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_sub['EASZT']['std5'], e_NZ_sub['EASZT']['std95'], alpha = 0.2)

## crustal dataset
# all tectonic classes
ax[1,0].plot(pSA_T, e_NZ_allcrus['E']['std'], '-o', label = 'Model E', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['E']['std5'], e_NZ_allcrus['E']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EA']['std5'], e_NZ_allcrus['EA']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EAS']['std5'], e_NZ_allcrus['EAS']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EASZ']['std5'], e_NZ_allcrus['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EAST']['std5'], e_NZ_allcrus['EAST']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_allcrus['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_allcrus['EASZT']['std5'], e_NZ_allcrus['EASZT']['std95'], alpha = 0.2)
# crustal
ax[1,1].plot(pSA_T, e_NZ_cruscrus['E']['std'], '-o', label = 'Model E', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['E']['std5'], e_NZ_cruscrus['E']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EA']['std5'], e_NZ_cruscrus['EA']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EAS']['std5'], e_NZ_cruscrus['EAS']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EASZ']['std5'], e_NZ_cruscrus['EASZ']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EAST']['std5'], e_NZ_cruscrus['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_cruscrus['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_cruscrus['EASZT']['std5'], e_NZ_cruscrus['EASZT']['std95'], alpha = 0.2)
# subduction
ax[1,2].plot(pSA_T, e_NZ_subcrus['E']['std'], '-o', label = 'Model E', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['E']['std5'], e_NZ_subcrus['E']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EA']['std5'], e_NZ_subcrus['EA']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EAS']['std5'], e_NZ_subcrus['EAS']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EASZ']['std5'], e_NZ_subcrus['EASZ']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EAST']['std5'], e_NZ_subcrus['EAST']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_subcrus['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_subcrus['EASZT']['std5'], e_NZ_subcrus['EASZT']['std95'], alpha = 0.2)

## subduction dataset
# all tectonic classes
ax[2,0].plot(pSA_T, e_NZ_allsub['E']['std'], '-o', label = 'Model E', lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['E']['std5'], e_NZ_allsub['E']['std95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EA']['std5'], e_NZ_allsub['EA']['std95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EAS']['std5'], e_NZ_allsub['EAS']['std95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EASZ']['std5'], e_NZ_allsub['EASZ']['std95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EAST']['std5'], e_NZ_allsub['EAST']['std95'], alpha = 0.2)
ax[2,0].plot(pSA_T, e_NZ_allsub['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[2,0].fill_between(pSA_T, e_NZ_allsub['EASZT']['std5'], e_NZ_allsub['EASZT']['std95'], alpha = 0.2)
# crustal
ax[2,1].plot(pSA_T, e_NZ_crussub['E']['std'], '-o', label = 'Model E', lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['E']['std5'], e_NZ_crussub['E']['std95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EA']['std5'], e_NZ_crussub['EA']['std95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EAS']['std5'], e_NZ_crussub['EAS']['std95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EASZ']['std5'], e_NZ_crussub['EASZ']['std95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EAST']['std5'], e_NZ_crussub['EAST']['std95'], alpha = 0.2)
ax[2,1].plot(pSA_T, e_NZ_crussub['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[2,1].fill_between(pSA_T, e_NZ_crussub['EASZT']['std5'], e_NZ_crussub['EASZT']['std95'], alpha = 0.2)
# subduction
ax[2,2].plot(pSA_T, e_NZ_subsub['E']['std'], '-o', label = 'Model E', lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['E']['std5'], e_NZ_subsub['E']['std95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EA']['std'], '-o', label = 'Model EA',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EA']['std5'], e_NZ_subsub['EA']['std95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EAS']['std'], '-o', label = 'Model EAS', lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EAS']['std5'], e_NZ_subsub['EAS']['std95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EASZ']['std'], '-o', label = 'Model EASZ',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EASZ']['std5'], e_NZ_subsub['EASZ']['std95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EAST']['std'], '-o', label = 'Model EAST',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EAST']['std5'], e_NZ_subsub['EAST']['std95'], alpha = 0.2)
ax[2,2].plot(pSA_T, e_NZ_subsub['EASZT']['std'], '-o', label = 'Model EASZT',  lw = 1)
ax[2,2].fill_between(pSA_T, e_NZ_subsub['EASZT']['std5'], e_NZ_subsub['EASZT']['std95'], alpha = 0.2)

for i in range(3):
    for j in range(3):
        ax[i,j].set_ylim([1,6])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[i,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\sigma_e$')

ax[0,0].set_title('(A)')
ax[0,1].set_title('(B)')
ax[0,2].set_title('(C)')
ax[0,1].legend(loc = 'lower center', ncol=2, fontsize = 10)
plt.show()

### 6.7.2 NZ-fitted vs. regional data

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_NZ['E']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grp1['E']['mean'], '-o', label = 'Wellington', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grp1['E']['mean5'], e_NZ_grp1['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grp2['E']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grp2['E']['mean5'], e_NZ_grp2['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grpn['E']['mean'], '-o', label = 'North Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grpn['E']['mean5'], e_NZ_grpn['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grps['E']['mean'], '-o', label = 'South Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grps['E']['mean5'], e_NZ_grps['E']['mean95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_NZ['EA']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ['EA']['mean5'], e_NZ['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grp1['EA']['mean'], '-o', label = 'Wellington', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grp1['EA']['mean5'], e_NZ_grp1['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grp2['EA']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grp2['EA']['mean5'], e_NZ_grp2['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grpn['EA']['mean'], '-o', label = 'North Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grpn['EA']['mean5'], e_NZ_grpn['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grps['EA']['mean'], '-o', label = 'South Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grps['EA']['mean5'], e_NZ_grps['EA']['mean95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_NZ['EAS']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ['EAS']['mean5'], e_NZ['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grp1['EAS']['mean'], '-o', label = 'Wellington', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grp1['EAS']['mean5'], e_NZ_grp1['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grp2['EAS']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grp2['EAS']['mean5'], e_NZ_grp2['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grpn['EAS']['mean'], '-o', label = 'North Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grpn['EAS']['mean5'], e_NZ_grpn['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grps['EAS']['mean'], '-o', label = 'South Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grps['EAS']['mean5'], e_NZ_grps['EAS']['mean95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_NZ['EASZ']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EASZ']['mean5'], e_NZ['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grp1['EASZ']['mean'], '-o', label = 'Wellington', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grp1['EASZ']['mean5'], e_NZ_grp1['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grp2['EASZ']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grp2['EASZ']['mean5'], e_NZ_grp2['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grpn['EASZ']['mean'], '-o', label = 'North Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grpn['EASZ']['mean5'], e_NZ_grpn['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grps['EASZ']['mean'], '-o', label = 'South Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grps['EASZ']['mean5'], e_NZ_grps['EASZ']['mean95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_NZ['EAST']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ['EAST']['mean5'], e_NZ['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grp1['EAST']['mean'], '-o', label = 'Wellington', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grp1['EAST']['mean5'], e_NZ_grp1['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grp2['EAST']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grp2['EAST']['mean5'], e_NZ_grp2['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grpn['EAST']['mean'], '-o', label = 'North Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grpn['EAST']['mean5'], e_NZ_grpn['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grps['EAST']['mean'], '-o', label = 'South Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grps['EAST']['mean5'], e_NZ_grps['EAST']['mean95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_NZ['EASZT']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ['EASZT']['mean5'], e_NZ['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grp1['EASZT']['mean'], '-o', label = 'Wellington', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grp1['EASZT']['mean5'], e_NZ_grp1['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grp2['EASZT']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grp2['EASZT']['mean5'], e_NZ_grp2['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grpn['EASZT']['mean'], '-o', label = 'North Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grpn['EASZT']['mean5'], e_NZ_grpn['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grps['EASZT']['mean'], '-o', label = 'South Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grps['EASZT']['mean5'], e_NZ_grps['EASZT']['mean95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([-5,5])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\mu_e$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=2, fontsize = 10)
plt.show(fig)

# plot: correlation deviation standard deviation
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_NZ['E']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['E']['std5'], e_NZ['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grp1['E']['std'], '-o', label = 'Wellington', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grp1['E']['std5'], e_NZ_grp1['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grp2['E']['std'], '-o', label = 'Canterbury', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grp2['E']['std5'], e_NZ_grp2['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grpn['E']['std'], '-o', label = 'North Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grpn['E']['std5'], e_NZ_grpn['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_grps['E']['std'], '-o', label = 'South Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ_grps['E']['std5'], e_NZ_grps['E']['std95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_NZ['EA']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ['EA']['std5'], e_NZ['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grp1['EA']['std'], '-o', label = 'Wellington', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grp1['EA']['std5'], e_NZ_grp1['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grp2['EA']['std'], '-o', label = 'Canterbury', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grp2['EA']['std5'], e_NZ_grp2['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grpn['EA']['std'], '-o', label = 'North Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grpn['EA']['std5'], e_NZ_grpn['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_NZ_grps['EA']['std'], '-o', label = 'South Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ_grps['EA']['std5'], e_NZ_grps['EA']['std95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_NZ['EAS']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ['EAS']['std5'], e_NZ['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grp1['EAS']['std'], '-o', label = 'Wellington', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grp1['EAS']['std5'], e_NZ_grp1['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grp2['EAS']['std'], '-o', label = 'Canterbury', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grp2['EAS']['std5'], e_NZ_grp2['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grpn['EAS']['std'], '-o', label = 'North Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grpn['EAS']['std5'], e_NZ_grpn['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_NZ_grps['EAS']['std'], '-o', label = 'South Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ_grps['EAS']['std5'], e_NZ_grps['EAS']['std95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_NZ['EASZ']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EASZ']['std5'], e_NZ['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grp1['EASZ']['std'], '-o', label = 'Wellington', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grp1['EASZ']['std5'], e_NZ_grp1['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grp2['EASZ']['std'], '-o', label = 'Canterbury', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grp2['EASZ']['std5'], e_NZ_grp2['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grpn['EASZ']['std'], '-o', label = 'North Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grpn['EASZ']['std5'], e_NZ_grpn['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_NZ_grps['EASZ']['std'], '-o', label = 'South Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ_grps['EASZ']['std5'], e_NZ_grps['EASZ']['std95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_NZ['EAST']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ['EAST']['std5'], e_NZ['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grp1['EAST']['std'], '-o', label = 'Wellington', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grp1['EAST']['std5'], e_NZ_grp1['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grp2['EAST']['std'], '-o', label = 'Canterbury', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grp2['EAST']['std5'], e_NZ_grp2['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grpn['EAST']['std'], '-o', label = 'North Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grpn['EAST']['std5'], e_NZ_grpn['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_NZ_grps['EAST']['std'], '-o', label = 'South Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ_grps['EAST']['std5'], e_NZ_grps['EAST']['std95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_NZ['EASZT']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ['EASZT']['std5'], e_NZ['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grp1['EASZT']['std'], '-o', label = 'Wellington', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grp1['EASZT']['std5'], e_NZ_grp1['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grp2['EASZT']['std'], '-o', label = 'Canterbury', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grp2['EASZT']['std5'], e_NZ_grp2['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grpn['EASZT']['std'], '-o', label = 'North Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grpn['EASZT']['std5'], e_NZ_grpn['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_NZ_grps['EASZT']['std'], '-o', label = 'South Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ_grps['EASZT']['std5'], e_NZ_grps['EASZT']['std95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([1,7])
        ax[i,j].set_xlim([0.01,8])
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\sigma_{e}$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=2, fontsize = 10)
plt.show(fig)

### 6.7.3 Regional-fitted vs. regional data

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_NZ['E']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grp1['E']['mean'], '-o', label = 'Wellington', lw = 1)
ax[0,0].fill_between(pSA_T, e_grp1['E']['mean5'], e_grp1['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grp2['E']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[0,0].fill_between(pSA_T, e_grp2['E']['mean5'], e_grp2['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grpn['E']['mean'], '-o', label = 'North Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_grpn['E']['mean5'], e_grpn['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grps['E']['mean'], '-o', label = 'South Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_grps['E']['mean5'], e_grps['E']['mean95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_NZ['EA']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ['EA']['mean5'], e_NZ['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grp1['EA']['mean'], '-o', label = 'Wellington', lw = 1)
ax[0,1].fill_between(pSA_T, e_grp1['EA']['mean5'], e_grp1['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grp2['EA']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[0,1].fill_between(pSA_T, e_grp2['EA']['mean5'], e_grp2['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grpn['EA']['mean'], '-o', label = 'North Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_grpn['EA']['mean5'], e_grpn['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grps['EA']['mean'], '-o', label = 'South Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_grps['EA']['mean5'], e_grps['EA']['mean95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_NZ['EAS']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ['EAS']['mean5'], e_NZ['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grp1['EAS']['mean'], '-o', label = 'Wellington', lw = 1)
ax[0,2].fill_between(pSA_T, e_grp1['EAS']['mean5'], e_grp1['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grp2['EAS']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[0,2].fill_between(pSA_T, e_grp2['EAS']['mean5'], e_grp2['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grpn['EAS']['mean'], '-o', label = 'North Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_grpn['EAS']['mean5'], e_grpn['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grps['EAS']['mean'], '-o', label = 'South Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_grps['EAS']['mean5'], e_grps['EAS']['mean95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_NZ['EASZ']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EASZ']['mean5'], e_NZ['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grp1['EASZ']['mean'], '-o', label = 'Wellington', lw = 1)
ax[1,0].fill_between(pSA_T, e_grp1['EASZ']['mean5'], e_grp1['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grp2['EASZ']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[1,0].fill_between(pSA_T, e_grp2['EASZ']['mean5'], e_grp2['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grpn['EASZ']['mean'], '-o', label = 'North Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_grpn['EASZ']['mean5'], e_grpn['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grps['EASZ']['mean'], '-o', label = 'South Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_grps['EASZ']['mean5'], e_grps['EASZ']['mean95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_NZ['EAST']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ['EAST']['mean5'], e_NZ['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grp1['EAST']['mean'], '-o', label = 'Wellington', lw = 1)
ax[1,1].fill_between(pSA_T, e_grp1['EAST']['mean5'], e_grp1['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grp2['EAST']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[1,1].fill_between(pSA_T, e_grp2['EAST']['mean5'], e_grp2['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grpn['EAST']['mean'], '-o', label = 'North Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_grpn['EAST']['mean5'], e_grpn['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grps['EAST']['mean'], '-o', label = 'South Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_grps['EAST']['mean5'], e_grps['EAST']['mean95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_NZ['EASZT']['mean'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ['EASZT']['mean5'], e_NZ['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grp1['EASZT']['mean'], '-o', label = 'Wellington', lw = 1)
ax[1,2].fill_between(pSA_T, e_grp1['EASZT']['mean5'], e_grp1['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grp2['EASZT']['mean'], '-o', label = 'Canterbury', lw = 1)
ax[1,2].fill_between(pSA_T, e_grp2['EASZT']['mean5'], e_grp2['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grpn['EASZT']['mean'], '-o', label = 'North Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_grpn['EASZT']['mean5'], e_grpn['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grps['EASZT']['mean'], '-o', label = 'South Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_grps['EASZT']['mean5'], e_grps['EASZT']['mean95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([-5,5])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\mu_e$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=2, fontsize = 10)
plt.show()

# plot: correlation deviation standard deviation
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_NZ['E']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,0].fill_between(pSA_T, e_NZ['E']['std5'], e_NZ['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grp1['E']['std'], '-o', label = 'Wellington', lw = 1)
ax[0,0].fill_between(pSA_T, e_grp1['E']['std5'], e_grp1['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grp2['E']['std'], '-o', label = 'Canterbury', lw = 1)
ax[0,0].fill_between(pSA_T, e_grp2['E']['std5'], e_grp2['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grpn['E']['std'], '-o', label = 'North Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_grpn['E']['std5'], e_grpn['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_grps['E']['std'], '-o', label = 'South Island', lw = 1)
ax[0,0].fill_between(pSA_T, e_grps['E']['std5'], e_grps['E']['std95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_NZ['EA']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,1].fill_between(pSA_T, e_NZ['EA']['std5'], e_NZ['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grp1['EA']['std'], '-o', label = 'Wellington', lw = 1)
ax[0,1].fill_between(pSA_T, e_grp1['EA']['std5'], e_grp1['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grp2['EA']['std'], '-o', label = 'Canterbury', lw = 1)
ax[0,1].fill_between(pSA_T, e_grp2['EA']['std5'], e_grp2['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grpn['EA']['std'], '-o', label = 'North Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_grpn['EA']['std5'], e_grpn['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_grps['EA']['std'], '-o', label = 'South Island', lw = 1)
ax[0,1].fill_between(pSA_T, e_grps['EA']['std5'], e_grps['EA']['std95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_NZ['EAS']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[0,2].fill_between(pSA_T, e_NZ['EAS']['std5'], e_NZ['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grp1['EAS']['std'], '-o', label = 'Wellington', lw = 1)
ax[0,2].fill_between(pSA_T, e_grp1['EAS']['std5'], e_grp1['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grp2['EAS']['std'], '-o', label = 'Canterbury', lw = 1)
ax[0,2].fill_between(pSA_T, e_grp2['EAS']['std5'], e_grp2['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grpn['EAS']['std'], '-o', label = 'North Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_grpn['EAS']['std5'], e_grpn['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_grps['EAS']['std'], '-o', label = 'South Island', lw = 1)
ax[0,2].fill_between(pSA_T, e_grps['EAS']['std5'], e_grps['EAS']['std95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_NZ['EASZ']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,0].fill_between(pSA_T, e_NZ['EASZ']['std5'], e_NZ['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grp1['EASZ']['std'], '-o', label = 'Wellington', lw = 1)
ax[1,0].fill_between(pSA_T, e_grp1['EASZ']['std5'], e_grp1['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grp2['EASZ']['std'], '-o', label = 'Canterbury', lw = 1)
ax[1,0].fill_between(pSA_T, e_grp2['EASZ']['std5'], e_grp2['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grpn['EASZ']['std'], '-o', label = 'North Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_grpn['EASZ']['std5'], e_grpn['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_grps['EASZ']['std'], '-o', label = 'South Island', lw = 1)
ax[1,0].fill_between(pSA_T, e_grps['EASZ']['std5'], e_grps['EASZ']['std95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_NZ['EAST']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,1].fill_between(pSA_T, e_NZ['EAST']['std5'], e_NZ['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grp1['EAST']['std'], '-o', label = 'Wellington', lw = 1)
ax[1,1].fill_between(pSA_T, e_grp1['EAST']['std5'], e_grp1['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grp2['EAST']['std'], '-o', label = 'Canterbury', lw = 1)
ax[1,1].fill_between(pSA_T, e_grp2['EAST']['std5'], e_grp2['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grpn['EAST']['std'], '-o', label = 'North Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_grpn['EAST']['std5'], e_grpn['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_grps['EAST']['std'], '-o', label = 'South Island', lw = 1)
ax[1,1].fill_between(pSA_T, e_grps['EAST']['std5'], e_grps['EAST']['std95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_NZ['EASZT']['std'], '-o', label = 'NZ-Wide', lw = 1)
ax[1,2].fill_between(pSA_T, e_NZ['EASZT']['std5'], e_NZ['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grp1['EASZT']['std'], '-o', label = 'Wellington', lw = 1)
ax[1,2].fill_between(pSA_T, e_grp1['EASZT']['std5'], e_grp1['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grp2['EASZT']['std'], '-o', label = 'Canterbury', lw = 1)
ax[1,2].fill_between(pSA_T, e_grp2['EASZT']['std5'], e_grp2['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grpn['EASZT']['std'], '-o', label = 'North Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_grpn['EASZT']['std5'], e_grpn['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_grps['EASZT']['std'], '-o', label = 'South Island', lw = 1)
ax[1,2].fill_between(pSA_T, e_grps['EASZT']['std5'], e_grps['EASZT']['std95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([1,7])
        ax[i,j].set_xlim([0.01,8])
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\sigma_{e}$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=2, fontsize = 10)
plt.show()

## 6.8 NZ-wide, Christchurch and Wellington

### 6.8.1 Compute correlation cloud: Evaluation dataset

In [ ]:
n_rec = 6
hmax = 25

cbb21_pSAobs_T_NZ_eval = []; cbb21_pSAobs_T_chc_eval = []; cbb21_pSAobs_T_wel_eval = []

for i in tqdm(range(len(pSAobs_T)), desc="pSA: "):

    # NZ-wide
    im_dwbar_temp = im_dwbar_pSA_files_n3[i]
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_NZ_eval.append(cbb21)

    # group 1: wellington
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(stations_wel.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_wel_eval.append(cbb21)

    # group 2: christchurch
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(stations_chc.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_chc_eval.append(cbb21)

### 6.8.2 Compute correlation deviations

In [ ]:
e_NZ_all_dev = BM23EXT_CBB21(cbb21_pSAobs_T, pSA_T, nz_params)
e_NZ_wel_dev = BM23EXT_CBB21(cbb21_pSAobs_T_wel, pSA_T, nz_params)
e_NZ_chc_dev = BM23EXT_CBB21(cbb21_pSAobs_T_chc, pSA_T, nz_params)


# evaluation dataset
e_NZ_eval = BM23EXT_CBB21(cbb21_pSAobs_T_NZ_eval, pSA_T, nz_params)
e_NZ_wel_eval = BM23EXT_CBB21(cbb21_pSAobs_T_wel_eval, pSA_T, nz_params)
e_NZ_chc_eval = BM23EXT_CBB21(cbb21_pSAobs_T_chc_eval, pSA_T, nz_params)

e_wel_eval = BM23EXT_CBB21(cbb21_pSAobs_T_wel_eval, pSA_T, grp1_params)
e_chc_eval = BM23EXT_CBB21(cbb21_pSAobs_T_chc_eval, pSA_T, grp2_params)
# e_NZ = BM23EXT_CBB21(cbb21_NZ, pSA_T, nz_params) # previously computed

### 6.8.3 Plots

#### 6.8.3.1 Development dataset

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# correlation deviation mean
ax[0,0].plot(pSA_T, e_NZ_all_dev['E']['mean'], '-o', label = 'E', lw = 1)
# ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_all_dev['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_all_dev['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_all_dev['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_all_dev['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_all_dev['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,1].plot(pSA_T, e_NZ_chc_dev['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_dev['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_dev['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_dev['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_dev['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_dev['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,2].plot(pSA_T, e_NZ_wel_dev['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_dev['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_dev['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_dev['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_dev['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_dev['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

# correlation deviation standard deviation
ax[1,0].plot(pSA_T, e_NZ_all_dev['E']['std'], '-o', label = 'E', lw = 1)
ax[1,0].plot(pSA_T, e_NZ_all_dev['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,0].plot(pSA_T, e_NZ_all_dev['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,0].plot(pSA_T, e_NZ_all_dev['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,0].plot(pSA_T, e_NZ_all_dev['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,0].plot(pSA_T, e_NZ_all_dev['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

ax[1,1].plot(pSA_T, e_NZ_chc_dev['E']['std'], '-o', label = 'E', lw = 1)
ax[1,1].plot(pSA_T, e_NZ_chc_dev['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,1].plot(pSA_T, e_NZ_chc_dev['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,1].plot(pSA_T, e_NZ_chc_dev['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,1].plot(pSA_T, e_NZ_chc_dev['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,1].plot(pSA_T, e_NZ_chc_dev['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[1,2].plot(pSA_T, e_NZ_wel_dev['E']['std'], '-o', label = 'E', lw = 1)
ax[1,2].plot(pSA_T, e_NZ_wel_dev['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,2].plot(pSA_T, e_NZ_wel_dev['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,2].plot(pSA_T, e_NZ_wel_dev['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,2].plot(pSA_T, e_NZ_wel_dev['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,2].plot(pSA_T, e_NZ_wel_dev['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

for i in range(2):
    for j in range(3):
        ax[0,j].set_ylim([-3,3])
        ax[1,j].set_ylim([1,6])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[0,0].set_ylabel(r'$\mu_e$')
        ax[1,0].set_ylabel(r'$\sigma_e$')
ax[0,0].set_title('NZ-wide')
ax[0,1].set_title('Christchurch')
ax[0,2].set_title('Wellington')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show(fig)

#### 6.8.3.2 Evaluation dataset

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# correlation deviation mean
ax[0,0].plot(pSA_T, e_NZ['E']['mean'], '-o', label = 'E', lw = 1)
# ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,1].plot(pSA_T, e_chc_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,2].plot(pSA_T, e_wel_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

# correlation deviation standard deviation
ax[1,0].plot(pSA_T, e_NZ['E']['std'], '-o', label = 'E', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

ax[1,1].plot(pSA_T, e_chc_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[1,2].plot(pSA_T, e_wel_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

for i in range(2):
    for j in range(3):
        ax[0,j].set_ylim([-3,3])
        ax[1,j].set_ylim([1,6])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[0,0].set_ylabel(r'$\mu_e$')
        ax[1,0].set_ylabel(r'$\sigma_e$')
ax[0,0].set_title('NZ-wide')
ax[0,1].set_title('Christchurch')
ax[0,2].set_title('Wellington')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show(fig)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(10, 5.5), dpi=200, constrained_layout = True)
# correlation deviation mean
ax[0,0].plot(pSA_T, e_NZ['E']['mean'], '-o', label = 'E', lw = 1)
# ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,0].plot(pSA_T, e_NZ['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,1].plot(pSA_T, e_chc_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,1].plot(pSA_T, e_chc_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,2].plot(pSA_T, e_wel_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,2].plot(pSA_T, e_wel_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

# correlation deviation standard deviation
ax[1,0].plot(pSA_T, e_NZ['E']['std'], '-o', label = 'E', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,0].plot(pSA_T, e_NZ['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

ax[1,1].plot(pSA_T, e_chc_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[1,2].plot(pSA_T, e_wel_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

for i in range(2):
    for j in range(3):
        ax[0,j].set_ylim([-3,3])
        ax[1,j].set_ylim([1,6])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)', fontsize = 16)
        ax[0,0].set_ylabel(r'$\mu_e$', fontsize = 16)
        ax[1,0].set_ylabel(r'$\sigma_e$', fontsize = 16)
        ax[i,j].tick_params(axis='both', labelsize = 14)
ax[0,0].set_title('NZ-wide', fontsize = 14)
ax[0,1].set_title('Christchurch', fontsize = 14)
ax[0,2].set_title('Wellington', fontsize = 14)
ax[0,1].legend(loc = 'upper center', ncol=2, fontsize = 12)
plt.show(fig)

#### 6.8.3.3 Evaluation dataset: NZ-wide vs. Regional

In [ ]:
corr_dev_bounds_by_region = pd.read_csv(SCM_DIR / "corr_dev_bounds_by_region.csv")

# make sure period is numeric
corr_dev_bounds_by_region["T"] = pd.to_numeric(corr_dev_bounds_by_region["T"], errors="coerce")

corr_dev_bounds_nz = (
    corr_dev_bounds_by_region
    .query("region == 'NZ GMDB'")
    .sort_values("T")
    .dropna(subset=["T", "mue_min", "mue_max", "sige_min", "sige_max"])
    .reset_index(drop=True)
)

corr_dev_bounds_chc = (
    corr_dev_bounds_by_region
    .query("region == 'Christchurch'")
    .sort_values("T")
    .dropna(subset=["T", "mue_min", "mue_max", "sige_min", "sige_max"])
    .reset_index(drop=True)
)

corr_dev_bounds_wel = (
    corr_dev_bounds_by_region
    .query("region == 'Wellington'")
    .sort_values("T")
    .dropna(subset=["T", "mue_min", "mue_max", "sige_min", "sige_max"])
    .reset_index(drop=True)
)

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# correlation deviation mean
ax[0,0].fill_between(corr_dev_bounds_nz['T'],corr_dev_bounds_nz['mue_min'],corr_dev_bounds_nz['mue_max'], color="gray",alpha=0.25, linewidth=0)
ax[0,0].plot(pSA_T, e_NZ_eval['E']['mean'], '-o', label = 'E', lw = 1)
# ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,1].fill_between(corr_dev_bounds_chc['T'],corr_dev_bounds_chc['mue_min'],corr_dev_bounds_chc['mue_max'], color="gray",alpha=0.25, linewidth=0)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[0,2].fill_between(corr_dev_bounds_wel['T'],corr_dev_bounds_wel['mue_min'],corr_dev_bounds_wel['mue_max'], color="gray",alpha=0.25, linewidth=0)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[1,1].fill_between(corr_dev_bounds_chc['T'],corr_dev_bounds_chc['mue_min'],corr_dev_bounds_chc['mue_max'], color="gray",alpha=0.25, linewidth=0)
ax[1,1].plot(pSA_T, e_chc_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

ax[1,2].fill_between(corr_dev_bounds_wel['T'],corr_dev_bounds_wel['mue_min'],corr_dev_bounds_wel['mue_max'], color="gray",alpha=0.25, linewidth=0)
ax[1,2].plot(pSA_T, e_wel_eval['E']['mean'], '-o', label = 'E', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EA']['mean'], '-o', label = 'EA', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAS']['mean'], '-o', label = 'EAS', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZ']['mean'], '-o', label = 'EASZ', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAST']['mean'], '-o', label = 'EAST', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZT']['mean'], '-o', label = 'EASZT', lw = 1)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([-3,3])
        ax[i,j].set_xlim([0.01,6])
        if not (i == 1 and j == 0):
            ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\mu_e$')

ax[0,0].set_title('NZ-wide')
ax[0,1].set_title('Christchurch')
ax[0,2].set_title('Wellington')
# ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
ax[1,0].set_axis_off()
# ax[1,0].set_visible(False)
ax[1,0].legend(*ax[0,1].get_legend_handles_labels(),loc='center',fontsize=12,frameon=True)
plt.show(fig)

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# correlation deviation mean
ax[0,0].fill_between(corr_dev_bounds_nz['T'],corr_dev_bounds_nz['sige_min'],corr_dev_bounds_nz['sige_max'], color="gray",alpha=0.25, linewidth=0)
ax[0,0].plot(pSA_T, e_NZ_eval['E']['std'], '-o', label = 'E', lw = 1)
# ax[0,0].fill_between(pSA_T, e_NZ['E']['mean5'], e_NZ['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_NZ_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[0,0].plot(pSA_T, e_NZ_eval['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

ax[0,1].fill_between(corr_dev_bounds_chc['T'],corr_dev_bounds_chc['sige_min'],corr_dev_bounds_chc['sige_max'], color="gray",alpha=0.25, linewidth=0)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[0,1].plot(pSA_T, e_NZ_chc_eval['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

ax[0,2].fill_between(corr_dev_bounds_wel['T'],corr_dev_bounds_wel['sige_min'],corr_dev_bounds_wel['sige_max'], color="gray",alpha=0.25, linewidth=0)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[0,2].plot(pSA_T, e_NZ_wel_eval['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

# correlation deviation standard deviation
ax[1,1].fill_between(corr_dev_bounds_chc['T'],corr_dev_bounds_chc['sige_min'],corr_dev_bounds_chc['sige_max'], color="gray",alpha=0.25, linewidth=0)
ax[1,1].plot(pSA_T, e_chc_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,1].plot(pSA_T, e_chc_eval['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

ax[1,2].fill_between(corr_dev_bounds_wel['T'],corr_dev_bounds_wel['sige_min'],corr_dev_bounds_wel['sige_max'], color="gray",alpha=0.25, linewidth=0)
ax[1,2].plot(pSA_T, e_wel_eval['E']['std'], '-o', label = 'E', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EA']['std'], '-o', label = 'EA', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAS']['std'], '-o', label = 'EAS', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZ']['std'], '-o', label = 'EASZ', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EAST']['std'], '-o', label = 'EAST', lw = 1)
ax[1,2].plot(pSA_T, e_wel_eval['EASZT']['std'], '-o', label = 'EASZT', lw = 1)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([1,6])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'$\sigma_e$')

ax[0,0].set_title('NZ-wide')
ax[0,1].set_title('Christchurch')
ax[0,2].set_title('Wellington')
# ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
ax[1,0].set_axis_off()
# ax[1,0].set_visible(False)
ax[1,0].legend(*ax[0,1].get_legend_handles_labels(),loc='center',fontsize=12,frameon=True)
plt.show(fig)

# 7.0 Non-stationary SCM performance (LPPD)

## 7.1 Pooled model performance

### 7.1.1 NZ-Wide

In [ ]:
# Load Relative Difference from LPPDP (Performance of Pooled Models)
fig, ax = plt.subplots(figsize=(12, 6), dpi=200, constrained_layout=True)

# Compute differences from baseline model E
diffs = {
        'EA': (LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
               LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAS': (LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelEAS']['rel_diff'].reset_index(drop=True) -
                LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZ': (LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelEASZ']['rel_diff'].reset_index(drop=True) -
                 LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAST': (LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelEAST']['rel_diff'].reset_index(drop=True) -
                 LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZT': (LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelEASZT']['rel_diff'].reset_index(drop=True) -
                  LPPDP_NZ_reldiff[LPPDP_NZ_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
}

# Define colors
colors = {
        'EA':      'royalblue',      # strong blue
        'EAS':     'crimson',        # vivid red
        'EASZ':    'seagreen',       # green with blue tint
        'EAST':    'darkorange',     # deep orange
        'EASZT':   'saddlebrown',    # earthy brown
}

# Define distinct markers
markers = {
        'EA': 's',      # square
        'EAS': 'o',     # circle
        'EASZ': '^',    # triangle up
        'EAST': 'D',   # diamond
        'EASZT': '<',  # triangle left
}

# Sort models by final relative difference
sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])

# Plot stacked differences
previous_diff = pd.Series([0] * len(pSA_T))  # Baseline

for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T, current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T, previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

# Baseline line and annotation
ax.axhline(y=0, xmin=0.001, xmax=6, linestyle='-', color='black', linewidth=3, zorder=0)
ax.text(0.12, 0.2, 'Baseline: LPPD of isotropic model E',
            fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', linewidth=0.8))

# Axis configuration
ax.set_ylabel('Relative LPPD Difference (%)')
ax.set_xlabel('Period T [s]')
ax.set_xscale('log')
ax.set_xlim([0.01, 6])
ax.set_ylim([0,5])
ax.set_ylim(bottom=0)
ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)

# Explicit ticks
ax.set_xticks([0.01, 0.03, 0.06, 0.1, 0.3, 1.0, 3.0, 6.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())

# Legend
ax.legend(loc='upper left', fontsize=10, frameon=True, framealpha=0.8)

# Title
# fig.suptitle(r'Relative Difference in LPPD', fontsize=12, y=1.02)
plt.show()

### 7.1.2 Crustal

In [ ]:
# Load Relative Difference from LPPDP (Performance of Pooled Models)
fig, ax = plt.subplots(figsize=(12, 6), dpi=200, constrained_layout=True)

# Compute differences from baseline model E
diffs = {
        'EA': (LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
               LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAS': (LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelEAS']['rel_diff'].reset_index(drop=True) -
                LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZ': (LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelEASZ']['rel_diff'].reset_index(drop=True) -
                 LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAST': (LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelEAST']['rel_diff'].reset_index(drop=True) -
                 LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZT': (LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelEASZT']['rel_diff'].reset_index(drop=True) -
                  LPPDP_crus_reldiff[LPPDP_crus_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
}

# Define colors
colors = {
        'EA':      'royalblue',      # strong blue
        'EAS':     'crimson',        # vivid red
        'EASZ':    'seagreen',       # green with blue tint
        'EAST':    'darkorange',     # deep orange
        'EASZT':   'saddlebrown',    # earthy brown
}

# Define distinct markers
markers = {
        'EA': 's',      # square
        'EAS': 'o',     # circle
        'EASZ': '^',    # triangle up
        'EAST': 'D',   # diamond
        'EASZT': '<',  # triangle left
}

# Sort models by final relative difference
sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])

# Plot stacked differences
previous_diff = pd.Series([0] * len(pSA_T))  # Baseline

for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T, current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T, previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

# Baseline line and annotation
ax.axhline(y=0, xmin=0.001, xmax=6, linestyle='-', color='black', linewidth=3, zorder=0)
ax.text(0.12, 0.2, 'Baseline: LPPD of isotropic model E',
            fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', linewidth=0.8))

# Axis configuration
ax.set_ylabel('Relative LPPD Difference (%)')
ax.set_xlabel('Period T [s]')
ax.set_xscale('log')
ax.set_xlim([0.01, 6])
ax.set_ylim([0,7])
ax.set_ylim(bottom=0)
ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)

# Explicit ticks
ax.set_xticks([0.01, 0.03, 0.06, 0.1, 0.3, 1.0, 3.0, 6.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())

# Legend
ax.legend(loc='upper left', fontsize=10, frameon=True, framealpha=0.8)

# Title
# fig.suptitle(r'Relative Difference in LPPD', fontsize=12, y=1.02)
plt.show()

### 7.1.3 Subduction

In [ ]:
# Load Relative Difference from LPPDP (Performance of Pooled Models)
fig, ax = plt.subplots(figsize=(12, 6), dpi=200, constrained_layout=True)

# Compute differences from baseline model E
diffs = {
        'EA': (LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
               LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAS': (LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelEAS']['rel_diff'].reset_index(drop=True) -
                LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZ': (LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelEASZ']['rel_diff'].reset_index(drop=True) -
                 LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAST': (LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelEAST']['rel_diff'].reset_index(drop=True) -
                 LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZT': (LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelEASZT']['rel_diff'].reset_index(drop=True) -
                  LPPDP_sub_reldiff[LPPDP_sub_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
}

# Define colors
colors = {
        'EA':      'royalblue',      # strong blue
        'EAS':     'crimson',        # vivid red
        'EASZ':    'seagreen',       # green with blue tint
        'EAST':    'darkorange',     # deep orange
        'EASZT':   'saddlebrown',    # earthy brown
}

# Define distinct markers
markers = {
        'EA': 's',      # square
        'EAS': 'o',     # circle
        'EASZ': '^',    # triangle up
        'EAST': 'D',   # diamond
        'EASZT': '<',  # triangle left
}

# Sort models by final relative difference
sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])

# Plot stacked differences
previous_diff = pd.Series([0] * len(pSA_T))  # Baseline

for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T, current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T, previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

# Baseline line and annotation
ax.axhline(y=0, xmin=0.001, xmax=6, linestyle='-', color='black', linewidth=3, zorder=0)
ax.text(0.12, 0.2, 'Baseline: LPPD of isotropic model E',
            fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', linewidth=0.8))

# Axis configuration
ax.set_ylabel('Relative LPPD Difference (%)')
ax.set_xlabel('Period T [s]')
ax.set_xscale('log')
ax.set_xlim([0.01, 6])
ax.set_ylim([0,7])
ax.set_ylim(bottom=0)
ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)

# Explicit ticks
ax.set_xticks([0.01, 0.03, 0.06, 0.1, 0.3, 1.0, 3.0, 6.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())

# Legend
ax.legend(loc='upper left', fontsize=10, frameon=True, framealpha=0.8)

# Title
# fig.suptitle(r'Relative Difference in LPPD', fontsize=12, y=1.02)
plt.show()

## 7.2 Event-specific model performance

### 7.2.1 NZ-Wide fitted SCM

In [ ]:
# Define periods
psa_EES = ['PGA','pSA_0.03','pSA_0.06','pSA_0.1','pSA_0.3','pSA_0.6','pSA_1.0','pSA_2.0','pSA_3.0','pSA_4.0','pSA_5.0','pSA_6.0']
psa_EEES = ['E_PGA','E_pSA_0.03','E_pSA_0.06','E_pSA_0.1','E_pSA_0.3','E_pSA_0.6','E_pSA_1.0','E_pSA_2.0','E_pSA_3.0','E_pSA_4.0','E_pSA_5.0','E_pSA_6.0']
psa_EAES = ['EA_PGA','EA_pSA_0.03','EA_pSA_0.06','EA_pSA_0.1','EA_pSA_0.3','EA_pSA_0.6','EA_pSA_1.0','EA_pSA_2.0','EA_pSA_3.0','EA_pSA_4.0','EA_pSA_5.0','EA_pSA_6.0']
psa_EASES = ['EAS_PGA','EAS_pSA_0.03','EAS_pSA_0.06','EAS_pSA_0.1','EAS_pSA_0.3','EAS_pSA_0.6','EAS_pSA_1.0','EAS_pSA_2.0','EAS_pSA_3.0','EAS_pSA_4.0','EAS_pSA_5.0','EAS_pSA_6.0']
psa_EASZES = ['EASZ_PGA','EASZ_pSA_0.03','EASZ_pSA_0.06','EASZ_pSA_0.1','EASZ_pSA_0.3','EASZ_pSA_0.6','EASZ_pSA_1.0','EASZ_pSA_2.0','EASZ_pSA_3.0','EASZ_pSA_4.0','EASZ_pSA_5.0','EASZ_pSA_6.0']
psa_EASTES = ['EAST_PGA','EAST_pSA_0.03','EAST_pSA_0.06','EAST_pSA_0.1','EAST_pSA_0.3','EAST_pSA_0.6','EAST_pSA_1.0','EAST_pSA_2.0','EAST_pSA_3.0','EAST_pSA_4.0','EAST_pSA_5.0','EAST_pSA_6.0']
psa_EASZTES = ['EASZT_PGA','EASZT_pSA_0.03','EASZT_pSA_0.06','EASZT_pSA_0.1','EASZT_pSA_0.3','EASZT_pSA_0.6','EASZT_pSA_1.0','EASZT_pSA_2.0','EASZT_pSA_3.0','EASZT_pSA_4.0','EASZT_pSA_5.0','EASZT_pSA_6.0']

vmin, vmax = 3.5, 7.5

# helper scatter function
def scatter_panel(ax, x, y, c, marker, label=None):
    return ax.scatter(
        x, y,
        c=c,
        cmap='coolwarm',
        vmin=vmin, vmax=vmax,
        marker=marker,
        edgecolor='k',
        lw=0.1,
        label=label
    )

def format_ticks_as_int(val, pos):
    return f"{int(val):d}" if val >= 1 else ""

# assign markers for tectonic classes
tect_markers = {
    'Crustal': 'o',
    'Interface': 's',
    'Slab': '^'
}

pSA_plot_idx = 8
fig, ax = plt.subplots(2,3, figsize=(10,5), dpi=200, constrained_layout=True)

# loop over tectonic classes
scat = None
for tect, marker in tect_markers.items():
    mask = LPPDES.tect_class == tect
    # top-left panel as example
    scat = scatter_panel(ax[0,0],LPPDES.n_recs[mask],((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE_NZ[psa_EEES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100,
        LPPDES.mag[mask],
        marker,
        label=tect
    )

    # repeat for other subplots
    scatter_panel(ax[0,1], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE_NZ[psa_EAES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[0,2], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE_NZ[psa_EASES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker, label = tect)
    scatter_panel(ax[1,0], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE_NZ[psa_EASZES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[1,1], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE_NZ[psa_EASTES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[1,2], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE_NZ[psa_EASZTES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)

# formatting
for i in range(2):
    ax[i,0].set_ylabel('Relative Difference in LPPD (%)')
    for j in range(3):
        ax[i,j].set_xscale('log')
        ax[i,j].grid(True, which='both', linestyle='--', alpha=0.5)
        ax[i,j].set_ylim([-20, 20])
        ax[i,j].xaxis.set_major_formatter(FuncFormatter(format_ticks_as_int))
        ax[i,j].tick_params(axis='x', which='major')
        ax[i,j].set_xticks([40, 60, 80, 100, 200])
        ax[i,j].plot([LPPDES.n_recs.min(), LPPDES.n_recs.max()], [0, 0], '--k')
        ax[i,j].set_xlim(LPPDES.n_recs.min(), LPPDES.n_recs.max())

xgrid_EES, _, mean_EES, std_EES = smooth(LPPDES.n_recs,((LPPDES[psa_EES[pSA_plot_idx]] - PO_LPPDE_NZ[psa_EEES[pSA_plot_idx]]) / LPPDES[psa_EES[pSA_plot_idx]]) * 100, K = 100, show_progress=False)
ax[0,0].plot(xgrid_EES, mean_EES, color = "green")
ax[0,0].fill_between(xgrid_EES, mean_EES-std_EES, mean_EES+std_EES, color = "green", alpha =0.2)
ax[0,0].fill_between(xgrid_EES, mean_EES-std_EES, mean_EES+std_EES, color = "green",alpha =0.2)

xgrid_EAES, _, mean_EAES, std_EAES = smooth(LPPDES.n_recs,((LPPDES[psa_EES[pSA_plot_idx]] - PO_LPPDE_NZ[psa_EAES[pSA_plot_idx]]) / LPPDES[psa_EES[pSA_plot_idx]]) * 100, K = 100, show_progress=False)
ax[0,1].plot(xgrid_EAES, mean_EAES, color = "green")
ax[0,1].fill_between(xgrid_EAES, mean_EAES-std_EAES, mean_EAES+std_EAES, color = "green", alpha =0.2)
ax[0,1].fill_between(xgrid_EAES, mean_EAES-std_EAES, mean_EAES+std_EAES, color = "green",alpha =0.2)

xgrid_EASES, _, mean_EASES, std_EASES = smooth(LPPDES.n_recs,((LPPDES[psa_EES[pSA_plot_idx]] - PO_LPPDE_NZ[psa_EASES[pSA_plot_idx]]) / LPPDES[psa_EES[pSA_plot_idx]]) * 100, K = 100, show_progress=False)
ax[0,2].plot(xgrid_EASES, mean_EASES, color = "green")
ax[0,2].fill_between(xgrid_EASES, mean_EASES-std_EASES, mean_EASES+std_EASES, color = "green", alpha =0.2)
ax[0,2].fill_between(xgrid_EASES, mean_EASES-std_EASES, mean_EASES+std_EASES, color = "green",alpha =0.2)

xgrid_EASZES, _, mean_EASZES, std_EASZES = smooth(LPPDES.n_recs,((LPPDES[psa_EES[pSA_plot_idx]] - PO_LPPDE_NZ[psa_EASZES[pSA_plot_idx]]) / LPPDES[psa_EES[pSA_plot_idx]]) * 100, K = 100, show_progress=False)
ax[1,0].plot(xgrid_EASZES, mean_EASZES, color = "green")
ax[1,0].fill_between(xgrid_EASZES, mean_EASZES-std_EASZES, mean_EASZES+std_EASZES, color = "green", alpha =0.2)
ax[1,0].fill_between(xgrid_EASZES, mean_EASZES-std_EASZES, mean_EASZES+std_EASZES, color = "green",alpha =0.2)

xgrid_EASTES, _, mean_EASTES, std_EASTES = smooth(LPPDES.n_recs,((LPPDES[psa_EES[pSA_plot_idx]] - PO_LPPDE_NZ[psa_EASTES[pSA_plot_idx]]) / LPPDES[psa_EES[pSA_plot_idx]]) * 100, K = 100, show_progress=False)
ax[1,1].plot(xgrid_EASTES, mean_EASTES, color = "green")
ax[1,1].fill_between(xgrid_EASTES, mean_EASTES-std_EASTES, mean_EASTES+std_EASTES, color = "green", alpha =0.2)
ax[1,1].fill_between(xgrid_EASTES, mean_EASTES-std_EASTES, mean_EASTES+std_EASTES, color = "green",alpha =0.2)

xgrid_EASZTES, _, mean_EASZTES, std_EASZTES = smooth(LPPDES.n_recs,((LPPDES[psa_EES[pSA_plot_idx]] - PO_LPPDE_NZ[psa_EASZTES[pSA_plot_idx]]) / LPPDES[psa_EES[pSA_plot_idx]]) * 100, K = 100, show_progress=False)
ax[1,2].plot(xgrid_EASZTES, mean_EASZTES, color = "green")
ax[1,2].fill_between(xgrid_EASZTES, mean_EASZTES-std_EASZTES, mean_EASZTES+std_EASZTES, color = "green", alpha =0.2)
ax[1,2].fill_between(xgrid_EASZTES, mean_EASZTES-std_EASZTES, mean_EASZTES+std_EASZTES, color = "green",alpha =0.2)

# titles
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[1,0].set_xlabel('Number of Records')
ax[1,1].set_xlabel('Number of Records')
ax[1,2].set_xlabel('Number of Records')

# shared colorbar
cbar = fig.colorbar(scat, ax=fig.axes, orientation='vertical', fraction=0.05, pad=0.02, aspect=40)
cbar.set_label('Magnitude', fontsize = 12)

# legend for tectonic class in top-right panel
ax[0,0].legend(title='Tectonic Class', loc='upper right')

plt.show()

### 7.2.2 Relative differences for all events

In [ ]:
rel_diff = LPPDES[["evid","n_recs"]].copy()
for i, pSA_label in enumerate(psa_EES):
    rel_diff[psa_EEES[i]] = ((LPPDES[psa_EES[i]] - PO_LPPDE_NZ[psa_EEES[i]]) / LPPDES[psa_EES[i]]) * 100
    rel_diff[psa_EASES[i]] = ((LPPDES[psa_EES[i]] - PO_LPPDE_NZ[psa_EASES[i]]) / LPPDES[psa_EES[i]]) * 100
    rel_diff[psa_EASZES[i]] = ((LPPDES[psa_EES[i]] - PO_LPPDE_NZ[psa_EASZES[i]]) / LPPDES[psa_EES[i]]) * 100
    rel_diff[psa_EASTES[i]] = ((LPPDES[psa_EES[i]] - PO_LPPDE_NZ[psa_EASTES[i]]) / LPPDES[psa_EES[i]]) * 100
    rel_diff[psa_EASZTES[i]] = ((LPPDES[psa_EES[i]] - PO_LPPDE_NZ[psa_EASZTES[i]]) / LPPDES[psa_EES[i]]) * 100

rel_diff.to_csv(RESULTS_DIR / "NZSCM_rel_diff.csv", index=False)
evid_EAZT_sa1s = rel_diff[rel_diff['EASZT_pSA_1.0']>=5].reset_index(drop=True).dropna()['evid'].tolist()
evid_EAZT_sa3s = rel_diff[rel_diff['EASZT_pSA_3.0']>=5].reset_index(drop=True).dropna()['evid'].tolist()
evid_EAZT_sa6s = rel_diff[rel_diff['EASZT_pSA_6.0']>=5].reset_index(drop=True).dropna()['evid'].tolist()

# 8.0 Advanced NZ-Specific SCM evaluation

## 8.1 Correlation deviations: Varying hmax

In [ ]:
hmax_vars = np.array([1,2,4,6,8,10,12,14,16,18,20,22,24,26,28,30,32,34,36,38,40,50,60,70,80,90,100])
periods = pSA_T
SCM_ns_models = ['E', 'EA', 'EAS', 'EASZ', 'EAST', 'EASZT']

### 8.1.1 Compute correlation cloud and store correlation deviations

#### 8.1.1.1 NZ wide

In [ ]:
mue_NZ_hmax = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['mean','mean5','mean95'], periods],
        names=['model','stat','period']
    )
)

sige_NZ_hmax = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['std','std5','std95'], periods],
        names=['model','stat','period']
    )
)

for hmax in tqdm(hmax_vars, desc = 'Processing hmax: '):

    cbb21_NZ = []

    # nz-wide correlation cloud across period
    for i in range(len(im_dwbar_pSA_files_n3)):
        # im_dwbar_pSA_files_n3[i]['dw_bar'] = im_dwbar_pSA_files_n3[i][pSAobs_T[i]]
        # cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_n3[i],n_rec,hmax)
        im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].evid.isin(evid_EAZT_sa6s)].reset_index(drop = True)
        im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
        cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
        cbb21_NZ.append(cbb21)

    # compute correlation devaitions
    e_NZ = BM23EXT_CBB21(cbb21_NZ, pSA_T, nz_params)

    for key in e_NZ:
        mue_NZ_hmax.loc[hmax, (key, 'mean')]   = e_NZ[key]['mean']
        mue_NZ_hmax.loc[hmax, (key, 'mean5')]  = e_NZ[key]['mean5']
        mue_NZ_hmax.loc[hmax, (key, 'mean95')] = e_NZ[key]['mean95']

        sige_NZ_hmax.loc[hmax, (key, 'std')]    = e_NZ[key]['std']
        sige_NZ_hmax.loc[hmax, (key, 'std5')]   = e_NZ[key]['std5']
        sige_NZ_hmax.loc[hmax, (key, 'std95')]  = e_NZ[key]['std95']

#### 8.1.1.2 Group 2

In [ ]:
mue_grp2_hmax = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['mean','mean5','mean95'], periods],
        names=['model','stat','period']
    )
)

sige_grp2_hmax = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['std','std5','std95'], periods],
        names=['model','stat','period']
    )
)

for hmax in tqdm(hmax_vars, desc = 'Processing hmax: '):

    cbb21_NZ = []

    # nz-wide correlation cloud across period
    for i in range(len(im_dwbar_pSA_files_grp2_n3)):
        im_dwbar_pSA_files_grp2_n3[i]['dw_bar'] = im_dwbar_pSA_files_grp2_n3[i][pSAobs_T[i]]
        cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grp2_n3[i],n_rec,hmax)
        cbb21_NZ.append(cbb21)

    # compute correlation devaitions
    e_NZ = BM23EXT_CBB21(cbb21_NZ, pSA_T, grp2_params)

    for key in e_NZ:
        mue_grp2_hmax.loc[hmax, (key, 'mean')]   = e_NZ[key]['mean']
        mue_grp2_hmax.loc[hmax, (key, 'mean5')]  = e_NZ[key]['mean5']
        mue_grp2_hmax.loc[hmax, (key, 'mean95')] = e_NZ[key]['mean95']

        sige_grp2_hmax.loc[hmax, (key, 'std')]    = e_NZ[key]['std']
        sige_grp2_hmax.loc[hmax, (key, 'std5')]   = e_NZ[key]['std5']
        sige_grp2_hmax.loc[hmax, (key, 'std95')]  = e_NZ[key]['std95']

### 8.1.2 SCM Models vs. hmax

In [ ]:
plot_periods = [0.01, 0.06, 0.6, 1, 3, 6]

# NZ-Wide
fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()

for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        mean   = mue_NZ_hmax[(model, 'mean',   T)].to_numpy(dtype=float)
        mean5  = mue_NZ_hmax[(model, 'mean5',  T)].to_numpy(dtype=float)
        mean95 = mue_NZ_hmax[(model, 'mean95', T)].to_numpy(dtype=float)
        a.plot(mue_NZ_hmax.index, mean, label=model, linewidth=1.5)
        a.fill_between(mue_NZ_hmax.index, mean5, mean95, alpha=0.15)
    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation mean, $\mu_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(),hmax_vars.max())
    a.set_ylim(-4,4)
    a.plot([hmax_vars.min(),hmax_vars.max()],[0,0],'--k', lw = 1.5)
handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="upper left")
plt.show()

fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()
for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        std    = sige_NZ_hmax[(model, 'std',   T)].to_numpy(dtype=float)
        std5   = sige_NZ_hmax[(model, 'std5',  T)].to_numpy(dtype=float)
        std95  = sige_NZ_hmax[(model, 'std95', T)].to_numpy(dtype=float)
        a.plot(mue_NZ_hmax.index, std, label=model, linewidth=1.5)
        a.fill_between(mue_NZ_hmax.index, std5, std95, alpha=0.15)

    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation standard deviation, $\sigma_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(), hmax_vars.max())
    a.set_ylim(1, 6)

handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="lower left")
plt.show()

# Group 2
fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()

for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        mean   = mue_grp2_hmax[(model, 'mean',   T)].to_numpy(dtype=float)
        mean5  = mue_grp2_hmax[(model, 'mean5',  T)].to_numpy(dtype=float)
        mean95 = mue_grp2_hmax[(model, 'mean95', T)].to_numpy(dtype=float)
        a.plot(mue_grp2_hmax.index, mean, label=model, linewidth=1.5)
        a.fill_between(mue_grp2_hmax.index, mean5, mean95, alpha=0.15)
    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation mean, $\mu_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(),hmax_vars.max())
    a.set_ylim(-4,4)
    a.plot([hmax_vars.min(),hmax_vars.max()],[0,0],'--k', lw = 1.5)
handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="upper left")
plt.show()

fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()
for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        std    = sige_grp2_hmax[(model, 'std',   T)].to_numpy(dtype=float)
        std5   = sige_grp2_hmax[(model, 'std5',  T)].to_numpy(dtype=float)
        std95  = sige_grp2_hmax[(model, 'std95', T)].to_numpy(dtype=float)
        a.plot(mue_NZ_hmax.index, std, label=model, linewidth=1.5)
        a.fill_between(mue_NZ_hmax.index, std5, std95, alpha=0.15)

    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation standard deviation, $\sigma_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(), hmax_vars.max())
    a.set_ylim(0, 6)

handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="lower left")
plt.show()

## 8.2 Correlation deviations: Mw > 5 and varying hmax

### 8.2.1 Compute correlation cloud and store correlation deviations

#### 8.2.1.1 NZ wide

In [ ]:
mue_NZ_hmax_mw5 = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['mean','mean5','mean95'], periods],
        names=['model','stat','period']
    )
)

sige_NZ_hmax_mw5 = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['std','std5','std95'], periods],
        names=['model','stat','period']
    )
)

for hmax in tqdm(hmax_vars, desc = 'Processing hmax: '):

    cbb21_NZ = []

    # nz-wide correlation cloud across period
    for i in range(len(im_dwbar_pSA_files_n3)):
        im_dwbar_pSA_files_n3[i]['dw_bar'] = im_dwbar_pSA_files_n3[i][pSAobs_T[i]]
        cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].evid.isin(evid_mw5)].reset_index(drop=True),n_rec,hmax)
        cbb21_NZ.append(cbb21)

    # compute correlation devaitions
    e_NZ = BM23EXT_CBB21(cbb21_NZ, pSA_T, nz_params)

    for key in e_NZ:
        mue_NZ_hmax_mw5.loc[hmax, (key, 'mean')]   = e_NZ[key]['mean']
        mue_NZ_hmax_mw5.loc[hmax, (key, 'mean5')]  = e_NZ[key]['mean5']
        mue_NZ_hmax_mw5.loc[hmax, (key, 'mean95')] = e_NZ[key]['mean95']

        sige_NZ_hmax_mw5.loc[hmax, (key, 'std')]    = e_NZ[key]['std']
        sige_NZ_hmax_mw5.loc[hmax, (key, 'std5')]   = e_NZ[key]['std5']
        sige_NZ_hmax_mw5.loc[hmax, (key, 'std95')]  = e_NZ[key]['std95']

#### 8.2.1.2 Group 2

In [ ]:
mue_grp2_hmax_mw5 = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['mean','mean5','mean95'], periods],
        names=['model','stat','period']
    )
)

sige_grp2_hmax_mw5 = pd.DataFrame(
    index=hmax_vars,
    columns=pd.MultiIndex.from_product(
        [SCM_ns_models, ['std','std5','std95'], periods],
        names=['model','stat','period']
    )
)

for hmax in tqdm(hmax_vars, desc = 'Processing hmax: '):

    cbb21_grp2 = []

    # group 2 correlation cloud across period
    for i in range(len(im_dwbar_pSA_files_grp2_n3)):
        im_dwbar_pSA_files_grp2_n3[i]['dw_bar'] = im_dwbar_pSA_files_grp2_n3[i][pSAobs_T[i]]
        cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_grp2_n3[i][im_dwbar_pSA_files_grp2_n3[i].evid.isin(evid_mw5)].reset_index(drop=True),n_rec,hmax)
        cbb21_grp2.append(cbb21)

    # compute correlation devaitions
    e_grp2 = BM23EXT_CBB21(cbb21_grp2, pSA_T, nz_params)

    for key in e_grp2:
        mue_grp2_hmax_mw5.loc[hmax, (key, 'mean')]   = e_grp2[key]['mean']
        mue_grp2_hmax_mw5.loc[hmax, (key, 'mean5')]  = e_grp2[key]['mean5']
        mue_grp2_hmax_mw5.loc[hmax, (key, 'mean95')] = e_grp2[key]['mean95']

        sige_grp2_hmax_mw5.loc[hmax, (key, 'std')]    = e_grp2[key]['std']
        sige_grp2_hmax_mw5.loc[hmax, (key, 'std5')]   = e_grp2[key]['std5']
        sige_grp2_hmax_mw5.loc[hmax, (key, 'std95')]  = e_grp2[key]['std95']

### 8.2.2 SCM models vs. hmax

In [ ]:
plot_periods = [0.01, 0.06, 0.6, 1, 3, 6]

# NZ-Wide
fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()

for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        mean   = mue_NZ_hmax_mw5[(model, 'mean',   T)].to_numpy(dtype=float)
        mean5  = mue_NZ_hmax_mw5[(model, 'mean5',  T)].to_numpy(dtype=float)
        mean95 = mue_NZ_hmax_mw5[(model, 'mean95', T)].to_numpy(dtype=float)
        a.plot(mue_NZ_hmax_mw5.index, mean, label=model, linewidth=1.5)
        a.fill_between(mue_NZ_hmax_mw5.index, mean5, mean95, alpha=0.15)
    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation mean, $\mu_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(),hmax_vars.max())
    a.set_ylim(-4,4)
    a.plot([hmax_vars.min(),hmax_vars.max()],[0,0],'--k', lw = 1.5)
handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="upper left")
plt.show()

fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()
for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        std    = sige_NZ_hmax_mw5[(model, 'std',   T)].to_numpy(dtype=float)
        std5   = sige_NZ_hmax_mw5[(model, 'std5',  T)].to_numpy(dtype=float)
        std95  = sige_NZ_hmax_mw5[(model, 'std95', T)].to_numpy(dtype=float)
        a.plot(mue_NZ_hmax_mw5.index, std, label=model, linewidth=1.5)
        a.fill_between(mue_NZ_hmax_mw5.index, std5, std95, alpha=0.15)

    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation standard deviation, $\sigma_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(), hmax_vars.max())
    a.set_ylim(1, 6)

handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="lower left")
plt.show()

# Group 2
fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()

for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        mean   = mue_grp2_hmax_mw5[(model, 'mean',   T)].to_numpy(dtype=float)
        mean5  = mue_grp2_hmax_mw5[(model, 'mean5',  T)].to_numpy(dtype=float)
        mean95 = mue_grp2_hmax_mw5[(model, 'mean95', T)].to_numpy(dtype=float)
        a.plot(mue_grp2_hmax_mw5.index, mean, label=model, linewidth=1.5)
        a.fill_between(mue_grp2_hmax_mw5.index, mean5, mean95, alpha=0.15)
    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation mean, $\mu_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(),hmax_vars.max())
    a.set_ylim(-4,4)
    a.plot([hmax_vars.min(),hmax_vars.max()],[0,0],'--k', lw = 1.5)
handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="upper left")
plt.show()

fig, ax = plt.subplots(2, 3, figsize=(12, 8), dpi=200, constrained_layout=True)
ax = ax.flatten()
for i, T in enumerate(plot_periods):
    a = ax[i]

    for model in SCM_ns_models:
        std    = sige_grp2_hmax_mw5[(model, 'std',   T)].to_numpy(dtype=float)
        std5   = sige_grp2_hmax_mw5[(model, 'std5',  T)].to_numpy(dtype=float)
        std95  = sige_grp2_hmax_mw5[(model, 'std95', T)].to_numpy(dtype=float)
        a.plot(mue_NZ_hmax_mw5.index, std, label=model, linewidth=1.5)
        a.fill_between(mue_NZ_hmax_mw5.index, std5, std95, alpha=0.15)

    a.set_title(f"T = {T}s")
    a.set_xlabel(r"$h_{max} (km)$")
    a.set_ylabel(r"Correlation deviation standard deviation, $\sigma_e$")
    a.grid(True, which='both', linestyle='--', alpha=0.3)
    a.set_xscale('log')
    a.set_xlim(hmax_vars.min(), hmax_vars.max())
    a.set_ylim(0, 6)

handles, labels = ax[0].get_legend_handles_labels()
ax[0].legend(handles, labels, loc="lower left")
plt.show()

# 9.0 Correlation Deviations of NZ-specific SCMs

## 9.1 Function definitions

In [ ]:
def plot_corr_deviation(results, obs_T_dict, e_dict, region, model_key, period_indices, period_labels, xlims = None, show_plot = True, save_path = None):
    """
    region: 'all', 'chc', 'wel'
    model_key: 'JB09', 'BM23_E', 'BM23_EA', 'BM23_EAS'
    period_indices: list of indices into period_BM23
    period_labels: labels for each period
    """

    feature_names = ['h', 'dA', 'dS', 'dTRI', 'dZ1']

    # Default x-limits
    if xlims is None:
        xlims = {
            'h': (0, 25),
            'dA': (0, 180),
            'dS': (0, 760),
            'dTRI': (0, 90),
            'dZ1': (0, 760)
        }

    # Observations for region
    obs_list = obs_T_dict[region]

    # Residuals for region + model
    if model_key == "JB09":
        y_list = e_dict[region]["JB09"]
    else:
        model_name, variant = model_key.split("_")
        y_list = e_dict[region][model_name][variant]

    # Layout
    nrows = len(period_indices)
    ncols = len(feature_names)

    fig, ax = plt.subplots(
        nrows, ncols,
        figsize=(3*ncols, 3*nrows),
        dpi=150,
        constrained_layout=True
    )

    if nrows == 1:
        ax = np.expand_dims(ax, axis=0)

    # Loop over periods
    for row_idx, (pi, plabel) in enumerate(zip(period_indices, period_labels)):

        # Loop over features
        for col_idx, feat in enumerate(feature_names):

            x = getattr(obs_list[pi], feat)
            p = obs_list[pi].p_hat

            # ----------------------------------------------------------
            # Get residuals safely
            # ----------------------------------------------------------
            if model_key == "JB09":
                y = y_list[pi]
            else:
                # BM23 always aligned to period_BM23
                y = y_list[pi] if y_list[pi] is not None else None

            # ----------------------------------------------------------
            # Scatter only if y exists
            # ----------------------------------------------------------
            if y is not None:
                sc = ax[row_idx, col_idx].scatter(
                    x, y, c=p, vmin=-1, vmax=1, cmap='bwr', s=12
                )
            else:
                ax[row_idx, col_idx].text(
                    0.5, 0.5,
                    "No BM23 data",
                    ha='center', va='center',
                    fontsize=9, color='gray',
                    transform=ax[row_idx, col_idx].transAxes
                )
                sc = None

            # ----------------------------------------------------------
            # Smoothed LOWESS results
            # ----------------------------------------------------------
            xgrid, mean, stderr = results[region][feat][model_key][pi]

            if xgrid is not None:
                ax[row_idx, col_idx].plot(xgrid, mean, '-k', linewidth=1)
                ax[row_idx, col_idx].fill_between(
                    xgrid,
                    mean - 1.96 * stderr,
                    mean + 1.96 * stderr,
                    alpha=0.3,
                    color='forestgreen'
                )

            ax[row_idx, col_idx].set_xlim(xlims[feat])
            ax[row_idx, col_idx].set_ylim([-5, 5])

    # Y labels
    for i, label in enumerate(period_labels):
        ax[i, 0].set_ylabel(fr'Corr. Deviation, $e$ ({label})')

    # X labels
    ax[-1, 0].set_xlabel(r'$h$ (km)')
    ax[-1, 1].set_xlabel(r'$dA$ $(\degree)$')
    ax[-1, 2].set_xlabel(r'$dS$ (m/s)')
    ax[-1, 3].set_xlabel(r'$dTRI$')
    ax[-1, 4].set_xlabel(r'$dZ_{1.0}$ (m)')

    # Colorbar (only if scatter exists)
    if sc is not None:
        fig.colorbar(
            sc, ax=ax,
            label=r'Correlation Coefficient, $\hat{p}$',
            aspect=40, pad=0.02
        )

    if save_path is not None:
        fig.savefig(save_path)

    if show_plot:
        plt.show()
    else:
        plt.close('all')

def plot_corr_deviation_multiregion(
    results,
    obs_T_dict,
    e_dict,
    regions,
    model_key,
    period_index,
    period_label,
    feature_names,
    xlabels,
    xlims_dict
):
    ncols = len(feature_names)
    fig, ax = plt.subplots(
        1, ncols,
        figsize=(3*ncols, 3),
        dpi=150,
        constrained_layout=True
    )

    if ncols == 1:
        ax = [ax]

    for col_idx, feat in enumerate(feature_names):
        axc = ax[col_idx]

        for region in regions:
            obs_list = obs_T_dict[region]

            # Extract residuals
            if model_key == "JB09":
                y_list = e_dict[region]["JB09"]
            else:
                model_name, variant = model_key.split("_")
                y_list = e_dict[region][model_name][variant]

            x = getattr(obs_list[period_index], feat)
            y = y_list[period_index]
            p = obs_list[period_index].p_hat

            # Scatter (same color scheme for all regions)
            axc.scatter(
                x, y,
                c=p, vmin=-1, vmax=1, cmap='bwr',
                s=14, alpha=0.8
            )

            # Smoothed curve (always black)
            xgrid, mean, stderr = results[region][feat][model_key][period_index]
            if xgrid is not None:
                axc.plot(
                    xgrid, mean,
                    color="black",
                    linewidth=1.4
                )
                axc.fill_between(
                    xgrid,
                    mean - 1.96 * stderr,
                    mean + 1.96 * stderr,
                    alpha=0.25,
                    color="forestgreen"
                )

        axc.set_xlim(xlims_dict[regions[0]][feat])
        axc.set_ylim([-5, 5])
        axc.set_xlabel(xlabels[col_idx], fontsize=11)

        if col_idx == 0:
            axc.set_ylabel(fr'$e$ ({period_label})', fontsize=12)

    # Unified colorbar
    norm = Normalize(vmin=-1, vmax=1)
    sm = ScalarMappable(norm=norm, cmap='bwr')
    sm.set_array([])
    fig.colorbar(
        sm, ax=ax,
        label=r'Correlation Coefficient, $\hat{\rho}$',
        aspect=30, pad=0.01
    )

    plt.show()

## 9.2 Compute correlation cloud and deviation

In [ ]:
n_rec = 6
hmax = 25

### 9.2.1 SCM development dataset correlation cloud

In [ ]:
cbb21_pSAobs_T_dev = []; cbb21_pSAobs_T_chc_dev = []; cbb21_pSAobs_T_wel_dev = []

for i in tqdm(range(len(pSAobs_T)), desc="pSA: "):

    # all data
    im_dwbar_pSA_files[i]['dw_bar'] = im_dwbar_pSA_files[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files[i],n_rec,hmax)
    cbb21_pSAobs_T_dev.append(cbb21)

    # group 1: wellington
    im_dwbar_temp = im_dwbar_pSA_files[i][im_dwbar_pSA_files[i].sta.isin(stations_wel.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_wel_dev.append(cbb21)

    # group 2: christchurch
    im_dwbar_temp = im_dwbar_pSA_files[i][im_dwbar_pSA_files[i].sta.isin(stations_chc.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_chc_dev.append(cbb21)

### 9.2.2 SCM evaluation dataset correlation cloud

In [ ]:
cbb21_pSAobs_T_eval = []; cbb21_pSAobs_T_chc_eval = []; cbb21_pSAobs_T_wel_eval = []

for i in tqdm(range(len(pSAobs_T)), desc="pSA: "):

    # all data
    im_dwbar_pSA_files_n3[i]['dw_bar'] = im_dwbar_pSA_files_n3[i][pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_pSA_files_n3[i],n_rec,hmax)
    cbb21_pSAobs_T_eval.append(cbb21)

    # group 1: wellington
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(stations_wel.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_wel_eval.append(cbb21)

    # group 2: christchurch
    im_dwbar_temp = im_dwbar_pSA_files_n3[i][im_dwbar_pSA_files_n3[i].sta.isin(stations_chc.sta)].reset_index(drop = True)
    im_dwbar_temp['dw_bar'] = im_dwbar_temp[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_temp,n_rec,hmax)
    cbb21_pSAobs_T_chc_eval.append(cbb21)

### 9.2.3 SCM Correlation deviations

#### 9.2.3.1 Jayaram and Baker 2009

In [ ]:
# Jayaram and Baker 2009 Inputs
case = 2

# Development dataset
# lists for storing information
JB09_ee_dev, JB09_e_dev, JB09_e5_dev, JB09_e95_dev, JB09_e_std_dev, JB09_e5_std_dev, JB09_e95_std_dev = [], [], [], [], [], [], []
JB09_ee_chc_dev, JB09_e_chc_dev, JB09_e5_chc_dev, JB09_e95_chc_dev, JB09_e_std_chc_dev, JB09_e5_std_chc_dev, JB09_e95_std_chc_dev = [], [], [], [], [], [], []
JB09_ee_wel_dev, JB09_e_wel_dev, JB09_e5_wel_dev, JB09_e95_wel_dev, JB09_e_std_wel_dev, JB09_e5_std_wel_dev, JB09_e95_std_wel_dev = [], [], [], [], [], [], []

for i in tqdm(range(len(pSA_T))):

    # NZ GMDB
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = gmscm.GMSCM_JB09(case, pSA_T[i], hmax, n_rec, cbb21_pSAobs_T_dev[i], plot=False)
    JB09_ee_dev.append(e); JB09_e_dev.append(e_mean); JB09_e5_dev.append(e5); JB09_e95_dev.append(e95)
    JB09_e_std_dev.append(e_std); JB09_e5_std_dev.append(estd_5); JB09_e95_std_dev.append(estd_95)

    # Christchurch
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = gmscm.GMSCM_JB09(case, pSA_T[i], hmax, n_rec, cbb21_pSAobs_T_chc_dev[i], plot=False)
    JB09_ee_chc_dev.append(e); JB09_e_chc_dev.append(e_mean); JB09_e5_chc_dev.append(e5); JB09_e95_chc_dev.append(e95)
    JB09_e_std_chc_dev.append(e_std); JB09_e5_std_chc_dev.append(estd_5); JB09_e95_std_chc_dev.append(estd_95)

    # Wellington
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = gmscm.GMSCM_JB09(1, pSA_T[i], hmax, n_rec, cbb21_pSAobs_T_wel_dev[i], plot=False)
    JB09_ee_wel_dev.append(e); JB09_e_wel_dev.append(e_mean); JB09_e5_wel_dev.append(e5); JB09_e95_wel_dev.append(e95)
    JB09_e_std_wel_dev.append(e_std); JB09_e5_std_wel_dev.append(estd_5); JB09_e95_std_wel_dev.append(estd_95)

# evaluation dataset
# lists for storing information
JB09_ee_eval, JB09_e_eval, JB09_e5_eval, JB09_e95_eval, JB09_e_std_eval, JB09_e5_std_eval, JB09_e95_std_eval = [], [], [], [], [], [], []
JB09_ee_chc_eval, JB09_e_chc_eval, JB09_e5_chc_eval, JB09_e95_chc_eval, JB09_e_std_chc_eval, JB09_e5_std_chc_eval, JB09_e95_std_chc_eval = [], [], [], [], [], [], []
JB09_ee_wel_eval, JB09_e_wel_eval, JB09_e5_wel_eval, JB09_e95_wel_eval, JB09_e_std_wel_eval, JB09_e5_std_wel_eval, JB09_e95_std_wel_eval = [], [], [], [], [], [], []

for i in tqdm(range(len(pSA_T))):

    # NZ GMDB
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = gmscm.GMSCM_JB09(case, pSA_T[i], hmax, n_rec, cbb21_pSAobs_T_eval[i], plot=False)
    JB09_ee_eval.append(e); JB09_e_eval.append(e_mean); JB09_e5_eval.append(e5); JB09_e95_eval.append(e95)
    JB09_e_std_eval.append(e_std); JB09_e5_std_eval.append(estd_5); JB09_e95_std_eval.append(estd_95)

    # Christchurch
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = gmscm.GMSCM_JB09(case, pSA_T[i], hmax, n_rec, cbb21_pSAobs_T_chc_eval[i], plot=False)
    JB09_ee_chc_eval.append(e); JB09_e_chc_eval.append(e_mean); JB09_e5_chc_eval.append(e5); JB09_e95_chc_eval.append(e95)
    JB09_e_std_chc_eval.append(e_std); JB09_e5_std_chc_eval.append(estd_5); JB09_e95_std_chc_eval.append(estd_95)

    # Wellington
    p, e, e_mean, e_std, e5, e95, estd_5, estd_95 = gmscm.GMSCM_JB09(1, pSA_T[i], hmax, n_rec, cbb21_pSAobs_T_wel_eval[i], plot=False)
    JB09_ee_wel_eval.append(e); JB09_e_wel_eval.append(e_mean); JB09_e5_wel_eval.append(e5); JB09_e95_wel_eval.append(e95)
    JB09_e_std_wel_eval.append(e_std); JB09_e5_std_wel_eval.append(estd_5); JB09_e95_std_wel_eval.append(estd_95)

#### 9.2.3.2 Rampersad at al., 2026

In [ ]:
# development dataset
RA26_nz_dev = gmscm.GMSCM_RA26(cbb21_pSAobs_T_dev, pSA_T, nz_params)
RA26_chc_dev = gmscm.GMSCM_RA26(cbb21_pSAobs_T_chc_dev, pSA_T, nz_params)
RA26_wel_dev = gmscm.GMSCM_RA26(cbb21_pSAobs_T_wel_dev, pSA_T, nz_params)

# evaluation dataset
RA26_nz_eval = gmscm.GMSCM_RA26(cbb21_pSAobs_T_eval, pSA_T, nz_params)
RA26_chc_eval = gmscm.GMSCM_RA26(cbb21_pSAobs_T_chc_eval, pSA_T, nz_params)
RA26_wel_eval = gmscm.GMSCM_RA26(cbb21_pSAobs_T_wel_eval, pSA_T, nz_params)

## 9.3 LOWESS Computation and CI

In [ ]:
features = ['h', 'dA', 'dS', 'dTRI', 'dZ1']
regions = ['all', 'chc', 'wel']

### 9.3.1 Development dataset

In [ ]:
# Attach the relevant datasets to each region
obs_T_dict_dev = {
    'all': cbb21_pSAobs_T_dev,
    'chc': cbb21_pSAobs_T_chc_dev,
    'wel': cbb21_pSAobs_T_wel_dev
}

e_dict_dev = {
    "all": {
        "JB09": JB09_ee_dev,
        "RA26": {
            "E":   RA26_nz_dev["E"]["e"],
            "EA":  RA26_nz_dev["EA"]["e"],
            "EAS": RA26_nz_dev["EAS"]["e"],
            "EASZ": RA26_nz_dev["EASZ"]["e"],
            "EAST": RA26_nz_dev["EAST"]["e"],
            "EASZT": RA26_nz_dev["EASZT"]["e"],
        }
    },
    "chc": {
        "JB09": JB09_ee_chc_dev,
        "RA26": {
            "E":   RA26_chc_dev["E"]["e"],
            "EA":  RA26_chc_dev["EA"]["e"],
            "EAS": RA26_chc_dev["EAS"]["e"],
            "EASZ": RA26_chc_dev["EASZ"]["e"],
            "EAST": RA26_chc_dev["EAST"]["e"],
            "EASZT": RA26_chc_dev["EASZT"]["e"],
        }
    },
    "wel": {
        "JB09": JB09_ee_wel_dev,
        "RA26": {
            "E":   RA26_wel_dev["E"]["e"],
            "EA":  RA26_wel_dev["EA"]["e"],
            "EAS": RA26_wel_dev["EAS"]["e"],
            "EASZ": RA26_wel_dev["EASZ"]["e"],
            "EAST": RA26_wel_dev["EAST"]["e"],
            "EASZT": RA26_wel_dev["EASZT"]["e"],
        }
    }
}

results_dev = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

region_bar = tqdm(total=len(regions), desc="Regions", position=0)
feat_bar   = tqdm(total=len(features), desc="Features", position=1)

for region in regions:
    region_bar.set_description(f"Region: {region}")
    region_bar.refresh()

    obs_list = obs_T_dict_dev[region]
    models = e_dict_dev[region]

    feat_bar.reset(total=len(features))
    feat_bar.set_description(f"{region}: features")
    # feat_bar.refresh()

    for feat in features:
        feat_bar.update(1)

        for model_name, model_data in models.items():

            # ----------------------------------------------------------
            # Case 1: JB09
            # ----------------------------------------------------------
            if model_name == "JB09":
                for i, T in enumerate(pSA_T):
                    try:
                        x = getattr(obs_list[i], feat)
                        y = model_data[i]

                        xgrid, y_grid, mean, stderr = smooth(
                            x, y, K=100, frac=0.5, it=3, show_progress=False
                        )

                        results_dev[region][feat][model_name].append(
                            (xgrid, mean, stderr)
                        )

                    except Exception as e:
                        print(f"Error in {region} | {feat} | JB09 | T={T}: {e}")
                        results_dev[region][feat][model_name].append(
                            (None, None, None)
                        )

            # ----------------------------------------------------------
            # Case 2: RA26 (E, EA, EAS, EASZ, EAST, EASZT)
            # ----------------------------------------------------------
            else:
                for variant, residual_list in model_data.items():
                    key = f"{model_name}_{variant}"

                    for i, T in enumerate(pSA_T):

                        try:
                            x = getattr(obs_list[i], feat)
                            y = residual_list[i]

                            xgrid, y_grid, mean, stderr = smooth(
                                x, y, K=100, frac=0.5, it=3, show_progress=False
                            )

                            results_dev[region][feat][key].append(
                                (xgrid, mean, stderr)
                            )

                        except Exception as e:
                            print(f"Error in {region} | {feat} | {key} | T={T}: {e}")
                            results_dev[region][feat][key].append(
                                (None, None, None)
                            )

    region_bar.update(1)

### 9.3.2 Evaluation dataset

In [ ]:
# Attach the relevant datasets to each region
obs_T_dict_eval = {
    'all': cbb21_pSAobs_T_eval,
    'chc': cbb21_pSAobs_T_chc_eval,
    'wel': cbb21_pSAobs_T_wel_eval
}

e_dict_eval = {
    "all": {
        "JB09": JB09_ee_eval,
        "RA26": {
            "E":   RA26_nz_eval["E"]["e"],
            "EA":  RA26_nz_eval["EA"]["e"],
            "EAS": RA26_nz_eval["EAS"]["e"],
            "EASZ": RA26_nz_eval["EASZ"]["e"],
            "EAST": RA26_nz_eval["EAST"]["e"],
            "EASZT": RA26_nz_eval["EASZT"]["e"],
        }
    },
    "chc": {
        "JB09": JB09_ee_chc_eval,
        "RA26": {
            "E":   RA26_chc_eval["E"]["e"],
            "EA":  RA26_chc_eval["EA"]["e"],
            "EAS": RA26_chc_eval["EAS"]["e"],
            "EASZ": RA26_chc_eval["EASZ"]["e"],
            "EAST": RA26_chc_eval["EAST"]["e"],
            "EASZT": RA26_chc_eval["EASZT"]["e"],
        }
    },
    "wel": {
        "JB09": JB09_ee_wel_eval,
        "RA26": {
            "E":   RA26_wel_eval["E"]["e"],
            "EA":  RA26_wel_eval["EA"]["e"],
            "EAS": RA26_wel_eval["EAS"]["e"],
            "EASZ": RA26_wel_eval["EASZ"]["e"],
            "EAST": RA26_wel_eval["EAST"]["e"],
            "EASZT": RA26_wel_eval["EASZT"]["e"],
        }
    }
}

results_eval = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

region_bar = tqdm(total=len(regions), desc="Regions", position=0)
feat_bar   = tqdm(total=len(features), desc="Features", position=1)

for region in regions:
    region_bar.set_description(f"Region: {region}")
    region_bar.refresh()

    obs_list = obs_T_dict_eval[region]
    models = e_dict_eval[region]

    feat_bar.reset(total=len(features))
    feat_bar.set_description(f"{region}: features")
    # feat_bar.refresh()

    for feat in features:
        feat_bar.update(1)

        for model_name, model_data in models.items():

            # ----------------------------------------------------------
            # Case 1: JB09
            # ----------------------------------------------------------
            if model_name == "JB09":
                for i, T in enumerate(pSA_T):
                    try:
                        x = getattr(obs_list[i], feat)
                        y = model_data[i]

                        xgrid, y_grid, mean, stderr = smooth(
                            x, y, K=100, frac=0.5, it=3, show_progress=False
                        )

                        results_eval[region][feat][model_name].append(
                            (xgrid, mean, stderr)
                        )

                    except Exception as e:
                        print(f"Error in {region} | {feat} | JB09 | T={T}: {e}")
                        results_eval[region][feat][model_name].append(
                            (None, None, None)
                        )

            # ----------------------------------------------------------
            # Case 2: RA26 (E, EA, EAS, EASZ, EAST, EASZT)
            # ----------------------------------------------------------
            else:
                for variant, residual_list in model_data.items():
                    key = f"{model_name}_{variant}"

                    for i, T in enumerate(pSA_T):

                        try:
                            x = getattr(obs_list[i], feat)
                            y = residual_list[i]

                            xgrid, y_grid, mean, stderr = smooth(
                                x, y, K=100, frac=0.5, it=3, show_progress=False
                            )

                            results_eval[region][feat][key].append(
                                (xgrid, mean, stderr)
                            )

                        except Exception as e:
                            print(f"Error in {region} | {feat} | {key} | T={T}: {e}")
                            results_eval[region][feat][key].append(
                                (None, None, None)
                            )

    region_bar.update(1)

### 9.3.3 Plots

In [ ]:
xlims = {'h': (0, 25), 'dA': (0, 20), 'dS': (0, 760), 'dTRI': (0, 90), 'dZ1': (0, 300)}

#### 9.3.3.1 Development dataset

In [ ]:
for region in regions:
    for model in ["RA26_E","RA26_EA","RA26_EAS","RA26_EASZ","RA26_EAST","RA26_EASZT"]:
        if region == 'wel':
            plot_corr_deviation(
                results = results_dev,
                obs_T_dict = obs_T_dict_dev,
                e_dict = e_dict_dev,
                region=region,
                model_key=model,
                period_indices=[0, 3, 6, 8],
                period_labels=['PGA', '0.1s', '1s', '3s'],
                xlims= xlims,
                show_plot = False,
                save_path = f'dev_{region}_{model}_PGA_0p1_SA1s_SA3s.png',
            )

        else:
            plot_corr_deviation(
                results = results_dev,
                obs_T_dict = obs_T_dict_dev,
                e_dict = e_dict_dev,
                region=region,
                model_key=model,
                period_indices=[0, 3, 6, 8],
                period_labels=['PGA', '0.1s', '1s', '3s'],
                show_plot = False,
                save_path = f'dev_{region}_{model}_PGA_0p1_SA1s_SA3s.png',
            )

#### 9.3.3.2 Evaluation dataset

In [ ]:
for region in regions:
    for model in ["RA26_E","RA26_EA","RA26_EAS","RA26_EASZ","RA26_EAST","RA26_EASZT"]:
        if region == 'wel':
            plot_corr_deviation(
                results = results_eval,
                obs_T_dict = obs_T_dict_eval,
                e_dict = e_dict_eval,
                region=region,
                model_key=model,
                period_indices=[0, 3, 6, 8],
                period_labels=['PGA', '0.1s', '1s', '3s'],
                xlims= xlims,
                show_plot = False,
                save_path = f'eval_{region}_{model}_PGA_0p1_SA1s_SA3s.png'
            )

        else:
            plot_corr_deviation(
                results = results_eval,
                obs_T_dict = obs_T_dict_eval,
                e_dict = e_dict_eval,
                region=region,
                model_key=model,
                period_indices=[0, 3, 6, 8],
                period_labels=['PGA', '0.1s', '1s', '3s'],
                show_plot = False,
                save_path = f'eval_{region}_{model}_PGA_0p1_SA1s_SA3s.png'
            )